# Macro Studio 5.1 · Generador autónomo de la aplicación

**Macroeconomía II · EC-500 · Universidad Fidélitas.** Proyecto académico desarrollado por Juan Ignacio Garbanzo Fallas.

Este notebook reconstruye la aplicación pública desde sus fuentes internas y genera un único archivo **`index.html`**, compatible con GitHub Pages. No es necesario instalar librerías Python adicionales fuera de Jupyter e IPython. Su ejecución no realiza publicaciones automáticas en GitHub: una vez que se guarda y se sube el notebook, el archivo `index.html` debe sincronizarse en el repositorio para activar Pages.

### Revisiones académicas

- En las rutas que nacen en el mercado monetario, el retorno de DA en la secuencia 3 procede de `O$2 → TC → X−M`; no se construyen relaciones `DL3 → DA3` ni `Md3 → DA3`.
- Se conservan las rutas de primer y segundo orden y el análisis de predominio **solo donde corresponda a antecedentes de secuencias anteriores**.
- La política fiscal (G/impuestos) empieza en bienes y servicios; la monetaria (Ms/BCCR) comienza en el monetario.
- Se incorpora formulario económico y diagnóstico **opcional** de los excesos IS–LM–BP, diferenciando el punto inicial de la intervención que se simula.
- La versión es cualitativa: no calcula elasticidades, magnitudes de shock ni equilibrio general empírico.

**Material docente principal:** Zamora Leiva, R. (2026). *Presentaciones de Macroeconomía II: mercados de bienes y servicios, laboral, monetario y cambiario*; *Casos 3Q26*; transcripción de la clase del **24 de noviembre de 2026**. Los materiales fuente de la cátedra no están redistribuidos en el HTML público.

In [ ]:
from pathlib import Path
import hashlib
from IPython.display import FileLink,display

In [ ]:
PLANTILLA_HTML = '<!DOCTYPE html>\n<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Macro Studio 5.1 · Laboratorio macroeconómico</title><style>@@MS_STYLE@@</style><script id="macro-snapshot" type="application/json">null</script></head><body><div id="app"></div><script>@@MS_ENGINE@@</script><script>@@MS_REFERENCE@@</script><script>@@MS_APPLICATION@@</script></body></html>'

In [ ]:
ESTILOS_CSS = ":root{--ink:#172b3d;--body:#33485e;--muted:#698197;--border:#d9e3ec;--blue:#167ab5;--sky:#e9f4fb;--rose:#c93e85;--paper:#ffffff;--bg:#edf2f6;--navy:#142c40;--shadow:0 6px 22px rgba(21,49,72,.07)}\n*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:var(--bg);font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Inter,Arial,sans-serif;color:var(--ink);font-size:14px;line-height:1.55}button,input,select,textarea{font:inherit}button{cursor:pointer}button:disabled{opacity:.35;cursor:not-allowed}button:focus-visible,input:focus-visible,select:focus-visible,textarea:focus-visible{outline:3px solid #80b9e5;outline-offset:2px}\n.topbar{height:78px;background:#fff;border-bottom:1px solid var(--border);display:flex;align-items:center;justify-content:space-between;gap:24px;padding:0 28px;position:sticky;top:0;z-index:30;box-shadow:0 2px 10px #122e450c}.identity{display:flex;align-items:center;gap:13px;min-width:205px}.identity strong{display:block;font-size:15px;letter-spacing:.055em;line-height:1.25}.identity small{display:block;color:var(--muted);font-size:11px}.logomark{width:45px;height:45px;border-radius:13px;background:var(--navy);color:white;display:flex;align-items:center;justify-content:center;font-size:25px;font-weight:900;letter-spacing:-2px;flex-shrink:0}.logomark span{font-size:11px;font-weight:800;color:#82c9ed;margin-left:1px;align-self:center}.toptools,.caseactions,.expo-buttons{display:flex;align-items:center;gap:9px;flex-wrap:wrap;justify-content:flex-end}.button{border:1px solid #cbd9e5;background:#fff;color:#23445e;padding:9px 13px;border-radius:9px;min-height:36px;font-size:12px;font-weight:750;white-space:nowrap;transition:background .16s,box-shadow .16s}.button:hover{background:#edf5fa;box-shadow:0 2px 8px #20354715}.button.primary{background:var(--navy);color:#fff;border-color:var(--navy)}.button.primary:hover{background:#234964}.button.ghost{background:transparent;color:#718599}.button.small{font-size:11px;padding:6px 10px;min-height:31px}.button.icon{font-size:24px;border:0;width:38px;height:38px}.workspace{display:grid;grid-template-columns:264px minmax(0,1fr);min-height:calc(100vh - 78px)}\n.rail{background:#f9fbfd;border-right:1px solid var(--border);padding:24px 16px;position:sticky;top:78px;height:calc(100vh - 78px);overflow-y:auto}.rail-label,.rail-row{font-size:10px;font-weight:900;letter-spacing:.13em;color:#788ea2;margin:0 8px 14px}.preslist,.case-list{display:grid;gap:5px}.rail-row{display:flex;justify-content:space-between;align-items:center;margin-top:30px;margin-bottom:10px}.rail-add{min-height:26px;padding:1px 10px;font-size:18px}.pres-entry,.case-entry{display:flex;align-items:center;width:100%;border:1px solid transparent;padding:12px 10px;border-radius:9px;background:none;color:#466075;text-align:left;gap:9px;font-size:12px}.pres-entry:hover,.case-entry:hover{background:#eff5fa}.pres-entry.on,.case-entry.on{background:#e9f2fa;border-color:#c6deee;color:#0f507e}.pres-entry b{margin-left:auto;color:#668ba5;font-size:11px}.truncate{flex:1;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}.folder-icon{font-size:17px;color:#557e9e}.case-index{font-size:10px;color:#9aafbf;letter-spacing:.05em;font-weight:800}.rail-actions{margin-top:20px;display:grid;gap:9px}.rail-wide{width:100%;background:#fff}.rail-wide.soft{color:#215f89;border-color:#b8d8ed}.rail-note{font-size:11px;color:#8fa0b0;border-top:1px solid var(--border);margin-top:26px;padding:19px 8px}\n.main{max-width:1620px;padding:26px 26px 48px;min-width:0;width:100%;margin:0 auto}.case-heading{display:flex;align-items:flex-start;justify-content:space-between;gap:24px;margin-bottom:21px}.eyebrow{font-size:11px;color:#467797;font-weight:900;letter-spacing:.10em;text-transform:uppercase;margin-bottom:6px}.eyebrow span{color:#b4c3cf;padding:0 8px}.case-heading h1{font-size:clamp(26px,2.1vw,34px);line-height:1.22;letter-spacing:-.035em;margin:0;max-width:900px}.subtitle{color:#73879a;margin-top:7px;font-size:12px}.caseactions{padding-top:3px}\n.controlpanel{background:#fff;border:1px solid var(--border);border-radius:14px;box-shadow:var(--shadow);padding:16px 17px;margin-bottom:22px}.control-main{display:grid;grid-template-columns:1.28fr 1.36fr .60fr;gap:14px}.control-main label{display:grid;gap:7px;font-weight:800;color:#466076;font-size:11px;text-transform:uppercase;letter-spacing:.06em}.control-main input,.control-main select,.assumption select{border:1px solid #cbd9e6;background:#fff;border-radius:8px;min-height:39px;padding:8px 10px;color:#173044;width:100%;outline:none;text-transform:none;font-weight:550;letter-spacing:0;font-size:13px}.control-options{border-top:1px solid #e5edf2;margin-top:14px;padding-top:12px;display:flex;align-items:center;gap:20px;flex-wrap:wrap}.toggle{display:flex;align-items:center;gap:9px;color:#425e73;font-size:12px;font-weight:700;white-space:nowrap}.toggle input{accent-color:#247cb2;width:16px;height:16px}.assumption{margin-left:auto;display:flex;align-items:center;gap:11px;color:#658096;font-weight:700;font-size:11px}.assumption select{width:175px;min-height:34px;font-size:11px}\n.paper{background:#fff;border:1px solid var(--border);border-radius:18px;padding:23px;box-shadow:var(--shadow)}.paper-header{display:flex;align-items:flex-end;justify-content:space-between;gap:16px;border-bottom:1px solid #e0e9f0;padding-bottom:18px;margin-bottom:21px}.paper-overline{font-size:10px;color:#7f96a8;letter-spacing:.14em;font-weight:900;margin-bottom:5px}.paper-header h2{margin:0 0 8px;line-height:1.18;letter-spacing:-.03em;font-size:clamp(24px,2vw,30px)}.paper-header p{margin:0;color:#72889b;font-size:12px}.paper-header p strong{color:#486b86}.result-pills{display:flex;align-items:center;gap:7px;flex-wrap:wrap;justify-content:flex-end}.pill{font-size:11px;font-weight:800;padding:6px 10px;border-radius:7px;color:#385c74;background:#e9f3fa;border:1px solid #d5e8f5}.pill.teal{background:#e9f7f2;color:#1f7767;border-color:#c7e9de}.pill.pink{background:#fff0f6;color:#a13872;border-color:#f5d5e5}.pill.amber{background:#fff6e8;color:#996021;border-color:#f1dfc3}.pill.violet{background:#f3effc;color:#694e98;border-color:#e2dbf4}\n.chart-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:22px;align-items:start}.market-card{border:1px solid #d8e3ec;border-radius:12px;overflow:hidden;background:#fff;min-width:0;break-inside:avoid;box-shadow:0 2px 8px #152e4110}.market-head{min-height:86px;padding:17px 19px;display:flex;justify-content:space-between;align-items:center;gap:15px;border-bottom:1px solid #e1eaf1;background:#f9fbfd}.market-eyebrow{font-size:9px;letter-spacing:.15em;font-weight:900;color:#90a0af;margin-bottom:5px}.market-head h3{margin:0;font-size:18px;line-height:1.1;letter-spacing:-.027em}.market-head h3 small{font-size:13px;font-weight:600;color:#7089a1}.market-numbers{display:flex;align-items:center;justify-content:flex-end;gap:7px}.stage-dot{display:flex;align-items:center;justify-content:center;height:37px;width:37px;background:#e8f4fb;border:2px solid #abcee4;color:#0e70aa;font-weight:900;font-size:19px;border-radius:50%}.stage-dot.stage-second{background:#fff0f7;border-color:#edb6d2;color:#b73a7e}.stage-none{font-size:10px;color:#869bad;white-space:nowrap}.is-status{font-weight:900;letter-spacing:.045em;color:#487f9f;font-size:11px;white-space:nowrap}.chart-zone{background:#fff;padding:10px 12px 2px;overflow:hidden}.economic-svg{display:block;width:100%;height:auto;overflow:visible}\n.gridline{stroke:#eaf0f5;stroke-width:1.2}.plot-axis{stroke:#24384b;stroke-width:3.1}.fixed-line{stroke:#223648;stroke-width:3.6;stroke-linecap:round}.curve-line{stroke-width:4.1;stroke-linecap:round;fill:none}.guide{stroke-width:1.55;stroke-dasharray:7 7;stroke-opacity:.75}.axis-title{font-weight:850;font-size:24px;fill:#1b3448}.axislabel{fill:#20394f;font-size:19px;font-weight:700}.curve-label{font-size:22px;font-weight:900;paint-order:stroke;stroke:#fff;stroke-width:5;stroke-linejoin:round}.curve-label.fixed{fill:#20394f}.projection-label{font-size:19px;font-weight:850;paint-order:stroke;stroke:#fff;stroke-width:4}.equilibrium-text{font-size:18px;font-weight:900;stroke:none}.sequence-caption{font-size:14px;letter-spacing:.15em;font-weight:900;fill:#7791a8}.stage-shape{fill:#e7f3fa;stroke:#a8d0e9;stroke-width:2}.stage-shape.stage-late{fill:#fce9f3;stroke:#ebb4d2}.stage-in-chart{font-size:24px;font-weight:950;fill:#254f71}.no-movement{font-size:15px;fill:#91a5b6;font-weight:750}.is-point{fill:#24384b}.is-point.moving{fill:#1879b9}.is-callout{font-size:19px;fill:#487a9c;font-weight:850;letter-spacing:.05em}\n.narrative{border-top:1px solid #dfe9f1;padding:15px 18px 14px;background:#fcfdff}.narrative-second{border-top:3px solid #efbed5;background:#fffafd}.narrative-header{display:flex;align-items:center;justify-content:space-between;gap:8px;margin-bottom:10px}.narrative-header strong,.narrative-header>span{font-size:12px;letter-spacing:.07em;color:#1d5f89;font-weight:900}.narrative-second .narrative-header strong{color:#b03c78}.narrative-header div{display:flex;align-items:center;gap:9px;justify-content:flex-end;flex-wrap:wrap}.small-stage{font-size:10px;font-weight:850;letter-spacing:.055em;color:#728ca0}.origin-tag{font-size:9px;color:#9caebe}.paragraphs{display:grid;gap:8px}.paragraph{display:grid;grid-template-columns:26px minmax(0,1fr);gap:10px;align-items:flex-start}.para-n{width:25px;height:25px;background:#e9f3fa;color:#1b6692;border-radius:7px;display:flex;align-items:center;justify-content:center;font-size:12px;font-weight:900}.narrative-second .para-n{background:#ffedf5;color:#bb487f}.paragraph p{font-size:12.5px;color:#334b60;line-height:1.5;margin:1px 0 0}.audit-note{border:1px solid #d6e4ed;background:#f4f9fc;border-radius:10px;padding:12px 15px;color:#648098;margin-top:18px;font-size:11px}.audit-note strong{color:#35617d}.supplement{margin-top:12px;font-size:12px;color:#586f83;border:1px solid #e1e9f0;border-radius:9px;padding:10px 14px}.supplement summary{font-weight:900;cursor:pointer}.supplement p{margin:9px 0}.method-footer{font-size:11px;color:#8aa0b1;line-height:1.55;margin:18px 5px 0}\n.expo{min-height:100vh;background:#e9eef3;padding-bottom:30px}.expo-toolbar{background:#fff;border-bottom:1px solid #dce6ee;position:sticky;top:0;z-index:20;padding:13px 24px;display:flex;align-items:center;justify-content:space-between;gap:18px}.expo-pages{max-width:1600px;margin:0 auto;padding:22px}.expo-pages .paper{margin-bottom:24px}.expo .paper-header h2{font-size:32px}.expo .chart-grid{gap:22px}.expo .paragraph p{font-size:13.5px}.expo .market-head h3{font-size:21px}\n.dialog-back{position:fixed;inset:0;background:#0b1e30bb;display:grid;place-items:center;z-index:60;padding:30px}.dialog{width:min(780px,100%);max-height:88vh;overflow:hidden;background:white;border-radius:17px;box-shadow:0 35px 100px #09172555;display:flex;flex-direction:column}.dialog.large{width:min(920px,100%)}.dialog.editor-dialog{width:min(1000px,100%)}.dialog-head{padding:24px 28px 18px;border-bottom:1px solid var(--border);display:flex;align-items:start;justify-content:space-between;gap:18px}.dialog-head h2{font-size:26px;letter-spacing:-.04em;margin:0 0 5px}.dialog-head p{margin:0;color:#7990a3;font-size:12px}.bank-list,.text-editor-scroll{overflow-y:auto;padding:12px 24px 25px}.bank-row{padding:14px 4px;display:grid;grid-template-columns:32px 1fr auto;gap:13px;align-items:center;border-bottom:1px solid #e4ecf2}.bank-number{font-weight:900;font-size:12px;color:#8ca5b6}.bank-row strong{display:block;color:#203c52;font-size:13px}.bank-row small{display:block;color:#9eafbc;font-size:11px;margin-top:2px}.edit-block{padding:12px 4px;border-bottom:1px solid #dde7ef}.edit-block h3{font-size:19px;margin:3px 0 10px}.edit-block h4{font-size:13px;color:#2b6f9b;margin:16px 0 6px}.edit-label{display:grid;gap:6px;color:#7590a5;font-weight:750;font-size:11px;margin-bottom:12px}.edit-label textarea{width:100%;min-height:63px;border:1px solid #cfdee8;border-radius:9px;padding:9px 11px;color:#1c374c;resize:vertical;font-size:12px;line-height:1.45}.dialog-footer{padding:12px 24px;border-top:1px solid #dde7ef;display:flex;justify-content:flex-end}.toast{position:fixed;bottom:22px;right:22px;background:#183f57;color:white;border-radius:9px;padding:12px 17px;z-index:100;box-shadow:0 12px 36px #132d4455;font-weight:700;font-size:12px}\n@media(max-width:1400px){.topbar{padding:0 19px}.toptools{gap:6px}.toptools .button{font-size:10px;padding:8px}.workspace{grid-template-columns:225px minmax(0,1fr)}.main{padding:22px 18px}.market-head{padding:14px}.paper{padding:16px}.chart-grid{gap:13px}.chart-zone{padding:8px 8px 0}.market-head h3{font-size:16px}.paragraph p{font-size:12px}}\n@media(max-width:1020px){.topbar{position:relative;height:auto;min-height:72px;flex-wrap:wrap;padding:13px}.rail{position:relative;height:auto;top:0}.workspace{grid-template-columns:190px 1fr}.chart-grid{grid-template-columns:1fr}.control-main{grid-template-columns:1fr 1fr}.control-main label:first-child{grid-column:1/-1}.paper-header{align-items:flex-start;flex-wrap:wrap}.case-heading{flex-wrap:wrap}.assumption{margin-left:0}}\n@media(max-width:680px){.workspace{display:block}.rail{position:relative;border-right:0;border-bottom:1px solid var(--border);padding:12px}.rail-label,.rail-note{display:none}.preslist,.case-list{display:flex;overflow-x:auto}.pres-entry,.case-entry{min-width:170px}.rail-row{margin:12px 0}.rail-actions{display:flex;margin-top:10px}.main{padding:12px}.control-main{grid-template-columns:1fr}.control-main label:first-child{grid-column:auto}.paper{padding:11px}.market-head{padding:12px}.stage-dot{width:31px;height:31px;font-size:17px}.market-head h3{font-size:16px}.chart-zone{padding:4px}.paragraph p{font-size:12px}.dialog-back{padding:8px}.expo-toolbar{position:relative;flex-wrap:wrap}.expo-pages{padding:10px}}\n@page{size:A4 landscape;margin:9mm}\n@media print{*{-webkit-print-color-adjust:exact!important;print-color-adjust:exact!important}body,.expo{background:#fff!important}.topbar,.rail,.case-heading,.controlpanel,.toptools,.expo-toolbar,.caseactions,.method-footer,.dialog-back,.toast,.supplement{display:none!important}.workspace{display:block}.main{max-width:none;padding:0}.paper,.expo-pages .paper{padding:0;box-shadow:none;border:0;margin:0}.expo-pages{max-width:none;padding:0}.paper-header{margin:0 0 10px;padding:0 0 8px}.paper-header h2{font-size:23px}.paper-overline{font-size:8px}.chart-grid{grid-template-columns:1fr 1fr;gap:7px}.market-card{box-shadow:none;break-inside:avoid;border-radius:5px}.market-head{min-height:40px;padding:5px 8px}.market-head h3{font-size:13px}.stage-dot{height:23px;width:23px;font-size:12px}.chart-zone{padding:2px}.narrative{padding:6px}.narrative-header{margin-bottom:3px}.narrative-header strong,.narrative-header>span{font-size:9px}.origin-tag{display:none}.paragraphs{gap:3px}.paragraph{grid-template-columns:16px 1fr;gap:4px}.para-n{height:15px;width:15px;font-size:9px;border-radius:4px}.paragraph p{font-size:8.5px!important;line-height:1.2}.audit-note{font-size:8px;padding:5px;margin:6px 0}.paper{break-after:page}.result-pills .pill{padding:3px 5px;font-size:9px}}\n.custom-form{display:grid;gap:12px;padding:21px 28px 0;overflow-y:auto}.custom-form>label{display:grid;gap:6px;font-size:11px;font-weight:850;color:#638197;text-transform:uppercase;letter-spacing:.05em}.custom-form input,.custom-form select,.custom-form textarea{font-size:13px;letter-spacing:0;text-transform:none;min-height:37px;border:1px solid #cddde9;border-radius:8px;padding:9px 10px;color:#234257;outline:none;background:#fff}.custom-form textarea{resize:vertical}.custom-form .dialog-footer{margin:8px -28px 0;padding:15px 28px}\n/* Exportación PDF: hoja A3 vertical, evita una portada huérfana por falta de altura. */\n@page{size:A3 portrait;margin:11mm}\n@media print{\n  .market-card{break-inside:avoid-page}\n  .chart-zone .economic-svg{max-height:275px;width:100%;}\n  .paper-header{break-after:avoid}\n}\n/* Estabilización: numeración en el encabezado; señalética y redacciones auditables. */\n.stage-dot.stage-1{background:#edf1f4;border-color:#a7b2be;color:#25394d}\n.stage-dot.stage-2{background:#e4f3fb;border-color:#92c2e1;color:#126ca4}\n.stage-dot.stage-3{background:#fde7f2;border-color:#eca8cc;color:#ad3071}\n.stage-dot.stage-4{background:#f2eafb;border-color:#bea3df;color:#7a46a7}\n.stage-dot.stage-later{background:#e4f7f1;border-color:#9ad8c5;color:#246f60}\n.economic-token{font-weight:850;color:var(--token);background:color-mix(in srgb,var(--token) 8%,#ffffff);padding:0 2px;border-radius:3px;white-space:nowrap}\n.paragraph p{font-size:13.3px;line-height:1.67;color:#253e53}\n.expo .paragraph p{font-size:13.6px}\n.narrative-header{align-items:center}.narrative-header strong{font-size:12.5px}.origin-tag{font-size:10px;color:#688096}\n.annotation-leader{fill:none;stroke-width:1.5;stroke-dasharray:3 3;opacity:.75}\n.equilibrium-callout{font-size:22px;font-weight:900;paint-order:stroke;stroke:#fff;stroke-width:6px}\n.market-card .chart-zone{padding:11px 12px 2px}\n.curve-label.fixed{font-weight:950;font-size:22px;stroke-width:6}\n.is-status{white-space:normal;text-align:right;max-width:220px;line-height:1.25}\n@media(max-width:1050px){.market-head{align-items:flex-start}.market-numbers{flex-shrink:0}}\n/* 2026-10-20: case chronology and IS–LM equilibrium panel */\n.stage-dot.stage-first{background:#e7f2fb;color:#176997;border-color:#9bc6e3}\n.stage-dot.stage-second{background:#fce8f2;color:#b32f75;border-color:#eaa3c8}\n.sequence-band{border:1px solid #d7e4ee;border-radius:12px;background:#f6faff;padding:14px 18px 15px;margin:0 0 22px}\n.sequence-title{letter-spacing:.14em;font-weight:900;color:#587991;font-size:10px;margin-bottom:9px}\n.sequence-steps{display:flex;gap:9px;flex-wrap:wrap}\n.sequence-event{font-weight:710;font-size:12px;color:#345d78;border:1px solid #bad5e9;padding:7px 10px;border-radius:8px;background:white;white-space:normal}\n.sequence-event b{display:inline-flex;align-items:center;justify-content:center;border-radius:100px;width:23px;height:23px;background:#d8ebfa;color:#1a73aa;margin-right:5px}\n.sequence-event.second{border-color:#efc2d9;color:#8b3864;background:#fff9fc}.sequence-event.second b{background:#fce4f0;color:#af3176}\n.combined-card{grid-column:1/-1}\n.combined-card .chart-zone{max-width:950px;margin:0 auto;width:100%}\n.combined-card .economic-svg{max-width:790px;margin:0 auto}\n.consistency-note{background:#fff7ed;color:#765124;border-top:1px solid #f1d8aa;padding:14px 22px;font-size:13px;line-height:1.55;font-weight:600}\n@media print{.sequence-band{margin-bottom:10px;padding:9px}.sequence-event{font-size:9px;padding:4px 6px}.sequence-event b{height:17px;width:17px}.combined-card{grid-column:1/-1}.combined-card .chart-zone{max-width:650px}.consistency-note{font-size:10px}.sequence-title{font-size:8px}}\n\n.region-explorer{border-top:1px solid #dae6f0;padding:16px 21px;background:#fbfdff;color:#2a4961;font-size:13px}.region-explorer summary{font-size:13px;font-weight:850;color:#255f88;cursor:pointer}.region-explorer p{line-height:1.6}.region-explorer label{display:flex;gap:10px;align-items:center;flex-wrap:wrap;font-weight:750}.region-explorer select{margin-left:4px;min-height:36px;color:#1d425d;border:1px solid #bad1e2;border-radius:8px;padding:6px;background:#fff}.zone-reading{border-left:3px solid #1879b9;padding:12px;background:#edf6fc;margin:12px 0;font-weight:750}.region-explorer:not([open]) p{display:none}@media print{.region-explorer{display:none!important}}\n\n/* Revisión de trazabilidad docente 2.1 */\n.evidence-warning{display:flex;flex-direction:column;gap:7px;background:#fff8ed;border:1px solid #e0bf84;border-left:5px solid #c98a2e;border-radius:11px;padding:13px 17px;color:#704915;font-size:13px;line-height:1.48;margin:12px 0 17px}.evidence-warning strong{font-size:13.5px;color:#6a3e0d}.evidence-warning span{max-width:1100px}\n@media print{.evidence-warning{background:#fff9ed!important;break-inside:avoid}}\n\n/* Alternativa contextual documentada de segundo orden */\n.balance-choice{display:grid;grid-template-columns:minmax(0,1fr) minmax(300px,.85fr);gap:24px;padding:20px 24px;margin:16px 0 20px;background:linear-gradient(115deg,#ffffff,#f5f9fd);border:1px solid #b3d3e8;border-left:5px solid #1879b9;border-radius:16px;box-shadow:0 6px 16px #19394d10}\n.balance-choice h3{margin:5px 0 7px;font-size:20px;letter-spacing:-.025em}.balance-choice p{color:#465e72;margin:0;line-height:1.6}.balance-choice .eyebrow{font-size:10px;font-weight:800;letter-spacing:.13em;color:#126a9c}\n.balance-radio{border:0;padding:0;margin:0;display:flex;flex-direction:column;gap:7px;min-width:0}.balance-radio legend{font-size:11px;font-weight:800;color:#5d7589;margin-bottom:8px;text-transform:uppercase;letter-spacing:.075em}\n.balance-option{display:flex;align-items:center;gap:13px;border:1px solid #d5e2ea;background:#fff;border-radius:10px;padding:11px 13px;cursor:pointer}.balance-option input{accent-color:#1879b9;width:17px;height:17px;flex-shrink:0}.balance-option b{display:block;font-size:13px}.balance-option small{display:block;color:#607a8d;font-size:11px;margin-top:2px}.balance-option.chosen{border-color:#1879b9;background:#eaf5fc;box-shadow:inset 0 0 0 1px #1879b9}\n.balance-declaration{padding:14px 18px;margin:16px 0;background:#eef7fd;border:1px solid #b7d9ef;border-radius:10px;color:#204a68;font-size:13px;line-height:1.55}.balance-declaration strong{color:#123f5d}.balance-declaration span{display:block;margin-top:5px;color:#60798a;font-size:12px}\n@media(max-width:860px){.balance-choice{grid-template-columns:1fr;gap:16px}}\n@media print{.balance-choice{display:none}.balance-declaration{background:#f0f6fa;break-inside:avoid;font-size:11px}}\n\n/* V3.1 readability and teaching notes */\n.islm-tutorial{margin:12px 18px 17px;padding:13px 18px;border:1px solid #bdd7e9;border-radius:14px;background:#f1f8ff;color:#1b405b;font-size:14px;line-height:1.65;}\n.islm-tutorial summary{font-weight:800;cursor:pointer;color:#145d8a;}\n.islm-tutorial p{margin:10px 0;}\n.source-direction{display:block;color:#975a20;font-size:11px;line-height:1.4;margin-top:4px;}\n.chart-zone{overflow:visible;}\n.economic-svg text.curve-label{paint-order:stroke;stroke:#fff;stroke-width:4px;stroke-linejoin:round;font-size:17px;}\n.economic-svg text.projection-label{paint-order:stroke;stroke:#fff;stroke-width:3px;}\n@media print{.islm-tutorial{display:none;}}\n\n\n/* Macro Studio 5.0 · Diagrama causal auditable */\n.control-main{grid-template-columns:repeat(auto-fit,minmax(210px,1fr))}\n.always-market{font-weight:800;color:#1a5b86;font-size:12px;padding:4px 12px;background:#ebf6fc;border:1px solid #b8d6e9;border-radius:8px}\n.causal-panel{border:1px solid #cfdfec;border-radius:16px;background:linear-gradient(120deg,#fff,#f5f9fc);padding:20px 22px;margin:18px 0 22px;box-shadow:0 3px 14px #193c5212;break-inside:avoid}\n.causal-title{display:flex;justify-content:space-between;align-items:center;gap:20px}.causal-title h3{font-size:20px;color:#1b384b;letter-spacing:-.024em;margin:4px 0 4px}.causal-title p{font-size:12px;color:#677e92;margin:0}.causal-cases-label{font-size:10px;font-weight:850;color:#1e6389;background:#e8f5fc;border:1px solid #b5d7ea;padding:7px 9px;border-radius:8px;white-space:nowrap}\n.causal-map{overflow-x:auto;width:100%;margin-top:12px;border:1px solid #dfebf3;background:#fff;border-radius:11px}\n.causal-svg{width:100%;min-width:825px;max-height:495px;display:block}\n.causal-col-label{font-size:13px;fill:#738ba0;font-weight:900;letter-spacing:.05em}.causal-row-label{font-size:16px;fill:#34546d;font-weight:850}.causal-node{cursor:pointer;transition:opacity .16s,transform .16s}.causal-node rect{filter:drop-shadow(0 2px 2px #1b42501a)}.causal-node:hover rect,.causal-node.is-related rect{stroke:#095b8e;stroke-width:4}.causal-node:focus rect{stroke:#0b5890;stroke-width:4;outline:none}\n.causal-edge{opacity:.68;transition:opacity .18s,stroke-width .18s}.causal-edge.is-related{opacity:1;stroke:#1484bc;stroke-width:3.6}\n.causal-legend{display:flex;flex-wrap:wrap;gap:16px;margin-top:12px;font-size:12px;color:#5e778d;font-weight:750;align-items:center}.causal-legend>span{display:inline-flex;gap:7px;align-items:center}.legend-first,.legend-second{width:15px;height:15px;border-radius:4px;border:1px solid}.legend-first{background:#e6eff5;border-color:#6c8ca4}.legend-second{background:#d8f6df;border-color:#59a66f}\n.causal-detail{background:#fff;border-left:3px solid #2387be;border-radius:5px;margin:14px 0 0;padding:10px 14px;min-height:45px;color:#35556b;font-size:13px;line-height:1.55}\n.combined-card .economic-svg{max-width:880px;margin:auto}.combined-card .chart-zone{max-width:1040px}\n@media(max-width:1100px){.causal-title{align-items:flex-start}.causal-map{overflow:auto}.causal-svg{min-width:930px}}\n@media print{.causal-panel{break-inside:avoid;padding:9px 12px;margin:8px 0 10px}.causal-svg{min-width:0;width:100%;height:auto;max-height:220px}.causal-title h3{font-size:13px}.causal-title p,.causal-detail,.causal-legend{font-size:9px}.causal-map{overflow:hidden}.causal-cases-label{font-size:7px}}\n\n/* Macro Studio 5 · cierre visual: colección reutilizable y paleta única */\n:root{--blue:#1879b9;--sky:#e7f3fb;--rose:#cc3889;--rose-wash:#fce8f2}\n.workspace{grid-template-columns:minmax(296px,320px) minmax(0,1fr)}\n.rail{padding:22px 14px 24px;min-width:0;scrollbar-gutter:stable;scrollbar-width:thin}\n.rail-heading{display:flex;align-items:center;justify-content:space-between;gap:9px;padding:0 6px 12px;margin-bottom:4px;border-bottom:1px solid #e4ebf2}\n.rail-heading .rail-label{margin:0 0 1px;font-size:9px}\n.rail-heading strong{font-size:15px;color:#29475f;font-weight:850}\n.rail-add{font-size:18px;min-width:32px;min-height:32px;width:32px;height:32px;border-radius:8px;display:inline-flex;align-items:center;justify-content:center;padding:0}\n.preslist,.case-list{display:grid;gap:6px;max-width:100%;min-width:0;padding:8px 1px 0}\n.pres-row,.case-row{min-width:0;max-width:100%;display:grid;grid-template-columns:minmax(0,1fr) auto;align-items:center;gap:2px;border-radius:10px;border:1px solid transparent;background:transparent;transition:background .15s,border-color .15s}\n.pres-row.on,.case-row.on{background:#e8f3fb;border-color:#c5dff0}\n.pres-row:hover,.case-row:hover{background:#edf6fb}\n.pres-entry,.case-entry{min-width:0;padding:10px 7px;border-radius:8px;font-size:12px;line-height:1.45;border:0!important;background:none!important;display:flex;gap:8px;text-align:left}\n.case-entry{min-height:43px}.pres-entry{min-height:42px}\n.pres-entry .truncate,.case-entry .truncate{min-width:0;display:block;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}\n.pres-entry b{flex:none;font-variant-numeric:tabular-nums}\n/* Dos líneas legibles para títulos extensos, en vez de cortar todo a la mitad. */\n.case-entry .truncate,.pres-entry .truncate{white-space:normal;overflow:hidden;text-overflow:clip;display:-webkit-box;-webkit-box-orient:vertical;-webkit-line-clamp:2;line-clamp:2;line-height:1.35}\n.case-row,.pres-row{align-items:stretch}.case-entry,.pres-entry{align-items:center}\n.rail-tools{display:flex;gap:1px;flex:none;padding-right:3px;align-items:center}\n.rail-tool{display:inline-flex;align-items:center;justify-content:center;width:27px;height:27px;border:0;border-radius:7px;background:transparent;color:#5e8099;font-size:16px;line-height:1}\n.rail-tool:hover{background:#deedf8;color:#145d92}.rail-tool.danger:hover{background:#fff0f5;color:#b43569}\n.rail-row{margin:22px 5px 4px;gap:10px;line-height:1.5;min-width:0}\n.rail-row>span{overflow:hidden;text-overflow:ellipsis;white-space:nowrap;min-width:0}\n.rail-actions{margin-top:18px;gap:7px}.rail-wide{min-height:36px}.reset-workspace{color:#8d5260;border-color:#e3cdd5;font-weight:730}.reset-workspace:hover{background:#fff2f5;color:#982f5b}\n.rail-note{line-height:1.5;padding:15px 4px;margin-top:20px}\n.causal-panel{border-color:#d1e1ed;background:linear-gradient(120deg,#fff,#f6f9fd)}\n.causal-node rect{filter:drop-shadow(0 2px 2px #17374d18)}\n.causal-node.is-related rect{stroke:#126fa8;stroke-width:4}\n.causal-edge{opacity:.53;transition:opacity .18s,stroke-width .18s}\n.causal-edge.first.is-related{stroke:#1879b9;opacity:1;stroke-width:3.6}\n.causal-edge.second.is-related{stroke:#cc3889;opacity:1;stroke-width:3.6}\n.legend-first{background:#e4f3fb;border-color:#1879b9}\n.legend-second{background:#fce8f2;border-color:#cc3889}\n.causal-detail{border-left-color:#1879b9;background:#f6fbff}\n.causal-svg{max-height:540px}\n.combined-card .chart-zone{max-width:1100px}.combined-card .economic-svg{max-width:940px}\n@media(max-width:1400px){.workspace{grid-template-columns:minmax(270px,296px) minmax(0,1fr)}}\n@media(max-width:1090px){.workspace{grid-template-columns:268px minmax(0,1fr)}.rail{position:relative;top:0;height:auto;max-height:100vh}.topbar{height:auto;min-height:78px;flex-wrap:wrap;padding:12px 17px}.main{padding:20px 16px}}\n@media(max-width:740px){.workspace{display:block}.rail{position:relative;top:auto;height:auto;max-height:none;padding:16px;border-right:0;border-bottom:1px solid var(--border)}.preslist,.case-list{display:grid;grid-template-columns:repeat(auto-fit,minmax(215px,1fr));gap:6px}.pres-row,.case-row{min-width:0}.rail-label{display:block}.rail-note{display:block}.rail-actions{grid-template-columns:repeat(3,minmax(0,1fr))}.rail-wide{white-space:normal;font-size:11px}.causal-svg{min-width:850px}.causal-map{overflow-x:auto}}\n@media print{.causal-panel{background:#fff}.causal-svg{min-width:0}.rail{display:none!important}}\n\n/* Macro Studio 5.1 · Formulario y diagnóstico opcional; azul/rosa consistentes. */\n.formula-library{border:1px solid #cfdfeb;border-radius:14px;background:#f9fcff;margin:12px 0 18px;color:#27475e;overflow:hidden}\n.formula-library summary{list-style:auto;cursor:pointer;padding:17px 19px;font-weight:900;font-size:14px;color:#20577b;background:#edf6fc}\n.formula-library[open] summary{border-bottom:1px solid #d5e4ef}\n.formula-intro,.formula-legend{margin:15px 18px 8px;line-height:1.65;color:#536f83;font-size:12px}\n.formula-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:13px;padding:12px 17px}\n.formula-block{border:1px solid #dce7ef;background:#fff;border-radius:10px;padding:12px 15px;min-width:0}\n.formula-block h4{font-size:14px;margin:0 0 7px;color:#193c56}\n.formula-main{font-size:14px;font-weight:900;color:#1477ad;padding:6px 0 10px}\n.formula-block p{font-size:12px;margin:6px 0;line-height:1.58;overflow-wrap:anywhere}\n.policy-panel{border:1px solid #c9ddea;border-radius:16px;background:linear-gradient(130deg,#fff,#f3f9fe);padding:20px;margin:14px 0 22px}\n.policy-heading h3{font-size:19px;margin:5px 0;color:#1b3a51}\n.policy-heading p{font-size:12.5px;line-height:1.65;color:#597489;max-width:970px}\n.policy-layout{display:grid;grid-template-columns:minmax(0,1.24fr) minmax(285px,.75fr);gap:17px;margin-top:15px}\n.policy-graph{border:1px solid #d7e5ef;background:#fff;border-radius:12px;overflow:hidden;padding:8px}\n.policy-svg{display:block;width:100%;height:auto;max-height:390px;cursor:crosshair}\n.policy-pin{cursor:pointer}.policy-pin circle{fill:#fff;stroke:#2381b5;stroke-width:2.5}.policy-pin text{font-size:13px;font-weight:900;fill:#1477a9;pointer-events:none}.policy-pin.active circle{fill:#e5f3fb;stroke:#c93e85;stroke-width:3.3}.policy-pin.active text{fill:#b62f73}\n.policy-curve{font-size:15px;font-weight:900;paint-order:stroke;stroke:#fff;stroke-width:4px}\n.policy-chart-note{font-size:11.5px;color:#648096;padding:0 8px;line-height:1.5}\n.policy-reading{background:#fff;border:1px solid #d7e5ef;border-radius:12px;padding:16px;align-self:start;color:#274359}\n.policy-choices{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:12px}.policy-choices .button{min-width:76px}\n.policy-reading h4{font-size:15px;margin:8px 0}.policy-reading p{font-size:12.5px;line-height:1.6;margin:7px 0}\n.policy-recommendation{margin:12px 0;padding:12px;background:#f4faff;border-left:4px solid #1879b9;border-radius:7px}\n.policy-recommendation strong{font-size:13px}.policy-note{display:block;margin-top:11px;line-height:1.55;color:#5b7489}\n@media(max-width:950px){.policy-layout{grid-template-columns:1fr}.formula-grid{grid-template-columns:1fr}}\n@media print{.formula-library:not([open]){display:none}.policy-panel{break-inside:avoid;page-break-inside:avoid}.policy-layout{grid-template-columns:1fr 1fr}}\n\n.market-formula{margin:0 17px 5px;padding:9px 13px;border:1px solid #e0eaf2;background:#f6fbfe;border-radius:8px;color:#446982;font-size:11.5px;line-height:1.55;font-weight:750}\n"

In [ ]:
MOTOR_ECONOMICO_JS = "/* Macro Studio · engine 0.1 · pure deterministic rules, no external libraries.\n * Signs are qualitative, not empirical elasticities. All functions can be tested without the UI.\n */\n'use strict';\nconst MS = (()=>{\n const MARKETS={goods:'Bienes y servicios (Y)',labor:'Mercado laboral (W, L)',money:'Mercado monetario (i)',fx:'Mercado cambiario (TC)'};\n const CURVES={goods:['da'],labor:['ol','dl'],money:['ms','md'],fx:['os','ds']};\n const CATALOG=[\n // Goods market: effect is change in planned aggregate expenditure.\n ['gasto_publico','Gasto público (política fiscal)','goods','da',1,'Política fiscal → G → DA','G','Gasto público: aumento expansivo, disminución restrictiva. No es emisión monetaria del BCCR.'],\n ['infra','Inversión pública en infraestructura','goods','da',1,'G: infraestructura','G','Gasto público destinado a infraestructura.'],\n ['compras','Compras del sector público','goods','da',1,'G: compras','G','Adquisiciones públicas de bienes y servicios.'],\n ['salarios_publicos','Gasto del gobierno en salarios','goods','da',1,'G: salarios públicos','G','Componente directo del gasto gubernamental.'],\n ['pensiones','Gasto público en pensiones','goods','da',1,'G: pensiones/transferencias','G','Simplificación del curso: aumento del gasto/ingreso disponible.'],\n ['transferencias','Transferencias a hogares','goods','da',1,'Yd → C','C','Transferencias que modifican ingreso disponible y consumo.'],\n ['impuestos','Impuestos a los hogares','goods','da',-1,'T → Yd → C','C','Más impuestos reducen ingreso disponible, ceteris paribus.'],\n ['consumo_aut','Consumo autónomo','goods','da',1,'C₀','C','Cambio del consumo autónomo.'],\n ['pmc','Propensión marginal a consumir','goods','da',1,'PmgC → C','C','Mayor propensión marginal a consumir, para ingreso disponible positivo.'],\n ['inversion_aut','Inversión privada autónoma','goods','da',1,'I autónoma','I','Cambio de inversión no atribuido a la tasa de interés.'],\n ['credito','Acceso de empresas al crédito','goods','da',1,'Crédito → I','I','Facilidad de crédito que aumenta inversión.'],\n ['tramitologia','Obstáculos administrativos a inversión','goods','da',-1,'Trámites → I','I','Tramitología o barreras reducen inversión.'],\n ['tasas_nacionales','Tasa de interés nacional (cambio exógeno)','goods','da',-1,'i → I','I','Cambio de i tomado como shock externo al gráfico IS, como en los ejercicios.'],\n ['exportaciones','Exportaciones autónomas','goods','da',1,'X','X','Más exportaciones elevan demanda agregada (efecto cambiario paralelo no cuantificado).'],\n ['importaciones','Importaciones autónomas','goods','da',-1,'M','M','Más importaciones reducen exportaciones netas; también implican dólares.'],\n ['ingreso_socios','Ingreso de socios comerciales','goods','da',1,'Y* → X','X','Más ingreso externo tiende a elevar exportaciones.'],\n ['tipo_cambio_exo','Tipo de cambio (cambio exógeno)','goods','da',1,'TC → X−M','X−M','TC definido como colones por dólar: TC↑ implica depreciación y X−M↑ bajo el supuesto del curso.'],\n ['salario_exo','Salarios exógenos de hogares','goods','da',1,'W → Yd → C','C','Shock sobre salarios ya dado, sin postular una curva laboral inicial.'],\n // Labor supply and labor demand.\n ['migracion_entrada','Inmigración laboral','labor','ol',1,'Migración → OL','OL','Entrada de trabajadores eleva la oferta laboral.'],\n ['migracion_salida','Emigración laboral','labor','ol',-1,'Migración → OL','OL','Salida de trabajadores reduce la oferta laboral.'],\n ['pea','Participación en la fuerza laboral','labor','ol',1,'PEA → OL','OL','Más participación incrementa oferta de trabajo.'],\n ['educacion','Disponibilidad de trabajadores capacitados','labor','ol',1,'Educación → OL','OL','Aumento de oferta de trabajo capacitado.'],\n ['desastre_laboral','Desastre que afecta disponibilidad laboral','labor','ol',-1,'Desastre → OL','OL','Reducción de la oferta de trabajo.'],\n ['tecnologia_sust','Tecnología que desplaza mano de obra a nivel nacional','labor','dl',-1,'Tecnología sustitutiva → DL','DL','El enunciado docente ya describe sustitución laboral; no se interpreta como tecnología complementaria.'],\n ['tecnologia_comp','Tecnología complementaria de mano de obra','labor','dl',1,'Tecnología → DL','DL','Tecnología complementaria eleva la demanda de ciertas ocupaciones.'],\n ['demanda_trabajo','Demanda autónoma de trabajo','labor','dl',1,'DL autónoma','DL','Cambio exógeno de demanda laboral.'],\n // Money market.\n ['expansion_monetaria','Inyección de liquidez del BCCR','money','ms',1,'BCCR → Ms','Ms','Aumento de oferta monetaria.'],\n ['contraccion_monetaria','Retiro de liquidez del BCCR','money','ms',-1,'BCCR → Ms','Ms','Reducción de oferta monetaria.'],\n ['encaje','Encaje mínimo legal','money','ms',-1,'EML → Ms','Ms','Un encaje mayor reduce oferta de dinero.'],\n ['bems_compra','Compra de bonos por el banco central','money','ms',1,'OMA compra → Ms','Ms','Compra de BEMs expande liquidez.'],\n ['bems_venta','Venta de bonos por el banco central','money','ms',-1,'OMA venta → Ms','Ms','Venta de BEMs contrae liquidez.'],\n ['tpm','Tasa de política monetaria / redescuento','money','ms',-1,'TPM → Ms','Ms','Convención simplificada usada en la materia.'],\n ['dinero_precaucion','Demanda de dinero por precaución','money','md',1,'Precaución → Md','Md','Mayor preferencia de liquidez.'],\n ['deuda_interna','deuda interna del gobierno','money','md',1,'Deuda interna → Md','Md','Relación pedagógica presentada en macro I.'],\n // FX: exchange rate quoted in local units per USD.\n ['ied','Inversión extranjera directa entrante','fx','os',1,'IED → O$','O$','Entrada de divisas eleva su oferta.'],\n ['turismo_entrada','Turismo extranjero receptor','fx','os',1,'Turismo receptor → O$','O$','Turistas traen divisas.'],\n ['remesas','Ingresos de remesas del exterior','fx','os',1,'Remesas → O$','O$','Más divisas entrantes.'],\n ['prestamos_externos','Préstamos provenientes del exterior','fx','os',1,'Crédito exterior → O$','O$','Préstamos desde afuera aumentan oferta de dólares.'],\n ['tasas_externas','Tasa de interés internacional','fx','os',-1,'i* → O$','O$','Tasas externas menores pueden aumentar el atractivo de invertir localmente.'],\n ['flujos_portafolio','Atractivo de activos locales','fx','os',1,'Capitales → O$','O$','Afluencia financiera de dólares.'],\n ['viajes_salida','Turismo de residentes en el exterior','fx','ds',1,'Turismo emisor → D$','D$','Mayor gasto de residentes fuera incrementa demanda de dólares.'],\n ['capital_salida','Inversión de residentes en el exterior','fx','ds',1,'Salida de capital → D$','D$','Más inversiones externas aumentan demanda de divisas.'],\n ['prestamos_al_exterior','Préstamos nacionales hacia el exterior','fx','ds',1,'Crédito saliente → D$','D$','Egreso de dólares.']\n ].map(x=>({id:x[0],label:x[1],market:x[2],curve:x[3],effect:x[4],channel:x[5],variable:x[6],description:x[7]}));\n const DEFAULT_SHOCKS=CATALOG;\n const S=v=>v>0?1:v<0?-1:0;\n const arrow=v=>v>0?'↑':v<0?'↓':'↔';\n const direction=v=>v>0?'aumenta':v<0?'disminuye':'no cambia de manera determinada';\n const curveNames={da:'DA',ol:'OL',dl:'DL',ms:'Ms',md:'Md',os:'O$',ds:'D$'};\n function describe(sign,name){return `${name} ${arrow(sign)}`;}\n function opposingChoice(channels){return channels.some(c=>c.sign>0)&&channels.some(c=>c.sign<0);}\n function choiceOf(spec){return spec.balanceChoice==='investment'?'investment':'consumption';}\n function balanceDirection(channels,choice,fallback){return opposingChoice(channels)?S((choice==='investment'?channels[1]:channels[0]).sign):fallback;}\n function rule(markets,stage,market,curve,sign,why,round='primer orden'){\n  if(!sign) return;\n  const ev={stage,market,curve,sign,why,round};\n  markets[market].push(ev);return ev;\n }\n function analyze(spec,extra=[]){\n  const shock=[...DEFAULT_SHOCKS,...extra].find(s=>s.id===spec.shockId)||DEFAULT_SHOCKS[0];\n  const dir=shock.id==='tecnologia_sust'?1:(spec.direction===-1?-1:1);const initial=S(shock.effect*dir);\n  const dom=choiceOf(spec); // El docente permite escoger qué canal predomina al existir efectos contrapuestos.\n  const changes={goods:[],labor:[],money:[],fx:[]};const steps=[];\n  let y=0,w=0,ie=0,tc=0,emp=0,stageD=0,base=0;\n  const stage=(n,title,text,market,round)=>steps.push({n,title,text,market,round});\n  const insert=(n,m,curve,sign,why,round)=>rule(changes,n,m,curve,sign,why,round);\n  const origin=shock.id==='tecnologia_sust'?'La tecnología desplaza mano de obra a nivel nacional':`${shock.label} ${arrow(dir)}`;\n  stage(1,`Shock inicial · ${MARKETS[shock.market]}`,`${origin}. Canal explicativo: ${shock.channel}.`,shock.market,'primero');\n  insert(1,shock.market,shock.curve,initial,`Variable exógena: ${origin}`,'primero');\n  const inspect=[];\n  let channels=[];let stageFeedback=0;let opposedAt=null;\n  if(shock.market==='goods'){\n    y=initial; w=y; emp=y; ie=y; tc=y;\n    insert(2,'labor','dl',y,'Producción Y → demanda laboral DL','primero');\n    insert(2,'money','md',y,'Producción Y → demanda de dinero para transacciones Md','primero');\n    insert(2,'fx','ds',y,'Y modifica importaciones y demanda de dólares D$','primero');\n    stage(2,'La producción modifica los mercados',`Y ${arrow(y)} → DL ${arrow(y)}, Md ${arrow(y)} y D$ ${arrow(y)} simultáneamente.`, 'goods','primero');\n    channels=[{name:'Salarios → consumo',sign:y},{name:'Tasa de interés → inversión',sign:-ie}];\n    stageD=balanceDirection(channels,dom,y); opposedAt=3;\n    insert(3,'goods','da',stageD,`Retorno C/I: se desarrolla ${dom==='investment'?'inversión predominante':'consumo predominante'}`,'segundo');\n    stage(3,'Retorno a bienes y servicios',`W ${arrow(w)} → C ${arrow(w)}; i ${arrow(ie)} → I ${arrow(-ie)}; predomina el efecto de ${dom==='investment'?'inversión (I)':'consumo (C)'}; DA ${arrow(stageD)}.`, 'goods','segundo');\n    insert(4,'labor','dl',stageD,'Segundo cambio en producción → demanda laboral DL','segundo');\n    insert(4,'money','md',stageD,'Segundo cambio en producción → demanda monetaria Md','segundo');\n    stage(4,'Segunda reacción de laboral y monetario',`La nueva Y ${arrow(stageD)} vuelve a modificar DL y Md. Se detiene la propagación antes de tercer orden.`, 'labor','segundo');\n    inspect.push(`Efectos contrapuestos en etapa 3: se escogió predominio de ${dom==='investment'?'inversión (I)':'consumo (C)'}. Este es un supuesto declarado, no una magnitud estimada.`);\n  } else if(shock.market==='labor'){\n    w=shock.curve==='ol'?-initial:initial;emp=initial;y=w;ie=y;tc=y;\n    insert(2,'goods','da',y,'Salarios W → ingreso disponible Yd → consumo C → DA','primero');\n    stage(2,'Salarios modifican consumo y producción',`W ${arrow(w)} → C ${arrow(w)} → DA y Y ${arrow(y)}.`, 'goods','primero');\n    insert(3,'labor','dl',y,'Y modifica demanda laboral DL (segundo desplazamiento laboral)','segundo');\n    insert(3,'money','md',y,'Y modifica Md por motivo transacciones','primero');\n    insert(3,'fx','ds',y,'Y modifica demanda de dólares por importaciones','primero');\n    stage(3,'Respuesta simultánea de mercados',`Y ${arrow(y)} → DL ${arrow(y)}, Md ${arrow(y)} y D$ ${arrow(y)}. DL es de segundo orden; Md es de primero.`, 'labor','mixto');\n    channels=[{name:'Salarios → consumo',sign:w},{name:'Interés → inversión',sign:-ie}];\n    stageD=balanceDirection(channels,dom,y); opposedAt=4;\n    insert(4,'goods','da',stageD,`Retorno desde W e i: predomina ${dom==='investment'?'inversión':'consumo'}`,'segundo');\n    stage(4,'Segundo desplazamiento de demanda agregada',`W ${arrow(w)} → C ${arrow(w)}; i ${arrow(ie)} → I ${arrow(-ie)}. Predomina ${dom==='investment'?'I':'C'}; Y ${arrow(stageD)}.`, 'goods','segundo');\n    insert(5,'money','md',stageD,'Nueva variación Y → nuevo cambio en Md','segundo');\n    stage(5,'Se completa el segundo orden monetario',`El cambio de Y derivado del efecto 4 produce Md ${arrow(stageD)} (efecto 5). No se siguen efectos de tercer orden.`, 'money','segundo');\n    inspect.push(`En la etapa 4 se eligió ${dom==='investment'?'inversión':'consumo'} predominante al comparar C e I; la propagación posterior utiliza el signo elegido.`);\n  } else if(shock.market==='money'){\n    ie=shock.curve==='ms'?-initial:initial;y=-ie;tc=-ie;w=y;emp=y;\n    insert(2,'goods','da',y,'Tasa de interés de equilibrio ie → inversión I → demanda agregada DA','primero');\n    insert(2,'fx','os',ie,'Interés nacional altera oferta de dólares (opcional, no evaluado aún)','primero');\n    stage(2,'Tasa de interés cambia inversión y producción',`i ${arrow(ie)} → I ${arrow(y)} → DA y Y ${arrow(y)}. El cambiario se conserva como complemento opcional.`, 'goods','primero');\n    insert(3,'labor','dl',y,'Producción Y → demanda laboral DL','primero');\n    insert(3,'money','md',y,'Producción Y → demanda monetaria transaccional Md; efecto menor que el shock inicial','segundo');\n    insert(3,'fx','ds',y,'Producción Y → importaciones y demanda de dólares (opcional)','segundo');\n    stage(3,'Dos efectos simultáneos de distinta orden',`Y ${arrow(y)} → DL ${arrow(y)} (primer orden) y Md ${arrow(y)} (segundo orden). Los salarios y la tasa de interés reaccionan simultáneamente.`, 'money','mixto');\n    channels=[{name:'Salarios → consumo',sign:w},{name:'Tasa de interés de rebote → inversión',sign:-y}];\n    stageD=balanceDirection(channels,dom,y); opposedAt=4;\n    insert(4,'goods','da',stageD,`Salarios modifican C y rebote de interés modifica I; predomina ${dom==='investment'?'inversión':'consumo'}`,'segundo');\n    stage(4,'Consumo e inversión afectan DA simultáneamente',`W ${arrow(w)} → C ${arrow(w)}; rebote de i ${arrow(y)} → I ${arrow(-y)}. Se elige predominio de ${dom==='investment'?'I':'C'} y Y ${arrow(stageD)}.`, 'goods','segundo');\n    insert(5,'labor','dl',stageD,'Y resultante del paso 4 → DL y salarios','segundo');\n    stage(5,'Se completa el segundo orden laboral',`Nuevo cambio de Y ${arrow(stageD)} → DL ${arrow(stageD)}. Se detiene antes de efectos de tercer orden.`, 'labor','segundo');\n    inspect.push('Caso monetario contrastado con la clase del 20 de octubre: secuencia 1,2,3 y 3,4,5. El efecto monetario de etapa 3 es un rebote MENOR que el shock inicial.');\n    inspect.push(`C e I producen signos opuestos: la alternativa elegida es ${dom==='investment'?'inversión':'consumo'} predominante; no se trata de un resultado estimado.`);\n  } else if(shock.market==='fx'){\n    tc=shock.curve==='os'?-initial:initial;y=tc;w=y;emp=y;ie=y;\n    insert(2,'goods','da',tc,'TC altera X−M y la DA','primero');\n    stage(2,'Cambio de exportaciones netas',`TC ${arrow(tc)} → X−M ${arrow(tc)} → Y ${arrow(y)}.`, 'goods','primero');\n    insert(3,'labor','dl',y,'Y → DL','primero');\n    insert(3,'money','md',y,'Y → Md','primero');\n    insert(3,'fx','ds',y,'Y → importaciones → D$','segundo');\n    stage(3,'Efectos simultáneos por Y',`Y ${arrow(y)} afecta DL, Md y D$.`, 'fx','mixto');\n    stageD=y;\n    insert(4,'goods','da',stageD,'Retroalimentación de salarios, interés y TC; supuesto cualitativo','segundo');\n    stage(4,'Segundo desplazamiento de DA',`W ${arrow(w)}, i ${arrow(ie)} y TC ${arrow(tc)} regresan a DA.`, 'goods','segundo');\n    inspect.push('Mercado cambiario conservado como complemento opcional. Su ruta de segundo orden aún no forma parte de la secuencia de tres mercados confirmada en esta clase.');\n  }\n  // In qualitative accounting, the sign of an induced variable only refers to its immediate equilibrium.\n  if(channels.some(v=>v.sign>0) && channels.some(v=>v.sign<0)){\n   inspect.unshift(`Elección docente explícita de efectos contrapuestos: se ha desarrollado ${dom==='investment'?'predominio de inversión':'predominio de consumo'}. Las magnitudes reales no se conocen.`);\n  }\n  // Aplicación uniforme para TODOS los casos: DA modifica IS; Ms modifica LM.\n  // El efecto de segundo orden se representa con una amplitud menor al primero.\n  const goodsEvents=changes.goods;\n  const daNet=goodsEvents.reduce((acc,e)=>acc+(e.round==='segundo'?0.34:1)*e.sign,0);\n  // Regla explícita de Ronney 1:56:17: IS por C/G/X/M/I autónoma;\n  // la inversión inducida únicamente por i NO desplaza IS.\n  // Registrar C vía salarios en cada etapa pertinente, inclusive si C e I\n  // tienen signos contrapuestos. Una caída de DA por I(i) no es shock IS.\n  const isDrivers=[];\n  const eligible=(stage,sign,cause,weight)=>{if(sign && (spec.showSecond!==false || weight===1))isDrivers.push({stage,sign,cause,weight});};\n  if(shock.market==='goods'){\n    if(shock.id!=='tasas_nacionales')eligible(1,initial,shock.variable==='I'?'inversión autónoma':shock.variable||'demanda autónoma',1);\n    eligible(3,w,'salarios → ingreso disponible → consumo',0.34);\n  }else if(shock.market==='labor'){\n    eligible(2,w,'primer cambio salarial → consumo',1);\n    eligible(4,y,'segundo cambio salarial tras DL → consumo',0.34);\n  }else if(shock.market==='money'){\n    eligible(4,w,'salarios después del cambio laboral → consumo',0.34);\n  }else if(shock.market==='fx'){\n    eligible(2,tc,'tipo de cambio → exportaciones netas',1);\n    eligible(4,w,'salarios → consumo',0.34);\n  }\n  const isSign=S(isDrivers.reduce((sum,d)=>sum+d.sign*d.weight,0));\n  const laborStart=changes.labor[0]?.sign||0;\n  const dominant={employment:emp,wage:w,interest:ie,exchange:tc,output:y,inflation:y>0?'presión alcista':y<0?'sin presión inflacionaria adicional':'indeterminada'};\n  const hasOpposing=opposingChoice(channels)&&shock.market!=='fx';\n  const balance={applicable:hasOpposing,choice:dom,stage:hasOpposing?opposedAt:null,channels,net:hasOpposing?stageD:0,\n    note:hasOpposing?`En la etapa ${opposedAt}, C ${arrow(channels[0].sign)} e I ${arrow(channels[1].sign)} tienen signos contrarios. Se eligió predominio de ${dom==='investment'?'inversión (I)':'consumo (C)'}, por lo que DA ${arrow(stageD)}. Los demás efectos se recalculan a partir de este resultado.`:'No hay una elección docente C/I pertinente en esta ruta.'};\n  // Numbered case explanatory paragraphs use author's course style but never invent empirical numbers.\n  const explanations=generateExplanations({shock,dir,changes,y,w,ie,tc,emp,stageD,dom,channels,steps,isSign});\n  const audit=[{n:1,rule:'Shock exógeno',source:shock.channel,detail:`${shock.label} ${arrow(dir)} → ${curveNames[shock.curve]} ${arrow(initial)}`},\n    ...changes.goods.map(c=>({n:c.stage,rule:'Bienes/servicios',source:'Y = C + I + G + X − M',detail:`DA ${arrow(c.sign)} · ${c.why}`})),\n    ...changes.labor.map(c=>({n:c.stage,rule:'Laboral',source:'DL = f(Y, tecnología); OL = f(migración, educación…)',detail:`${curveNames[c.curve]} ${arrow(c.sign)} · ${c.why}`})),\n    ...changes.money.map(c=>({n:c.stage,rule:'Monetario',source:'Ms = f(BCCR); Md = f(Y, precaución…)',detail:`${curveNames[c.curve]} ${arrow(c.sign)} · ${c.why}`})),\n    ...changes.fx.map(c=>({n:c.stage,rule:'Cambiario',source:'O$ = f(IED, i−i*…); D$ = f(M, turismo…)',detail:`${curveNames[c.curve]} ${arrow(c.sign)} · ${c.why}`})),\n    {n:'IS',rule:'IS por demanda agregada',source:'DA = C + I + G + X − M',detail:`${isSign>0?'IS a la derecha':isSign<0?'IS a la izquierda':'IS sin desplazamiento'}; el equilibrio conjunto IS–LM se determina en el diagrama combinado, cuando el shock está identificado.`}\n  ].sort((a,b)=>Number(a.n)-Number(b.n));\n  // Regla operativa documentada de la clase: únicamente los cambios de Ms\n  // desplazan LM en estos ejercicios. Los cambios inducidos de Md por Y\n  // son movimientos sobre LM. No atribuir un cambio de Ms al shock de deuda.\n  // La diapositiva 26 también comenta saldos reales Ms/P; no hay un shock P\n  // parametrizado en el catálogo actual.\n  const mdInitial=shock.market==='money'&&shock.curve==='md';\n  const interestGiven=shock.id==='tasas_nacionales';\n  const islm={is:isSign,isDrivers,lm:(shock.market==='money'&&shock.curve==='ms')?initial:0,\n    evidence:mdInitial?'money-demand-course-rule':interestGiven?'interest-origin-unspecified':'course-rule',\n    note:''};\n  const sequence=Object.entries(changes).flatMap(([market,evs])=>evs.map(e=>({...e,market}))).filter(e=>e.market!=='fx').sort((a,b)=>a.stage-b.stage);\n  return {shock,dir,initial,changes,steps,sequence,islm,channels,inspect,dominant,stageD,isSign,explanations,audit,origin,balance};\n }\n function generateExplanations(c){\n  const {shock,dir,changes,y,w,ie,tc,stageD,dom}=c;\n  let out={};\n  for(const [market,evs] of Object.entries(changes)){\n   const arr=[];\n   for(let k=0;k<evs.length;k++){\n    const e=evs[k],sign=e.sign,seq=k+1;let texts=[];\n    if(market==='goods'){\n      const isStart=e.stage===1 && shock.market==='goods';\n      const causal=isStart?`Al ${dir===1?'aumentar':'disminuir'} ${shock.label.toLowerCase()}, la Demanda Agregada ${direction(sign)} porque afecta ${shock.channel} en DA = C + I + G + X − M.`:\n        `Debido a los resultados simultáneos de los mercados anteriores, los salarios modifican C y la tasa de interés modifica I dentro de DA = C + I + G + X − M. Se eligió predominio de ${dom==='investment'?'inversión':'consumo'}, por lo que la demanda agregada ${direction(sign)} (${e.why}).`;\n      texts=[causal,`Al nivel de producción de equilibrio anterior, la ${sign>0?'DA es mayor que la OA y se genera exceso de demanda agregada':'OA es mayor que la DA y se genera exceso de oferta agregada'}.`,`Los inventarios de los productores se ${sign>0?'empiezan a agotar, hay presión para que suban los precios e incentivos para aumentar':'empiezan a acumular, hay presión para que bajen los precios e incentivos para disminuir'} la producción.`,`Se alcanza un nuevo equilibrio con un nivel de producción ${sign>0?'mayor y presión inflacionaria':'menor y sin presión inflacionaria adicional'}.`];\n    }else if(market==='labor'){\n      const ctype=e.curve==='ol'?'oferta':'demanda';const wl=e.curve==='ol'?-sign:sign,employment=e.curve==='ol'?sign:sign;\n      texts=[`La ${ctype} de trabajo (${curveNames[e.curve]}) ${direction(sign)}. ${e.why}.`,`Al salario de equilibrio anterior la ${ctype==='demanda'?(sign>0?'cantidad demandada excede la ofrecida':'cantidad ofrecida excede la demandada'):(sign>0?'cantidad ofrecida excede la demandada':'cantidad demandada excede la ofrecida')} de trabajo.`,`Se presenta presión para que el salario ${direction(wl)}; los agentes ajustan las cantidades ofrecidas y demandadas de trabajo.`,`Hay un nuevo equilibrio con salario ${wl>0?'mayor':'menor'} y nivel de empleo ${employment>0?'mayor':'menor'}.`];\n    }else if(market==='money'){\n      const i=e.curve==='ms'?-sign:sign;\n      texts=[`La ${e.curve==='ms'?'oferta':'demanda'} de dinero (${curveNames[e.curve]}) ${direction(sign)}. ${e.why}.`,`A la tasa de interés anterior hay exceso de ${i>0?'demanda':'oferta'} de dinero.`,`Hay presión para que la tasa de interés ${direction(i)}.`,`Se establece un equilibrio monetario de referencia con tasa de interés ${i>0?'mayor':'menor'}, manteniendo los otros determinantes constantes.`];\n    }else{\n      const t=e.curve==='os'?-sign:sign;\n      texts=[`La ${e.curve==='os'?'oferta':'demanda'} de dólares (${curveNames[e.curve]}) ${direction(sign)}. ${e.why}.`,`Al tipo de cambio anterior hay exceso de ${t>0?'demanda':'oferta'} de dólares.`,`Hay presión para que el tipo de cambio ${direction(t)}.`,`Se obtiene un nuevo equilibrio cambiario de referencia con tipo de cambio ${t>0?'mayor':'menor'}, ceteris paribus.`];\n    }\n    arr.push({event:e,paragraphs:texts});\n   }\n   out[market]=arr;\n  }\n  out.is=`IS ${c.isSign>0?'se desplaza a la derecha':c.isSign<0?'se desplaza a la izquierda':'permanece fija'} por los componentes de DA distintos de la inversión inducida por i. LM solo cambia por Ms del Banco Central.`;\n  return out;\n }\n return {MARKETS,CURVES,CATALOG,curveNames,analyze,arrow,direction,S};\n})();\n\n\n/* Macro Studio 5.0 · Rutas docentes trazables. Sin dependencias externas.\n   No se suman elasticidades inexistentes: signo neto cuando coexisten efectos\n   contrapuestos responde a un supuesto explícito del ejercicio. */\n(function(){\n'use strict';\nconst mName={goods:'Bienes y servicios',labor:'Laboral',money:'Monetario',fx:'Cambiario'};\nconst N={goods:'Y',labor:'W',money:'i',fx:'TC'};\nconst sign=x=>x>0?1:x<0?-1:0;\nconst arrow=x=>x>0?'↑':x<0?'↓':'↔';\nconst verb=x=>x>0?'aumenta':x<0?'disminuye':'permanece constante';\nconst plural=x=>x>0?'aumentan':x<0?'disminuyen':'no varían';\nconst clean=s=>String(s||'').trim();\nconst fmt=(m,s)=>m+':'+s;\nconst seen=(xs,id)=>xs.some(x=>x.id===id);\nfunction analyze(spec={},extra=[]){\n  const sh=[...MS.CATALOG,...extra].find(s=>s.id===spec.shockId)||MS.CATALOG[0];\n  const dir=sh.id==='tecnologia_sust'?1:(spec.direction===-1?-1:1), inc=sign(dir*sh.effect);\n  const choice=['consumption','investment','exchange'].includes(spec.balanceChoice)?spec.balanceChoice:'consumption';\n  const routeMode=(sh.id==='ied'&&spec.routeMode==='dual')?'dual':'cambiario';\n  const dualChoice=spec.dualChoice==='investment'?'investment':'exchange';\n  const changes={goods:[],labor:[],money:[],fx:[]}, edges=[], stages=[];\n  let balance=null, isOverride=null, bp=0, bpWhy='', wage=0, interest=0, tc=0, y=0;\n  function add(m,stage,curve,v,why,deps=[],kind='induced',eligible=false){\n    v=sign(v);if(!v)return null;\n    const id=fmt(m,stage);if(changes[m].some(e=>e.id===id))throw new Error('Secuencia duplicada '+id+' '+sh.id);\n    const round=changes[m].length?'segundo':'primero';\n    const ev={id,stage,market:m,curve,sign:v,why:clean(why),round,kind,eligible,parents:deps};\n    changes[m].push(ev);\n    for(const parent of deps){\n      const stageParent=Number(parent.split(':')[1]);\n      if(!Number.isInteger(stageParent)||stageParent>=stage)throw new Error('Dependencia causal sin avance temporal: '+parent+' → '+id+' ('+sh.id+')');\n      edges.push({from:parent,to:id,variable:parent.split(':')[0],why:ev.why});\n    }\n    return ev;\n  }\n  const step=(n,msg)=>stages.push({n,title:'Secuencia '+n,text:msg,round:n<3?'primero':'segundo'});\n  const moneyI=x=>x; // aumento de Md sube i; aumento de Ms baja i\n  function balanceFor(stage,sources,parents){\n    const signs=sources.filter(s=>s.sign).map(s=>s.sign),opposing=signs.some(x=>x>0)&&signs.some(x=>x<0);\n    const candidate=sources.find(x=>x.key===choice)||sources[0];\n    const net=opposing?candidate.sign:(signs[0]||0);\n    balance={applicable:opposing,stage,channels:sources.map(x=>({...x})),choice,net,selected:candidate.key,\n      note:sources.map(x=>`${x.label}: ${verb(x.sign)} (${arrow(x.sign)})`).join('; ')+'; '+(opposing?'por el supuesto elegido, ':'')+'DA '+arrow(net),parents};\n    return net;\n  }\n  function goodsShock(){\n    const cause={\n      gasto_publico:'La política fiscal modifica directamente el gasto público G',infra:'La inversión pública en infraestructura modifica G',compras:'Las compras gubernamentales modifican G',\n      salarios_publicos:'Los salarios del sector público forman parte de G',pensiones:'Las pensiones y transferencias modifican el ingreso disponible y el consumo',\n      transferencias:'Las transferencias cambian el ingreso disponible Yd y C',impuestos:'Los impuestos cambian inversamente Yd y el consumo C',\n      consumo_aut:'El consumo autónomo modifica C',pmc:'La propensión marginal a consumir modifica C',\n      inversion_aut:'La inversión autónoma modifica I sin un cambio previo de i',credito:'El acceso al crédito modifica I autónomamente',\n      tramitologia:'Las trabas administrativas modifican la inversión autónoma',\n      tipo_cambio_exo:'El tipo de cambio dado exógenamente modifica X−M',salario_exo:'Los salarios exógenos modifican el ingreso disponible y C',\n    }[sh.id]||`${sh.label} modifica ${sh.channel} de DA`;\n    add('goods',1,'da',inc,`${cause}. DA ${verb(inc)} (DA1→DA2) dentro de DA=C+I+G+X−M.`,[],'autonomous',sh.id!=='tasas_nacionales');\n    add('labor',2,'dl',inc,`Debido a que la producción nacional Y ${verb(inc)} en bienes y servicios (secuencia 1), la demanda laboral DL ${verb(inc)}, por lo que los salarios ${plural(inc)}.`,['goods:1']);\n    add('money',2,'md',inc,`Debido a que Y ${verb(inc)} en bienes y servicios (secuencia 1), las transacciones y la demanda de dinero Md ${plural(inc)}, por lo que la tasa de interés ${verb(inc)}.`,['goods:1']);\n    add('fx',2,'ds',inc,`Debido a que Y ${verb(inc)} en bienes y servicios (secuencia 1), las importaciones y la demanda de dólares D$ ${plural(inc)}; el tipo de cambio ${verb(inc)}.`,['goods:1']);\n    const opts=[{key:'consumption',label:'Salarios → consumo C',sign:inc},{key:'investment',label:'Tasa de interés → inversión I',sign:-inc},{key:'exchange',label:'Tipo de cambio → X−M',sign:inc}];\n    const net=balanceFor(3,opts,['labor:2','money:2','fx:2']);\n    add('goods',3,'da',net,`En la secuencia 2 los salarios ${plural(inc)} y C ${verb(inc)}; la tasa de interés ${verb(inc)} y la inversión I ${verb(-inc)}; el tipo de cambio ${verb(inc)} y X−M ${verb(inc)}. ${balance.note}.`,['labor:2','money:2','fx:2'],'net',choice!=='investment');\n    add('labor',4,'dl',net,`Como la producción nacional volvió a ${verb(net)} por el efecto 3 de bienes y servicios, DL ${verb(net)} y los salarios ${plural(net)}.`,['goods:3']);\n    add('money',4,'md',net,`Como la producción nacional volvió a ${verb(net)} por el efecto 3 de bienes y servicios, la demanda de dinero Md ${verb(net)} y la tasa de interés ${verb(net)}.`,['goods:3']);\n    add('fx',4,'ds',net,`Como la producción nacional volvió a ${verb(net)} por el efecto 3 de bienes y servicios, las importaciones y D$ ${plural(net)}.`,['goods:3']);\n    step(1,`Cambio autónomo de DA (${arrow(inc)}).`);step(2,`La producción Y ${arrow(inc)} afecta simultáneamente DL, Md y D$.`);\n    step(3,`Retorno único a DA mediante C, I y X−M: ${balance.note}.`);step(4,`Nueva Y ${arrow(net)} produce el segundo efecto en DL, Md y D$.`);\n  }\n  function laborShock(){\n    const w=sh.curve==='ol'?-inc:inc;\n    add('labor',1,sh.curve,inc,`${sh.label}: ${sh.curve==='ol'?'la oferta de trabajo OL':'la demanda laboral DL'} ${verb(inc)}, por lo que el salario W ${verb(w)}.`,[],'autonomous');\n    add('goods',2,'da',w,`Debido a que los salarios W ${plural(w)} en el mercado laboral (secuencia 1), el ingreso disponible Yd y el consumo C ${plural(w)}, y la demanda agregada DA ${verb(w)}.`,['labor:1'],'autonomous',true);\n    add('labor',3,'dl',w,`La producción nacional Y ${verb(w)} en bienes y servicios (secuencia 2), por lo que DL ${verb(w)} y los salarios ${plural(w)}.`,['goods:2']);\n    add('money',3,'md',w,`La producción Y ${verb(w)} en bienes y servicios (secuencia 2), por lo que la demanda de dinero Md ${verb(w)} y la tasa de interés ${verb(w)}.`,['goods:2']);\n    add('fx',3,'ds',w,`La producción Y ${verb(w)} en bienes y servicios (secuencia 2), por lo que importaciones y D$ ${plural(w)} y TC ${verb(w)}.`,['goods:2']);\n    const opts=[{key:'consumption',label:'Salarios → C',sign:w},{key:'investment',label:'i → I',sign:-w},{key:'exchange',label:'TC → X−M',sign:w}];\n    const net=balanceFor(4,opts,['labor:3','money:3','fx:3']);\n    add('goods',4,'da',net,`En la secuencia 3 los salarios ${plural(w)} y C ${verb(w)}; la tasa de interés ${verb(w)} e I ${verb(-w)}; el tipo de cambio ${verb(w)} y X−M ${verb(w)}. ${balance.note}.`,['labor:3','money:3','fx:3'],'net',choice!=='investment');\n    add('money',5,'md',net,`Y ${verb(net)} por el efecto 4 en bienes y servicios, por lo que Md ${verb(net)} y la tasa de interés ${verb(net)} en segundo orden.`,['goods:4']);\n    add('fx',5,'ds',net,`Y ${verb(net)} por el efecto 4 en bienes y servicios, por lo que importaciones y D$ ${plural(net)} en segundo orden.`,['goods:4']);\n    step(1,`Oferta o demanda de trabajo ${arrow(inc)}, W ${arrow(w)}.`);step(2,`W ${arrow(w)} modifica consumo y Y ${arrow(w)}.`);\n    step(3,`Y ${arrow(w)} cambia DL, Md y D$ simultáneamente.`);step(4,`Los tres mecanismos afectan una sola DA: ${balance.note}.`);\n    step(5,`La respuesta final de Y ${arrow(net)} llega a Md y D$.`);\n  }\n  function fxShock(){\n    const tc0=sh.curve==='os'?-inc:inc, Y=tc0;\n    add('fx',1,sh.curve,inc,`${sh.label} ${arrow(dir)}: la ${sh.curve==='os'?'oferta de dólares O$':'demanda de dólares D$'} ${verb(inc)} y el tipo de cambio TC ${verb(tc0)}.`,[],'autonomous');\n    add('goods',2,'da',Y,`Debido a que TC ${verb(tc0)} en el mercado cambiario (secuencia 1), las exportaciones netas X−M ${plural(Y)}: las exportaciones ${plural(Y)} y las importaciones ${plural(-Y)}. Por ello DA ${verb(Y)}.`,['fx:1'],'autonomous',true);\n    add('labor',3,'dl',Y,`La producción Y ${verb(Y)} en bienes y servicios (secuencia 2), por lo que DL ${verb(Y)} y los salarios ${plural(Y)}.`,['goods:2']);\n    add('money',3,'md',Y,`La producción Y ${verb(Y)} en bienes y servicios (secuencia 2), por lo que Md ${verb(Y)} y la tasa de interés ${verb(Y)}.`,['goods:2']);\n    add('fx',3,'ds',Y,`La producción Y ${verb(Y)} en bienes y servicios (secuencia 2), por lo que las importaciones y la demanda de dólares D$ ${plural(Y)} y el TC recibe una segunda presión ${Y>0?'alcista':'bajista'}.`,['goods:2']);\n    const opts=[{key:'consumption',label:'W → C',sign:Y},{key:'investment',label:'i → I',sign:-Y},{key:'exchange',label:'TC → X−M',sign:Y}];\n    const net=balanceFor(4,opts,['labor:3','money:3','fx:3']);\n    add('goods',4,'da',net,`Los salarios ${plural(Y)} y el consumo C ${verb(Y)}; la tasa de interés ${verb(Y)} y la inversión I ${verb(-Y)}; el tipo de cambio ${verb(Y)} y X−M ${verb(Y)}. Los tres efectos 3 llegan al único efecto 4. ${balance.note}.`,['labor:3','money:3','fx:3'],'net',choice!=='investment');\n    add('labor',5,'dl',net,`Como Y ${verb(net)} tras el efecto 4 de bienes y servicios, DL ${verb(net)} y los salarios ${plural(net)}.`,['goods:4']);\n    add('money',5,'md',net,`Como Y ${verb(net)} tras el efecto 4 de bienes y servicios, la demanda de dinero Md ${verb(net)} y la tasa de interés ${verb(net)}.`,['goods:4']);\n    step(1,`Shock cambiario ${arrow(inc)}; TC ${arrow(tc0)}.`);step(2,`TC ${arrow(tc0)} modifica X−M, DA y Y ${arrow(Y)}.`);\n    step(3,`Y ${arrow(Y)} modifica DL, Md y D$; W, i y TC quedan definidos.`);\n    step(4,`Los tres efectos 3 apuntan al mismo efecto 4: ${balance.note}.`);step(5,`La segunda Y modifica DL y Md: ${arrow(net)}.`);\n    if(sh.curve==='os' && ['ied','tasas_externas','turismo_entrada','remesas','prestamos_externos','flujos_portafolio'].includes(sh.id)){\n      bp=-inc;bpWhy=`El shock inicial de ${sh.label} cambia directamente los ingresos de dólares (O$ ${arrow(inc)}).`;\n    } else if(sh.curve==='ds') {bp=inc;bpWhy='El shock inicial modifica directamente la salida de dólares.';}\n  }\n  function iedDual(){\n    // Dos entradas independientes válidas según la clase del 17/11.\n    // El neto global NO puede inferirse sin decidir qué canal domina.\n    const da0=inc,fx0=inc,tc0=-fx0;\n    add('goods',1,'da',da0,`En este modelo se SUPONE que la IED se incorpora a la inversión nacional I. Al disminuir la IED, la inversión autónoma I ${verb(da0)}, por lo que DA ${verb(da0)}.`,[],'autonomous',true);\n    add('fx',1,'os',fx0,`En paralelo, la IED disminuye la entrada de dólares: O$ ${verb(fx0)}, por lo que TC ${verb(tc0)}.`,[],'autonomous');\n    add('labor',2,'dl',da0,`El primer Y, derivado del canal inversión nacional de la secuencia 1, ${verb(da0)}; por eso DL y los salarios ${plural(da0)}.`,['goods:1']);\n    add('money',2,'md',da0,`El primer Y, derivado de la inversión nacional en la secuencia 1, ${verb(da0)}; Md y la tasa de interés ${plural(da0)}.`,['goods:1']);\n    add('fx',2,'ds',da0,`El primer Y del canal inversión nacional ${verb(da0)}; importaciones y D$ ${plural(da0)}.`,['goods:1']);\n    add('goods',2,'da',tc0,`Simultáneamente, O$ ${verb(fx0)} en el mercado cambiario (efecto 1), TC ${verb(tc0)} y X−M ${verb(tc0)}, por lo que DA recibe una presión ${tc0>0?'alcista':'bajista'}.`,['fx:1'],'autonomous',true);\n    add('labor',3,'dl',tc0,`La nueva Y del retorno cambiario (secuencia 2) ${verb(tc0)}; DL y salarios reciben una presión ${tc0>0?'alcista':'bajista'}.`,['goods:2']);\n    add('money',3,'md',tc0,`La nueva Y del retorno cambiario (secuencia 2) ${verb(tc0)}; Md y la tasa de interés reciben una presión ${tc0>0?'alcista':'bajista'}.`,['goods:2']);\n    bp=-fx0;bpWhy='IED cae: ingreso autónomo de dólares disminuye.';\n    isOverride=dualChoice==='investment'?da0:tc0;\n    balance={applicable:true,stage:2,channels:[{key:'consumption',label:'Canal inversión nacional',sign:da0},{key:'investment',label:'Canal cambiario X−M',sign:tc0},{key:'exchange',label:'Canal cambiario X−M',sign:tc0}],choice:dualChoice==='investment'?'consumption':'exchange',net:isOverride,\n      note:`Las presiones iniciales sobre DA son contrapuestas (I ${arrow(da0)}, X−M ${arrow(tc0)}). Se supone que domina el ${dualChoice==='investment'?'canal de inversión nacional':'canal cambiario'}; el signo neto no es una magnitud estimada.`};\n    step(1,`Doble inicio explícito: I nacional ${arrow(da0)} y O$ ${arrow(fx0)}.`);step(2,`Y por inversión ${arrow(da0)} llega a DL/Md/D$; TC ${arrow(tc0)} llega a DA.`);\n    step(3,`La Y del canal cambiario ${arrow(tc0)} vuelve a DL y Md; la comparación neta queda supuesta.`);\n  }\n  function moneyShock(){\n    if(sh.curve==='ms'){\n      const i=-inc,Y=-i,fxOS=i,tc0=-fxOS;\n      add('money',1,'ms',inc,`El Banco Central modifica Ms ${arrow(inc)}; la tasa de interés nacional ${verb(i)}.`,[],'autonomous');\n      add('goods',2,'da',Y,`La tasa de interés nacional ${verb(i)} por Ms de la secuencia 1; la inversión I ${verb(Y)}, y DA ${verb(Y)}. El movimiento de I por i NO desplaza IS.`,['money:1'],'i-investment',false);\n      add('fx',2,'os',fxOS,`La tasa de interés nacional ${verb(i)} por el efecto monetario 1; entran ${fxOS>0?'más':'menos'} dólares de inversión y O$ ${verb(fxOS)}; TC ${verb(tc0)}.`,['money:1'],'i-capital');\n      add('labor',3,'dl',Y,`Y ${verb(Y)} por bienes y servicios (efecto 2); DL y los salarios ${plural(Y)}.`,['goods:2']);\n      add('money',3,'md',Y,`Y ${verb(Y)} por bienes y servicios (efecto 2); las transacciones y Md ${plural(Y)}.`,['goods:2']);\n      add('fx',3,'ds',Y,`Y ${verb(Y)} por bienes y servicios (efecto 2); las importaciones y D$ ${plural(Y)}.`,['goods:2']);\n      // Clase 24/11, ruta monetaria: solo O$2 → TC2 puede justificar DA3.\n      // DL3 y Md3 son respuestas SIMULTÁNEAS de Y2: no producen otra DA3.\n      // La elección de C/I/X−M corresponde a otros recorridos con antecedentes anteriores.\n      const net=tc0;\n      add('goods',3,'da',net,`En la secuencia 2, la oferta de dólares O$ ${verb(fxOS)} y el tipo de cambio TC ${verb(tc0)}. Según X = f(TC, Y*, P, P*) y M = f(TC, Y, P, P*), las exportaciones ${plural(tc0)} y las importaciones ${plural(-tc0)}; por ello X−M ${verb(tc0)} y DA ${verb(tc0)} (DA2→DA3).`,['fx:2'],'fx-net',true);\n      add('labor',4,'dl',net,`La producción Y ${verb(net)} por el cambio de X−M de DA3, modifica DL ${verb(net)} y los salarios ${plural(net)} en la secuencia 4.`,['goods:3']);\n      step(1,`BCCR Ms ${arrow(inc)} → i ${arrow(i)}.`);step(2,`i ${arrow(i)} provoca I ${arrow(Y)} y O$ ${arrow(fxOS)}.`);\n      step(3,`Y2 ${arrow(Y)} provoca simultáneamente DL3, Md3 y D$3; O$2 ${arrow(fxOS)} → TC ${arrow(tc0)} modifica X−M y genera DA3 ${arrow(net)}.`);\n      step(4,`La nueva Y3 modifica DL4 ${arrow(net)}; no se crea un efecto 5.`);\n      // BP por política monetaria se interpreta desde el canal externo observado en la presentación, no por i aislada.\n      bp=inc;bpWhy=`La política del BCCR induce salida/entrada de capitales representada por O$ ${arrow(fxOS)}.`;\n    }else{\n      const i=inc,Y=-i,os=i,tc0=-os;\n      add('money',1,'md',inc,`La ${sh.label} ${arrow(dir)} genera una demanda de dinero Md ${verb(inc)}, y la tasa de interés nacional ${verb(i)}. La oferta Ms permanece fija.`,[],'autonomous');\n      add('goods',2,'da',Y,`El interés nacional ${verb(i)} a raíz del efecto monetario 1; la inversión nacional I ${verb(Y)}, y DA ${verb(Y)}. Este cambio de inversión inducido por i NO desplaza IS.`,['money:1'],'i-investment',false);\n      add('fx',2,'os',os,`La tasa de interés nacional ${verb(i)} en el mercado monetario 1 cambia el atractivo financiero; O$ ${verb(os)} y el tipo de cambio ${verb(tc0)}.`,['money:1'],'i-capital');\n      add('labor',3,'dl',Y,`Y ${verb(Y)} en bienes y servicios 2; DL y los salarios ${plural(Y)}.`,['goods:2']);\n      add('money',3,'md',Y,`Y ${verb(Y)} en bienes y servicios 2; las transacciones y Md ${plural(Y)}.`,['goods:2']);\n      add('fx',3,'ds',Y,`Y ${verb(Y)} en bienes y servicios 2; las importaciones y D$ ${plural(Y)}.`,['goods:2']);\n      const net=tc0; // El segundo efecto bienes y servicios viene de FX2 (no del gasto público).\n      add('goods',3,'da',net,`El TC ${verb(tc0)} tras O$ ${verb(os)} de la secuencia 2, por lo que X−M ${verb(tc0)} y DA ${verb(tc0)}. No se supone ningún cambio autónomo de gasto público G.`,['fx:2'],'fx-net',true);\n      add('labor',4,'dl',net,`La nueva Y por X−M ${verb(net)} en bienes y servicios 3; DL y salarios ${plural(net)}.`,['goods:3']);\n      step(1,`Md ${arrow(inc)} por ${sh.label}; i ${arrow(i)} y Ms fija.`);step(2,`i ${arrow(i)} → I ${arrow(Y)} y O$ ${arrow(os)}.`);\n      step(3,`Y ${arrow(Y)} modifica DL, Md, D$ y TC ${arrow(tc0)} produce DA3 por X−M.`);\n      step(4,`Y del retorno cambiario ${arrow(net)} produce DL4. No hay DA4 ni DL5.`);\n      bp=0;bpWhy='Un cambio aislado de la tasa de interés causado por Md no constituye un shock autónomo de BP en la regla docente.';\n    }\n  }\n  function doubleTrade(){\n    const goods0=inc,fx0=sh.id==='importaciones'?-inc:inc,fxCurve=sh.id==='importaciones'?'ds':'os';\n    const tradeChoice=spec.tradeChoice==='exchange'?'exchange':'autonomous';\n    const tc0=fxCurve==='os'?-fx0:fx0;\n    add('goods',1,'da',goods0,`La variable exógena modifica directamente ${sh.id==='importaciones'?'las importaciones M':'las exportaciones X'}: DA ${verb(goods0)}.`,[],'autonomous',true);\n    add('fx',1,fxCurve,fx0,`La misma variable inicia una segunda ruta independiente: ${fxCurve==='os'?'O$':'D$'} ${verb(fx0)}, por lo que TC ${verb(tc0)}.`,[],'autonomous');\n    add('labor',2,'dl',goods0,`Y ${verb(goods0)} por DA1 de bienes y servicios; DL y W ${verb(goods0)}.`,['goods:1']);\n    add('money',2,'md',goods0,`Y ${verb(goods0)} por DA1 de bienes y servicios; Md e i ${verb(goods0)}.`,['goods:1']);\n    add('fx',2,'ds',goods0,`Y ${verb(goods0)} por el inicio de DA; importaciones y D$ reciben presión ${goods0>0?'alcista':'bajista'}.`,['goods:1']);\n    add('goods',2,'da',tc0,`El TC ${verb(tc0)} desde el shock cambiario 1; X−M ${verb(tc0)} y DA ${verb(tc0)}. No se crea un tercer desplazamiento de DA.`,['fx:1'],'fx-net',true);\n    add('labor',3,'dl',tc0,`El nuevo Y del canal cambiario (bienes y servicios 2) ${verb(tc0)}; DL y salarios ${plural(tc0)}.`,['goods:2']);\n    add('money',3,'md',tc0,`El nuevo Y del canal cambiario (bienes y servicios 2) ${verb(tc0)}; Md y tasa de interés ${verb(tc0)}.`,['goods:2']);\n    balance={applicable:goods0!==tc0,stage:2,channels:[{key:'consumption',label:'Cambio exportaciones/importaciones autónomas',sign:goods0},{key:'investment',label:'Retorno del tipo de cambio',sign:tc0},{key:'exchange',label:'Retorno del tipo de cambio',sign:tc0}],choice:tradeChoice==='autonomous'?'consumption':'exchange',net:tradeChoice==='autonomous'?goods0:tc0,\n      note:`DA inicial ${arrow(goods0)} y reacción cambiaria ${arrow(tc0)}. El resultado final depende de cuál predomina; no se infiere la magnitud de la diferencia.`};\n    isOverride=balance.net;\n    bp=fxCurve==='os'?-fx0:fx0;bpWhy='La variación comercial altera directamente el ingreso o salida autónoma de dólares.';\n    step(1,`Dos inicios simultáneos: DA ${arrow(goods0)} y ${fxCurve==='os'?'O$':'D$'} ${arrow(fx0)}.`);\n    step(2,`El primer Y ${arrow(goods0)} modifica DL, Md y D$; el primer TC ${arrow(tc0)} modifica DA.`);\n    step(3,`La respuesta cambiaria de DA produce DL y Md ${arrow(tc0)}; la ruta termina.`);\n  }\n  if(sh.id==='ied'&&routeMode==='dual')iedDual();\n  else if(['exportaciones','importaciones','ingreso_socios'].includes(sh.id))doubleTrade();\n  else if(sh.id==='tasas_nacionales'){\n    // Shock exógeno en i: para IS–LM–BP es un movimiento a lo largo de curvas.\n    // No inventar desplazamiento autónomo de IS / LM / BP.\n    const i=dir,Y=-i,os=i,tc0=-os;\n    add('goods',1,'da',Y,`i nacional ${verb(i)} como variable explicativa; I ${verb(Y)} y DA ${verb(Y)}. El cambio de I por i no desplaza la IS.`,[],'i-investment',false);\n    add('fx',1,'os',os,`i nacional ${verb(i)} exógenamente; O$ ${verb(os)} y TC ${verb(tc0)}. No se postula desplazamiento autónomo de BP solo por i.`,[],'i-capital');\n    add('labor',2,'dl',Y,`Y ${verb(Y)} en bienes y servicios 1; DL y W ${verb(Y)}.`,['goods:1']);\n    add('money',2,'md',Y,`Y ${verb(Y)} en bienes y servicios 1; Md responde ${arrow(Y)} sin cambio de Ms.`,['goods:1']);\n    add('fx',2,'ds',Y,`Y ${verb(Y)} en bienes y servicios 1; importaciones y D$ ${verb(Y)}.`,['goods:1']);\n    add('goods',2,'da',tc0,`TC ${verb(tc0)} por O$ en el mercado cambiario 1; X−M ${verb(tc0)} y DA ${verb(tc0)}.`,['fx:1'],'fx-net',true);\n    add('labor',3,'dl',tc0,`Y ${verb(tc0)} por el segundo efecto de DA; DL ${verb(tc0)}.`,['goods:2']);\n    add('money',3,'md',tc0,`Y ${verb(tc0)} por el segundo efecto de DA; Md ${verb(tc0)}.`,['goods:2']);\n    isOverride=tc0;bp=0;\n    step(1,'Cambio dado de i: I y O$ responden; no es por sí mismo un cambio de IS, LM o BP.');step(2,'Y inicial afecta DL/Md/D$; TC afecta X−M y DA.');step(3,'El cambio de Y por X−M retorna a DL y Md.');\n  }\n  else if(sh.market==='goods')goodsShock();\n  else if(sh.market==='labor')laborShock();\n  else if(sh.market==='money')moneyShock();\n  else fxShock();\n  for(const m of ['goods','labor','money','fx']){\n    if(changes[m].length>2)throw new Error('Más de segundo orden en '+m+' '+sh.id);\n  }\n  const seq=Object.values(changes).flat().sort((a,b)=>a.stage-b.stage||(['goods','labor','money','fx'].indexOf(a.market)-['goods','labor','money','fx'].indexOf(b.market)));\n  const eligible=changes.goods.filter(e=>e.eligible).map(e=>({stage:e.stage,sign:e.sign,cause:e.why,weight:1}));\n  const is=isOverride===null?sign(eligible.reduce((a,e)=>a+e.sign,0)):isOverride;\n  const lm=sh.market==='money'&&sh.curve==='ms'?inc:0;\n  const g=changes.goods,lab=changes.labor,money=changes.money,fx=changes.fx;\n  const net=balance?.net??(g.at(-1)?.sign||0);\n  const w=lab.length?(lab.at(-1).curve==='ol'?-lab.at(-1).sign:lab.at(-1).sign):0;\n  const ie=money.length?(money.at(-1).curve==='ms'?-money.at(-1).sign:money.at(-1).sign):0;\n  const exch=fx.length?(fx.at(-1).curve==='os'?-fx.at(-1).sign:fx.at(-1).sign):0;\n  // Resultado cualitativo sujeto a las magnitudes no observadas; evitar atribuir un número a inflación/desempleo.\n  const result={shock:sh,dir,initial:inc,changes,steps:stages,sequence:seq,edges,routeMode,dualChoice,\n    balance:balance||{applicable:false,stage:0,channels:[],net,choice,note:''},stageD:net,\n    dominant:{output:net,wage:w,interest:ie,exchange:exch,employment:lab.at(-1)?.sign||0,inflation:net?'presión '+(net>0?'alcista':'bajista'):'no determinada'},\n    isSign:is,islm:{is,lm,bp,isDrivers:eligible,bpReason:bpWhy},explanations:{},inspect:[],audit:[]};\n  result.explanations.is=`IS ${is>0?'se desplaza a la derecha':is<0?'se desplaza a la izquierda':'permanece fija'}. ${eligible.length?eligible.map(e=>'Secuencia '+e.stage+': '+e.cause).join(' '):'No hay variación autónoma que desplace IS.'} LM ${lm?'se desplaza por Ms del BCCR':'permanece fija porque Ms no cambió por intervención del BCCR'}. BP ${bp?'se desplaza por el flujo de dólares definido':'mantiene su posición'}. ${bpWhy}`;\n  result.audit=seq.map(e=>({n:e.stage,market:e.market,origin:e.parents,why:e.why}));\n  return result;\n}\nMS.analyze=analyze;\n})();\n"

In [ ]:
REFERENCIAS_JS = 'const REFERENCE_CASES = {\n  "infra:1": {\n    "goods": [\n      [\n        "Al aumentar el gasto público (G) eso genera que la Demanda Agregada aumente. Porque: DA = C + I + G + X − M. Donde G es gasto público.",\n        "Al Nivel de producción de equilibrio anterior la DA2 es mayor a la OA generándose un exceso de demanda agregada.",\n        "Los inventarios de los productores se empiezan a agotar, hay presión para que suban los precios e incentivos para aumentar la producción (Ye1).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción mayor (Ye2) con inflación"\n      ],\n      [\n        "Al aumentar los salarios eso genera que la Demanda Agregada aumente. Porque: DA = C + I + G + X − M. Donde C es consumo.",\n        "Al Nivel de producción de equilibrio anterior la DA3 es mayor a la OA generándose un exceso de demanda agregada.",\n        "Los inventarios de los productores se empiezan a agotar, hay presión para que suban los precios e incentivos para aumentar la producción (Ye2).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción mayor (Ye3) con inflación"\n      ]\n    ],\n    "labor": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2) aumenta la demandada de trabajo (DL).",\n        "Al salario de equilibrio anterior (We1) la cantidad demandada de trabajo (DL2) es mayor a la cantidad ofrecida de trabajo (OL). Hay un exceso de demanda de trabajo",\n        "Los empresas con tal de conseguir los empleados están dispuestos a que el salario aumente (We1 a We2).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E2) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario mayor (We2) y a un nivel de empleo mayor (Le2)."\n      ],\n      [\n        "Al aumentar la producción nacional (Ye2 a Ye3) aumenta la demandada de trabajo (DL).",\n        "Al salario de equilibrio anterior (We2) la cantidad demandada de trabajo (DL3) es mayor a la cantidad ofrecida de trabajo (OL). Hay un exceso de demanda de trabajo",\n        "Los empresas con tal de conseguir los empleados están dispuestos a que el salario aumente (We2 a We3).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E3) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario mayor (We3) y a un nivel de empleo mayor (Le3)."\n      ]\n    ],\n    "money": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2) aumenta la Demanda de Dinero (Md1) y se dezplaza la curva de demanda monetaria a la derecha",\n        "A la tasa de interés de equilibrio anterior (ie1) la Oferta de Dinero (Ms1) es menor a la Demanda de Dinero (Md2) generándose un exceso de Demanda de Dinero.",\n        "Hay presión para que la tasa de interés (ie1) suba a ie2.",\n        "Se genera un nuevo equilibrio a una tasa de interés más alta (E1 a E2)."\n      ]\n    ],\n    "fx": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2), aumentan las importaciones (M) del país y, por tanto, la demanda de dólares (D$1 a D$2).",\n        "Al tipo de cambio de equilibrio anterior (TCe1) existe ahora un exceso de demanda de dólares.",\n        "Ese exceso de demanda genera presión para que el tipo de cambio aumente (Tce2).",\n        "Finalmente, se alcanza un nuevo equilibrio con un tipo de cambio mayor (E2)."\n      ]\n    ],\n    "is": [\n      "Al haber aumentado la Demanda Agregada DA2 sin afectar i1 entonces la curva IS1 se desplaza hacia la derecha (IS2)."\n    ]\n  },\n  "tipo_cambio_exo:-1": {\n    "goods": [\n      [\n        "Al disminuir el tipo de cambio aumentan las importaciones lo que genera que la Demanda Agregada disminuya. Porque: DA = C + I + G + (X − M). Donde M (IMPORTACIONES).",\n        "Al Nivel de producción de equilibrio anterior la DA2 es menor a la OA generándose un exceso de OA.",\n        "Los inventarios de los productores se empiezan a acumular, hay presión para que bajen los precios e incentivos para disminuir la producción (Ye1).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción menor (Ye2) sin presión para que haya inflación"\n      ],\n      [\n        "Al disminuir los salarios eso genera que la Demanda Agregada disminuya. Porque: DA = C + I + G + (X − M). Donde C es consumo.",\n        "Al Nivel de producción de equilibrio anterior la DA3 es menor a la OA generándose un exceso de OA.",\n        "Los inventarios de los productores se empiezan a acumular, hay presión para que bajen los precios e incentivos para disminuir la producción (Ye2).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción menor (Ye3) sin presión para que haya inflación"\n      ]\n    ],\n    "labor": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2)se disminuye la demandada de trabajo (DL).",\n        "Al salario de equilibrio anterior (We1) la cantidad demandada de trabajo (DL2) es menor a la cantidad de trabajo (OL). Hay un exceso de oferta de trabajo (OL)",\n        "Los trabajadores con tal de conseguir empleo están dispuestos a que el salario baje (We1 a We2).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E2) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario menor (We2) y a un nivel de empleo menor (Le2)."\n      ],\n      [\n        "Al disminuir la producción nacional (Ye2 a Ye3)se disminuye la demandada de trabajo (DL2).",\n        "Al salario de equilibrio anterior (We2) la cantidad demandada de trabajo (DL3) es menor a la cantidad ofrecida de trabajo (OL). Hay un exceso de oferta de trabajo (OL)",\n        "Los trabajadores con tal de conseguir empleo están dispuestos a que el salario baje (We2 a We3).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E3) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario menor (We3) y a un nivel de empleo menor (Le3)."\n      ]\n    ],\n    "money": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2) disminuye la Demanda de Dinero (Md1) y se dezplaza la curva de demanda monetaria a la izquierda",\n        "A la tasa de interés de equilibrio anterior (ie1) la Oferta de Dinero (Ms1) es mayor a la Demanda de Dinero (Md2) generándose un exceso de Oferta de Dinero.",\n        "Hay presión para que la tasa de interés (ie1) baje a ie2.",\n        "Se genera un nuevo equilibrio a una tasa de interés mas baja (E1 a E2)."\n      ]\n    ],\n    "fx": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2), disminuyen las importaciones (M) del país y, por tanto, la demanda de dólares (D$1 a D$2).",\n        "Al tipo de cambio de equilibrio anterior (TCe1) existe ahora un exceso de oferta de dólares (O$1).",\n        "Ese exceso de oferta genera presión para que el tipo de cambio disminuya (Tce2).",\n        "Finalmente, se alcanza un nuevo equilibrio con un tipo de cambio menor (E2)."\n      ]\n    ],\n    "is": [\n      "Al haber disminuido la Demanda Agregada DA2 sin afectar i1 entonces la curva IS1 se desplaza hacia la izquierda (IS2)."\n    ]\n  },\n  "tasas_nacionales:-1": {\n    "goods": [\n      [\n        "Al disminuir la tasa de interés (i) eso genera que la Demanda Agregada aumente. Porque: DA = C + I + G + X − M. Donde I (INVERSIÓN) dada por F(i).",\n        "Al Nivel de producción de equilibrio anterior la DA2 es mayor a la OA generándose un exceso de demanda agregada.",\n        "Los inventarios de los productores se empiezan a agotar, hay presión para que suban los precios e incentivos para aumentar la producción (Ye1).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción mayor (Ye2) con inflación"\n      ]\n    ],\n    "labor": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2) aumenta la demandada de trabajo (DL).",\n        "Al salario de equilibrio anterior (We1) la cantidad demandada de trabajo (DL2) es mayor a la cantidad de trabajo (OL). Hay un exceso de demanda de trabajo",\n        "Los empresas con tal de conseguir los empleados están dispuestos a que el salario aumente (We1 a We2).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E2) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario mayor (We2) y a un nivel de mayor (Le2)."\n      ]\n    ],\n    "money": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2) aumenta la Demanda de Dinero (Md1) y se dezplaza la curva de demanda monetaria a la derecha",\n        "A la tasa de interés de equilibrio anterior (ie1) la Oferta de Dinero (Ms1) es menor a la Demanda de Dinero (Md2) generándose un exceso de Demanda de Dinero.",\n        "Hay presión para que la tasa de interés (ie1) suba a ie2.",\n        "Se genera un nuevo equilibrio a una tasa de interés más alta (E1 a E2)."\n      ]\n    ],\n    "fx": [\n      [\n        "Al aumentar la producción nacional (Ye1 a Ye2), aumentan las importaciones (M) del país y, por tanto, la demanda de dólares (D$1 a D$2).",\n        "Al tipo de cambio de equilibrio anterior (TCe1) existe ahora un exceso de demanda de dólares.",\n        "Ese exceso de demanda genera presión para que el tipo de cambio aumente (Tce2).",\n        "Finalmente, se alcanza un nuevo equilibrio con un tipo de cambio mayor (E2)."\n      ]\n    ],\n    "is": [\n      "Al haber disminuido la tasa de interés y haber aumentado la inversión la curva IS1 no se mueve, es un movimiento sobre la curva a i2 en E2."\n    ]\n  },\n  "importaciones:1": {\n    "goods": [\n      [\n        "Al aumentar las importaciones genera que la Demanda Agregada disminuya. Porque: DA = C + I + G + (X − M). Donde M (IMPORTACIONES).",\n        "Al Nivel de producción de equilibrio anterior la DA2 es menor a la OA generándose un exceso de OA.",\n        "Los inventarios de los productores se empiezan a acumular, hay presión para que bajen los precios e incentivos para disminuir la producción (Ye1).",\n        "Se alcanza un nuevo equilibrio con un nivel de producción menor (Ye2) sin presión para que haya inflación"\n      ]\n    ],\n    "labor": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2) se disminuye la demanda de trabajo (DL).",\n        "Al salario de equilibrio anterior (We1) la cantidad demandada de trabajo (DL2) es menor a la cantidad de trabajo (OL). Hay un exceso de oferta de trabajo (OL)",\n        "Los trabajadores con tal de conseguir empleo están dispuestos a que el salario baje (We1 a We2).",\n        "Nuevo equilibrio. Encontramos un nuevo equilibrio (E2) donde la cantidad ofrecida y demanda de trabajo son iguales pero a un nivel de salario menor (We2) y a un nivel de empleo menor (Le2)."\n      ]\n    ],\n    "money": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2) disminuye la Demanda de Dinero (Md1) y se dezplaza la curva de demanda monetaria a la izquierda",\n        "A la tasa de interés de equilibrio anterior (ie1) la Oferta de Dinero (Ms1) es mayor a la Demanda de Dinero (Md2) generándose un exceso de Oferta de Dinero.",\n        "Hay presión para que la tasa de interés (ie1) baje a ie2.",\n        "Se genera un nuevo equilibrio a una tasa de interés mas baja (E1 a E2)."\n      ]\n    ],\n    "fx": [\n      [\n        "Al disminuir la producción nacional (Ye1 a Ye2), disminuyen las importaciones (M) del país y, por tanto, la demanda de dólares (D$1 a D$2).",\n        "Al tipo de cambio de equilibrio anterior (TCe1) existe ahora un exceso de oferta de dólares (O$1).",\n        "Ese exceso de oferta genera presión para que el tipo de cambio disminuya (Tce2).",\n        "Finalmente, se alcanza un nuevo equilibrio con un tipo de cambio menor (E2)."\n      ]\n    ],\n    "is": [\n      "Al haber disminuido la Demanda Agregada DA2 sin afectar i1 entonces la curva IS1 se desplaza hacia la izquierda (IS2)."\n    ]\n  }\n};\n'

In [ ]:
APLICACION_JS = '/* Macro Studio 1.0 — standalone presentation and qualitative analysis editor. */\n\'use strict\';\n(function(){\n  const KEY=\'macro-studio-visual-v50\';\n  const root=document.getElementById(\'app\');\n  const esc=s=>String(s??\'\').replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\n  const uid=()=>(\'id-\'+Math.random().toString(36).slice(2,9));\n  const clone=x=>JSON.parse(JSON.stringify(x));\n  const TITLES={goods:\'Bienes y servicios\',labor:\'Mercado laboral\',money:\'Mercado monetario\',fx:\'Mercado cambiario\'};\n  const CODES={goods:\'Y\',labor:\'W · L\',money:\'i\',fx:\'TC\'};\n  const CHARTS=[\'goods\',\'labor\',\'money\',\'fx\'];\n  const BANK=[\n    [\'¿Qué sucede si el gobierno aumenta el gasto público?\',\'gasto_publico\',1,\'Política fiscal expansiva · bienes y servicios\'],\n    [\'¿Qué sucede si el gobierno disminuye el gasto público?\',\'gasto_publico\',-1,\'Política fiscal restrictiva · bienes y servicios\'],\n    [\'¿Qué sucede si el BCCR aplica una política monetaria restrictiva?\',\'contraccion_monetaria\',1,\'Política monetaria restrictiva · mercado monetario\'],\n    [\'¿Qué sucede si aumenta el gasto en infraestructura del sector público?\',\'infra\',1,\'Mercado laboral · ejercicio #2\'],\n    [\'¿Qué sucede si disminuye el tipo de cambio?\',\'tipo_cambio_exo\',-1,\'Mercado laboral · ejercicio #2\'],\n    [\'¿Qué sucede si disminuyen las tasas de interés?\',\'tasas_nacionales\',-1,\'Ejercicio #1\'],\n    [\'¿Qué sucede si aumentan las importaciones?\',\'importaciones\',1,\'Ejercicio #1 / Macroeconomía I\'],\n    [\'¿Qué sucede si las importaciones disminuyen?\',\'importaciones\',-1,\'Ejercicio #2 / presentación laboral\'],\n    [\'¿Qué sucede si aumentan las tasas de interés?\',\'tasas_nacionales\',1,\'Presentación laboral\'],\n    [\'¿Qué sucede si aumenta el tipo de cambio?\',\'tipo_cambio_exo\',1,\'Ejercicios y presentaciones\'],\n    [\'¿Qué sucede si disminuye el gasto público en infraestructura?\',\'infra\',-1,\'Ejercicios y presentaciones\'],\n    [\'¿Qué sucede si aumentan los salarios?\',\'salario_exo\',1,\'Ejercicios y presentaciones\'],\n    [\'¿Qué sucede si hay un proceso migratorio hacia Costa Rica?\',\'migracion_entrada\',1,\'Caso laboral / migración\'],\n    [\'¿Qué sucede si hay un proceso migratorio fuera de Costa Rica?\',\'migracion_salida\',1,\'Caso laboral / migración\'],\n    [\'¿Qué sucede si hay una mayor demanda por precaución a nivel nacional?\',\'dinero_precaucion\',1,\'Ejercicios monetarios\'],\n    [\'¿Qué sucede si el gobierno disminuye la deuda interna?\',\'deuda_interna\',-1,\'Mercado monetario · ejercicio de clase\'],\n    [\'¿Qué sucede si aumenta la deuda interna del gobierno?\',\'deuda_interna\',1,\'Ejercicios monetarios\'],\n    [\'¿Qué sucede si aumentan los impuestos?\',\'impuestos\',1,\'Ejercicios bienes y servicios\'],\n    [\'¿Qué sucede si disminuye la inversión extranjera directa?\',\'ied\',-1,\'Ejercicios cambiarios\'],\n    [\'¿Qué pasa si disminuyen las tasas de interés internacionales?\',\'tasas_externas\',-1,\'Mercado cambiario · ejemplo de clase\'],\n    [\'¿Qué pasa si aumentan las exportaciones como resultado de un TLC?\',\'exportaciones\',1,\'Mercado cambiario · ejemplo de clase\'],\n    [\'¿Qué pasa si el BCCR aplica una política monetaria expansiva?\',\'expansion_monetaria\',1,\'Mercado cambiario · ejemplo de clase\'],\n        \n    [\'¿Qué sucede si la tecnología desplaza mano de obra a nivel nacional?\',\'tecnologia_sust\',1,\'Mercado monetario · ejercicio de clase\'],\n    \n  ];\n  const mkcase=(title=\'Nuevo caso\',shockId=\'infra\',direction=1)=>({id:uid(),title,shockId,direction:shockId===\'tecnologia_sust\'?1:direction,showSecond:true,showIS:true,showFX:true,showExplanation:true,showPolicyMap:false,balanceChoice:\'consumption\',routeMode:\'cambiario\',dualChoice:\'exchange\',tradeChoice:\'autonomous\',manualNumbers:{},overrides:{},note:\'\'});\n  const mkpres=(name=\'Nueva colección\')=>({id:uid(),title:name,cases:[mkcase(\'Nuevo caso\')],description:\'\'});\n  const seed=()=>({version:50,presentations:[\n  {id:\'p-start\',title:\'Mis casos\',description:\'Colección editable de escenarios económicos.\',cases:[mkcase(\'¿Qué sucede si el gobierno disminuye la deuda interna?\',\'deuda_interna\',-1),mkcase(\'¿Qué sucede si la tecnología desplaza mano de obra a nivel nacional?\',\'tecnologia_sust\',1)]},\n  {id:\'p-clase\',title:\'Casos de referencia\',description:\'Colección de escenarios económicos para comparar rutas y supuestos.\',cases:[\n    mkcase(\'¿Qué sucede si disminuye la inversión extranjera directa?\',\'ied\',-1),\n    {...mkcase(\'IED ↓: modelo con doble inicio en inversión nacional y oferta de dólares\',\'ied\',-1),routeMode:\'dual\',dualChoice:\'investment\'},\n    mkcase(\'¿Qué sucede si aumenta la deuda interna del gobierno?\',\'deuda_interna\',1),\n    mkcase(\'¿Qué sucede si aumenta la migración hacia Costa Rica?\',\'migracion_entrada\',1),\n    mkcase(\'¿Qué sucede si hay una política monetaria expansiva del BCCR?\',\'expansion_monetaria\',1),\n    mkcase(\'¿Qué sucede si las tasas de interés internacionales disminuyen?\',\'tasas_externas\',-1),\n    mkcase(\'¿Qué sucede si aumentan las exportaciones como resultado de un TLC?\',\'exportaciones\',1)\n  ]}\n ],activePres:\'p-start\',activeCase:null,customShocks:[]});\n  let initial;try{initial=JSON.parse(document.getElementById(\'macro-snapshot\')?.textContent||\'null\')}catch(_){initial=null}\n  let state=initial; if(!state){try{state=JSON.parse(localStorage.getItem(KEY))}catch(_){state=null}}if(!state?.presentations?.length){try{const archived=JSON.parse(localStorage.getItem(\'macro-studio-visual-v32\'));if(archived?.presentations?.length)state=archived;}catch(_){}}if(!state?.presentations?.length)state=seed(); state.version=51; for(const pr of state.presentations){for(const cs of pr.cases||[]){if(![\'consumption\',\'investment\',\'exchange\'].includes(cs.balanceChoice))cs.balanceChoice=\'consumption\';cs.overrides??={};}}\n  for(const p of state.presentations)for(const c of p.cases){if(c.shockId===\'tecnologia_sust\'){c.direction=1;if(/IA desplaza mano de obra/i.test(c.title))c.title=\'¿Qué sucede si la tecnología desplaza mano de obra a nivel nacional?\';}c.showSecond??=true;c.showIS??=true;c.showFX??=true;c.showExplanation??=true;c.showPolicyMap??=false;c.manualNumbers??={};c.overrides??={};c.note??=\'\';c.routeMode??=\'cambiario\';c.dualChoice??=\'exchange\';c.tradeChoice??=\'autonomous\';}\n  for(const savedPres of state.presentations){\n    if(/^Ejercicios del grupo \\d+ · IS–LM$/.test(savedPres.title)) savedPres.title=\'Mis casos\';\n    if(/^Ejercicio del grupo \\d+ y casos para contrastar primero y segundo orden$/.test(savedPres.description)) savedPres.description=\'Colección editable de escenarios económicos.\';\n  }\n  if(!Array.isArray(state.customShocks))state.customShocks=[];\n  // Las colecciones eliminadas por el usuario no se restauran automáticamente.\n  for(const col of state.presentations){\n    if(col.id===\'p-clase\' && col.title===\'Ejercicios de clase · 17 noviembre\')col.title=\'Casos de referencia\';\n  }\n  state.activePres ||=state.presentations[0].id;\n  let mode=\'editor\',slideIndex=0,help=false,toastT,diagnosticPoint={Y:42,i:77};\n  const pres=()=>state.presentations.find(p=>p.id===state.activePres)||state.presentations[0];\n  const selected=()=>pres().cases.find(c=>c.id===state.activeCase)||pres().cases[0];\n  if(!state.activeCase)state.activeCase=selected().id;\n  const allShocks=()=>[...MS.CATALOG,...state.customShocks];\n  const shock=c=>allShocks().find(s=>s.id===c.shockId)||MS.CATALOG[0];\n  const titleForCase=c=>BANK.find(b=>b[1]===c.shockId&&b[2]===c.direction)?.[0]||`Escenario: ${shock(c).label} ${c.direction>0?\'↑\':\'↓\'}`;\n  const analyze=c=>MS.analyze(c,state.customShocks);\n  const save=()=>{try{localStorage.setItem(KEY,JSON.stringify(state))}catch(_){/* JSON export remains available */}};\n  const btn=(t,a,cls=\'\',extra=\'\')=>`<button type="button" class="button ${cls}" data-act="${a}" ${extra}>${t}</button>`;\n  const pill=(t,kind=\'\')=>`<span class="pill ${kind}">${esc(t)}</span>`;\n  function toast(msg){document.querySelector(\'.toast\')?.remove();let div=document.createElement(\'div\');div.className=\'toast\';div.textContent=msg;document.body.append(div);clearTimeout(toastT);toastT=setTimeout(()=>div.remove(),3500)}\n  const sign=v=>v>0?\'↑\':v<0?\'↓\':\'↔\';\n  const SUFFIX={goods:\'Ye\',labor:\'Le\',money:\'ie\',fx:\'TCe\'};\n  const QDIR=v=>v<0?\'disminuye\':\'aumenta\';\n  // Colecciones y casos: selección, renombrado, duplicado y eliminación sin salir del panel.\n  const railIcon=(action,id,glyph,title,extra=\'\')=>`<button type="button" class="rail-tool ${extra}" data-act="${action}" data-id="${esc(id)}" title="${esc(title)}" aria-label="${esc(title)}">${glyph}</button>`;\n  const panelList=()=>`<aside class="rail" aria-label="Espacio de trabajo">\n    <div class="rail-heading"><div><div class="rail-label">ESPACIO DE TRABAJO</div><strong>Colecciones</strong></div>${btn(\'+\',\'new-pres\',\'rail-add\',\'title="Nueva colección" aria-label="Nueva colección"\')}</div>\n    <div class="preslist">${state.presentations.map(p=>`<div class="pres-row ${p.id===pres().id?\'on\':\'\'}">\n      <button class="pres-entry ${p.id===pres().id?\'on\':\'\'}" data-act="pick-pres" data-id="${esc(p.id)}" title="${esc(p.title)}" aria-label="Abrir colección ${esc(p.title)}"><span class="folder-icon">▤</span><span class="truncate">${esc(p.title)}</span><b>${p.cases.length}</b></button>\n      <div class="rail-tools">${railIcon(\'rename-pres\',p.id,\'✎\',\'Renombrar colección \'+p.title)}${railIcon(\'duplicate-pres\',p.id,\'▢\',\'Duplicar colección \'+p.title)}${railIcon(\'delete-pres\',p.id,\'×\',\'Eliminar colección \'+p.title,\'danger\')}</div></div>`).join(\'\')}</div>\n    <div class="rail-row"><span>CASOS · ${esc(pres().title)}</span>${btn(\'+\',\'new-case\',\'rail-add\',\'title="Nuevo caso" aria-label="Nuevo caso"\')}</div>\n    <div class="case-list">${pres().cases.map((c,i)=>`<div class="case-row ${c.id===selected().id?\'on\':\'\'}">\n      <button class="case-entry ${c.id===selected().id?\'on\':\'\'}" data-act="pick-case" data-id="${esc(c.id)}" title="${esc(c.title)}" aria-label="Abrir caso ${esc(c.title)}"><span class="case-index">${String(i+1).padStart(2,\'0\')}</span><span class="truncate">${esc(c.title)}</span></button>\n      <div class="rail-tools">${railIcon(\'rename-case\',c.id,\'✎\',\'Renombrar caso \'+c.title)}${railIcon(\'delete-case\',c.id,\'×\',\'Eliminar caso \'+c.title,\'danger\')}</div></div>`).join(\'\')}</div>\n    <div class="rail-actions">${btn(\'Nueva colección\',\'new-pres\',\'rail-wide\')}${btn(\'Banco de ejercicios\',\'bank\',\'rail-wide soft\')}${btn(\'Limpiar espacio\',\'clear-workspace\',\'rail-wide reset-workspace\')}</div>\n    <div class="rail-note">Los cambios quedan guardados en este navegador. Usá <b>Guardar JSON</b> para respaldar las colecciones antes de limpiarlas o cambiar de equipo.</div>\n    </aside>`;\n  const controls=c=>`<div class="controlpanel"><div class="control-main"><label>Enunciado / título<input id="title-input" value="${esc(c.title)}" spellcheck="true"/></label><label>Variable explicativa<select id="shock-input">${Object.entries(MS.MARKETS).map(([market,title])=>`<optgroup label="${esc(title)}">${allShocks().filter(s=>s.market===market).map(s=>`<option value="${esc(s.id)}" ${c.shockId===s.id?\'selected\':\'\'}>${esc(s.label)}</option>`).join(\'\')}</optgroup>`).join(\'\')}</select></label><label>Variación inicial${c.shockId===\'tecnologia_sust\'?\'<small class="source-direction">El enunciado ya establece la sustitución laboral: DL disminuye.</small>\':\'\'}<select id="direction-input" ${c.shockId===\'tecnologia_sust\'?\'disabled\':\'\'}><option value="1" ${c.direction>0?\'selected\':\'\'}>${c.shockId===\'tecnologia_sust\'?\'Sustituye mano de obra → DL ↓\':\'Aumenta ↑\'}</option>${c.shockId===\'tecnologia_sust\'?\'\':`<option value="-1" ${c.direction<0?\'selected\':\'\'}>Disminuye ↓</option>`}</select></label>${[\'exportaciones\',\'importaciones\',\'ingreso_socios\'].includes(c.shockId)?`<label>Predominio entre dos vías<select id="trade-input"><option value="autonomous" ${c.tradeChoice!==\'exchange\'?\'selected\':\'\'}>Predomina cambio comercial inicial</option><option value="exchange" ${c.tradeChoice===\'exchange\'?\'selected\':\'\'}>Predomina retorno cambiario</option></select></label>`:\'\'}${c.shockId===\'ied\'?`<label>Ruta económica<select id="route-input"><option value="cambiario" ${c.routeMode!==\'dual\'?\'selected\':\'\'}>Inicio cambiario · ruta presentada</option><option value="dual" ${c.routeMode===\'dual\'?\'selected\':\'\'}>Dos inicios: inversión nacional y O$</option></select></label>${c.routeMode===\'dual\'?`<label>Supuesto del efecto neto<select id="dual-input"><option value="exchange" ${c.dualChoice!==\'investment\'?\'selected\':\'\'}>Predomina retorno cambiario</option><option value="investment" ${c.dualChoice===\'investment\'?\'selected\':\'\'}>Predomina I nacional</option></select></label>`:\'\'}`:\'\'}</div><div class="control-options"><label class="toggle"><input id="second-input" type="checkbox" ${c.showSecond?\'checked\':\'\'}/> Segundo orden</label><label class="toggle"><input id="is-input" type="checkbox" ${c.showIS?\'checked\':\'\'}/> Modelo IS–LM–BP</label><span class="always-market">Los cuatro mercados integrados</span><label class="toggle"><input id="explain-input" type="checkbox" ${c.showExplanation?\'checked\':\'\'}/> Explicaciones completas</label><label class="toggle"><input id="policy-input" type="checkbox" ${c.showPolicyMap?\'checked\':\'\'}/> Diagnóstico de política económica</label>${btn(\'+ Variable personalizada\',\'custom-shock\',\'outline small\')}</div></div>`;\n  function balanceSelector(c){const r=analyze(c);if(!c.showSecond||!r.balance?.applicable||c.shockId===\'ied\'&&c.routeMode===\'dual\'||[\'exportaciones\',\'importaciones\',\'ingreso_socios\'].includes(c.shockId))return \'\';\n    const b=r.balance;return `<section class="balance-choice" aria-label="Alternativa de segundo orden"><div class="balance-copy"><div class="eyebrow">DECISIÓN DEL EJERCICIO · SECUENCIA ${b.stage}</div><h3>¿Cuál efecto contrapuesto predomina?</h3><p><b>Consumo (C) ${sign(b.channels[0].sign)}</b> por salarios y <b>Inversión (I) ${sign(b.channels[1].sign)}</b> por tasa de interés. ${b.channels[2]?`Además, el tipo de cambio modifica X − M (${sign(b.channels[2].sign)}).`:\'\'} Se desarrolla el canal que predomina en el resultado neto.</p></div><fieldset class="balance-radio"><legend>Resultado para continuar el segundo orden</legend><label class="balance-option ${c.balanceChoice===\'consumption\'?\'chosen\':\'\'}"><input type="radio" name="balanceChoice" value="consumption" ${c.balanceChoice===\'consumption\'?\'checked\':\'\'}/><span><b>Predomina consumo (C)</b><small>DA ${sign(b.channels[0].sign)} · recálculo de efectos posteriores</small></span></label><label class="balance-option ${c.balanceChoice===\'investment\'?\'chosen\':\'\'}"><input type="radio" name="balanceChoice" value="investment" ${c.balanceChoice===\'investment\'?\'checked\':\'\'}/><span><b>Predomina inversión (I)</b><small>DA ${sign(b.channels[1].sign)} · recálculo de efectos posteriores</small></span></label>${b.channels[2]?`<label class="balance-option ${c.balanceChoice===\'exchange\'?\'chosen\':\'\'}"><input type="radio" name="balanceChoice" value="exchange" ${c.balanceChoice===\'exchange\'?\'checked\':\'\'}/><span><b>Predominan exportaciones netas (X − M)</b><small>DA ${sign(b.channels[2].sign)} · se recalculan los efectos posteriores</small></span></label>`:\'\'}</fieldset></section>`;}\n  function topbar(){return `<header class="topbar"><div class="identity"><div class="logomark">M<span>Σ</span></div><div><strong>MACRO STUDIO</strong><small>Simulador gráfico · laboratorio de casos</small></div></div><div class="toptools">${btn(\'Banco de ejercicios\',\'bank\',\'outline\')}${btn(\'Nueva colección\',\'new-pres\',\'outline\')}${btn(\'Nuevo caso\',\'new-case\',\'outline\')}${btn(\'Modo exposición\',\'present\',\'primary\')}${btn(\'Guardar JSON\',\'export-json\',\'outline\')}${btn(\'Importar JSON\',\'import-json\',\'outline\')}${btn(\'Exportar HTML\',\'export-html\',\'outline\')}</div></header>`}\n  function editor(){const c=selected(),p=pres();return `${topbar()}<div class="workspace">${panelList()}<main class="main"><div class="case-heading"><div><div class="eyebrow">${esc(p.title)} <span>/</span> caso ${p.cases.indexOf(c)+1} de ${p.cases.length}</div><h1>${esc(c.title)}</h1><div class="subtitle">Modelo de transmisión entre mercados · ecuaciones, desplazamientos y secuencia causal</div></div><div class="caseactions">${btn(\'Duplicar\',\'duplicate\',\'outline small\')}${btn(\'Borrar\',\'delete\',\'ghost small\')}${btn(\'Editar párrafos\',\'edit-paragraphs\',\'outline small\')}</div></div>${controls(c)}${balanceSelector(c)}${caseSheet(c,false)}${footer()}</main></div>`}\n  function footer(){return `<p class="method-footer">Análisis cualitativo de primer y segundo orden, con números de secuencia, curvas y equilibrios de referencia.</p>`}\n  function stageList(c,r,m){let events=r.changes[m];if(c.showSecond===false)events=events.filter(e=>e.round!==\'segundo\');let manual=c.manualNumbers?.[m];return manual?.trim()?manual.split(/[\\s,;]+/).filter(x=>/^\\d{1,2}$/.test(x)).map(Number):events.map(e=>e.stage)}\n  function stepDots(c,r,m){const events=(c.showSecond?r.changes[m]:r.changes[m].filter(e=>e.round!==\'segundo\'));\n    const manual=c.manualNumbers?.[m];if(manual?.trim())return manual.split(/[\\s,;]+/).filter(x=>/^\\d{1,2}$/.test(x)).map(x=>`<span class="stage-dot stage-first">${x}</span>`).join(\'\');\n    if(!events.length)return \'<span class="stage-none">Sin desplazamiento</span>\';\n    return events.map(e=>`<span class="stage-dot ${e.round===\'segundo\'?\'stage-second\':\'stage-first\'}">${e.stage}</span>`).join(\'\');}\n\n  const exactKey=c=>`${c.shockId}:${c.direction}`;\n  const refSection=(c,m)=>REFERENCE_CASES[exactKey(c)]?.[m];\n  const SERIES={da:\'DA\',oa:\'OA\',dl:\'DL\',ol:\'OL\',ms:\'Ms\',md:\'Md\',os:\'O$\',ds:\'D$\'};\n  const RANK_COLORS=[\'#24384b\',\'#1879b9\',\'#cc3889\',\'#8550ba\',\'#167d6c\',\'#bd7513\',\'#d64c47\'];\n  const STAGE_COLORS={1:\'#172d41\',2:\'#1879b9\',3:\'#cc3889\',4:\'#8550ba\'};\n  const stageColor=n=>STAGE_COLORS[n]||RANK_COLORS[(n-1)%RANK_COLORS.length];\n  const changedColor=(stage,revision,round)=>round===\'segundo\'?\'#cc3889\':\'#1879b9\';\n  const movename=v=>v>0?\'aumenta\':\'disminuye\';\n  const pastname=v=>v>0?\'aumentó\':\'disminuyó\';\n  const specialRefFix=t=>String(t).replace(/\\bdezplaza\\b/gi,\'desplaza\').replace(/\\bLos empresas\\b/g,\'Las empresas\').replace(/\\bdemandada de trabajo\\b/g,\'demanda de trabajo\').replace(/\\bse disminuye la demanda\\b/g,\'disminuye la demanda\').replace(/\\s+se\\s+disminuye\\s+la\\s+demanda/gi,\' disminuye la demanda\');\n  const variantIndex=(r,m,ev)=>1+r.changes[m].filter(x=>x.curve===ev.curve && x.stage<=ev.stage).length;\n  const movingCurve=(r,m,ev)=>SERIES[ev.curve]+variantIndex(r,m,ev);\n  const previousCurve=(r,m,ev)=>SERIES[ev.curve]+(variantIndex(r,m,ev)-1);\n  const hasShock=(r,m)=>r.shock.market===m;\n  function subjectPhrase(label){\n    const t=String(label).toLowerCase();\n    if(/^(compras|transferencias|exportaciones|importaciones|remesas|tasas|pensiones|restricciones)/.test(t))return \'las \'+t;\n    if(/^(impuestos|salarios|flujos|préstamos|obstáculos|desastres)/.test(t))return \'los \'+t;\n    if(/^(deuda|inversión|tasa|demanda|oferta|emisión|política|participación|inmigración|emigración|migración|tecnología|disponibilidad|propensión|inteligencia)/.test(t))return \'la \'+t;\n    if(/^(gasto|tipo|encaje|consumo|ingreso|crédito|acceso|turismo|atractivo|capital|desastre)/.test(t))return \'el \'+t;\n    return t;\n  }\n  function causalLead(c,r,m,e,prior,next,priorEq,nextEq){\n    const inc=movename(e.sign),dec=movename(-e.sign),verb=(c?.direction??r.dir)>0?\'aumenta\':\'disminuye\';\n    const yStart=r.shock.market===\'goods\'?1:r.shock.market===\'labor\'?1:r.shock.market===\'money\'?1:1;\n    if(e.stage===1){\n      if(m===\'goods\')return `Debido a que ${subjectPhrase(r.shock.label)} ${verb}, se modifica ${r.shock.channel} en la expresión DA = C + I + G + X − M. Por ello, la demanda agregada ${movename(e.sign)}: ${prior} se desplaza a ${next}.`;\n      if(m===\'labor\')return r.shock.id===\'tecnologia_sust\'?`Debido a que la tecnología desplaza mano de obra a nivel nacional, las empresas necesitan menos trabajadores para una misma producción; por ello, la demanda laboral disminuye. Según DL = f(Y, tecnología), ${prior} se desplaza a la izquierda hasta ${next}, manteniéndose OL1 fija.`:`Debido a que ${subjectPhrase(r.shock.label)} ${verb}, la ${e.curve===\'ol\'?\'oferta\':\'demanda\'} de trabajo ${movename(e.sign)}. Según ${e.curve===\'ol\'?\'OL = f(migración, educación y capacitación, desastres naturales)\':\'DL = f(Y, tecnología)\'}, ${prior} se desplaza ${e.sign>0?\'a la derecha\':\'a la izquierda\'} hasta ${next}.`;\n      if(m===\'money\')return `Debido a que ${subjectPhrase(r.shock.label)} ${verb}, la ${e.curve===\'ms\'?\'oferta\':\'demanda\'} de dinero ${movename(e.sign)} en el mercado monetario. Por ello, ${prior} se desplaza ${e.sign>0?\'a la derecha\':\'a la izquierda\'} hasta ${next}, de acuerdo con ${e.curve===\'ms\'?\'Ms = f(EML, BEMs, TPM, emisión monetaria)\':\'Md = f(Y, transacciones, precaución, especulación, deuda interna)\'}.`;\n      return `Debido a que ${subjectPhrase(r.shock.label)} ${verb}, la ${e.curve===\'os\'?\'oferta\':\'demanda\'} de dólares ${movename(e.sign)} en el mercado cambiario: ${prior} se desplaza ${e.sign>0?\'a la derecha\':\'a la izquierda\'} hasta ${next}.`;\n    }\n    if(m===\'goods\'){\n      if(e.stage===2 && r.shock.market===\'money\')return `Debido a que la tasa de interés ${movename(r.dominant.interest)} de ie1 a ie2 como resultado del efecto 1 monetario, se modifica la inversión (I), conforme a I = f(i, acceso al crédito y restricciones). Al aplicar DA = C + I + G + X − M, la curva ${prior} se desplaza hacia ${next}.`;\n      if(e.stage===2 && r.shock.market===\'labor\')return `Debido a que en el mercado laboral los salarios ${r.dominant.wage>0?\'aumentan\':\'disminuyen\'} de We1 a We2 (efecto 1), se modifica el ingreso disponible y, por tanto, el consumo (C). Como C forma parte de DA = C + I + G + X − M, la curva ${prior} se desplaza hacia ${next}.`;\n      if(e.stage===2 && r.shock.market===\'fx\')return `Debido a que el tipo de cambio ${movename(r.dominant.exchange)} de TCe1 a TCe2 en el mercado cambiario (efecto 1), se modifican las exportaciones (X) y las importaciones (M). En DA = C + I + G + X − M, el resultado desplaza ${prior} hacia ${next}.`;\n      if(e.stage===3 && r.shock.market===\'money\')return `En la secuencia 2, la tasa de interés modificó la oferta de dólares y por tanto el tipo de cambio. Este nuevo tipo de cambio modifica las exportaciones (X) y las importaciones (M). Por ello, DA ${movename(e.sign)} de ${prior} a ${next} mediante X − M.`;\n      if(e.stage===4 && r.shock.market===\'labor\')return `Debido al cambio en la demanda de trabajo de la secuencia 3, los salarios modifican el consumo (C); al mismo tiempo, el cambio de Md modifica la tasa de interés (i) y la inversión (I). Dentro de DA = C + I + G + X − M se escoge predominio de ${r.balance?.choice===\'investment\'?\'inversión (I)\':\'consumo (C)\'}, por lo que DA ${movename(e.sign)} y ${prior} pasa a ${next}.`;\n      if(e.stage===4 && r.shock.market===\'money\')return `Debido a los dos efectos simultáneos de la secuencia 3, en el mercado laboral los salarios ${r.dominant.wage>0?\'aumentan\':\'disminuyen\'} de We1 a We2 y modifican el consumo (C); en el mercado monetario, la tasa de interés ${r.dominant.interest>0?\'disminuye parcialmente desde ie2 hacia ie3\':\'aumenta parcialmente desde ie2 hacia ie3\'} y modifica la inversión (I). Ambas variables participan en DA = C + I + G + X − M. Se escoge predominio de ${r.balance?.choice===\'investment\'?\'inversión (I)\':\'consumo (C)\'}, por lo que DA ${movename(e.sign)} y ${prior} pasa a ${next}.`;\n      if(e.stage===4 && r.shock.market===\'fx\')return `En la secuencia 3 los salarios afectan el consumo, el cambio de la tasa de interés modifica la inversión y el tipo de cambio cambia X − M. En el efecto 4 predomina ${r.balance?.choice===\'exchange\'?\'X − M\':r.balance?.choice===\'investment\'?\'la inversión (I)\':\'el consumo (C)\'}. Por ello, DA ${movename(e.sign)} y ${prior} se desplaza a ${next}.`;\n      if(e.stage===3)return `Debido a los efectos simultáneos de la secuencia 2, los salarios pasan de We1 a We2, el tipo de cambio de TCe1 a TCe2 y la tasa de interés de ie1 a ie2. Estos cambios afectan C, (X − M) e I, respectivamente, dentro de DA = C + I + G + X − M. Según la alternativa escogida, la demanda agregada ${movename(e.sign)} de ${prior} a ${next}.`;\n      return `Debido a los cambios de salarios, tipo de cambio e interés derivados de la secuencia 3, se modifican C, (X − M) e I dentro de DA = C + I + G + X − M. La demanda agregada pasa de ${prior} a ${next} en la secuencia ${e.stage}.`;\n    }\n    if(m===\'labor\'){\n      if(e.stage===1)return `Debido a que ${subjectPhrase(r.shock.label)} ${verb}, la curva ${prior} se desplaza a ${next} en DL = f(Y, tecnología) u OL = f(migración, educación, desastres).`;\n      return `Debido a que la producción nacional ${movename(e.sign)} de Ye${e.stage>3?2:1} a Ye${e.stage>3?3:2} como resultado del efecto anterior en bienes y servicios, la demanda de trabajo también ${movename(e.sign)}. Conforme DL = f(Y, tecnología), ${prior} se desplaza hasta ${next}.`;\n    }\n    if(m===\'money\')return `Debido a que la producción nacional ${movename(e.sign)} de Ye${e.stage>3?2:1} a Ye${e.stage>3?3:2} en bienes y servicios, la demanda de dinero para transacciones también ${movename(e.sign)}. Según Md = f(Y, transacciones, precaución, especulación y deuda interna), ${prior} se desplaza ${e.sign>0?\'a la derecha\':\'a la izquierda\'} hasta ${next}.`;\n    if(e.curve===\'os\')return `Debido al cambio de la tasa de interés nacional (ie1 a ie2) en el mercado monetario, cambian los flujos de capital y la oferta de dólares. La curva ${prior} se desplaza hacia ${next} en la secuencia ${e.stage}.`;\n    return `Debido a que la producción nacional ${movename(e.sign)} de Ye1 a Ye2 en bienes y servicios, las importaciones (M) ${e.sign>0?\'aumentan\':\'disminuyen\'} y, por ello, la demanda de dólares (D$). La curva ${prior} se desplaza hacia ${next} en la secuencia ${e.stage}.`;\n  }\n  function numberedFour(c,r,m,e){\n    const n=r.changes[m].indexOf(e),idx=n+2,prev=idx-1,eq0=\'E\'+prev,eq1=\'E\'+idx;\n    const prior=previousCurve(r,m,e),next=movingCurve(r,m,e),down=e.sign<0;\n    const lead=causalLead(c,r,m,e,prior,next,eq0,eq1);\n    if(m===\'goods\')return [\n      lead,\n      `Al nivel de producción de equilibrio anterior (Ye${prev}), la ${next} es ${down?\'menor\':\'mayor\'} que la Oferta Agregada (OA), generándose un exceso de ${down?\'oferta\':\'demanda\'} agregada. La OA permanece en la misma posición.`,\n      `Los inventarios de los productores se empiezan a ${down?\'acumular\':\'agotar\'}, hay presión para que ${down?\'bajen\':\'suban\'} los precios e incentivos para ${down?\'disminuir\':\'aumentar\'} la producción desde Ye${prev}.`,\n      `Se alcanza un nuevo equilibrio (${eq1}) con un nivel de producción ${down?\'menor\':\'mayor\'} (Ye${idx}) ${down?\'sin presión adicional para que haya inflación\':\'con presión para que haya inflación\'}.`\n    ];\n    if(m===\'labor\'){\n      const offer=e.curve===\'ol\', wsign=offer?-e.sign:e.sign,excess=offer?(e.sign>0?\'oferta\':\'demanda\'):(e.sign>0?\'demanda\':\'oferta\');\n      return [lead,\n        `Al salario de equilibrio anterior (We${prev}), la cantidad ${excess===\'demanda\'?\'demandada\':\'ofrecida\'} de trabajo es mayor a la ${excess===\'demanda\'?\'ofrecida\':\'demandada\'}; por tanto, existe un exceso de ${excess} de trabajo entre ${next} y la curva opuesta.`,\n        `${excess===\'demanda\'?\'Las empresas, con tal de conseguir trabajadores, están dispuestas a aumentar\':\'Los trabajadores, con tal de conseguir empleo, están dispuestos a reducir\'} el salario, que pasa de We${prev} a We${idx}.`,\n        `Nuevo equilibrio (${eq1}): la cantidad ofrecida y demandada de trabajo son iguales a un salario ${wsign>0?\'mayor\':\'menor\'} (We${idx}) y con un nivel de empleo ${e.sign>0?\'mayor\':\'menor\'} (Le${idx}).`\n      ];\n    }\n    if(m===\'money\'){\n      const ir=e.curve===\'ms\'?-e.sign:e.sign;\n      return [lead,\n        `A la tasa de interés de equilibrio anterior (ie${prev}), existe un exceso de ${ir>0?\'demanda\':\'oferta\'} de dinero, porque la ${e.curve===\'ms\'?\'oferta monetaria\':\'demanda monetaria\'} ha pasado de ${prior} a ${next} mientras la otra curva permanece fija.`,\n        `Ese exceso genera presión para que la tasa de interés ${ir>0?\'aumente\':\'disminuya\'} de ie${prev} a ie${idx}.`,\n        `Se establece el nuevo equilibrio monetario (${eq1}) con una tasa de interés ${ir>0?\'mayor\':\'menor\'} (ie${idx}).${e.round===\'segundo\'&&r.shock.market===\'money\'?` Se trata de un rebote parcial: ie${idx} permanece ${r.dominant.interest>0?\'por encima\':\'por debajo\'} de ie1, porque el efecto de segundo orden es menor que el inicial.`:\'\'}`\n      ];\n    }\n    const ts=e.curve===\'os\'?-e.sign:e.sign;\n    const opposite=e.curve===\'os\'?\'ds\':\'os\';\n    const oppositeAtPrev=SERIES[opposite]+(1+r.changes.fx.filter(ev=>ev.curve===opposite&&ev.stage<e.stage).length);\n    return [lead,\n      `Al tipo de cambio de equilibrio anterior (TCe${prev}), la ${e.curve===\'os\'?\'oferta\':\'demanda\'} de dólares ${next} es ${e.sign>0?\'mayor\':\'menor\'} que ${oppositeAtPrev}; se genera un exceso de ${ts>0?\'demanda\':\'oferta\'} de dólares.`,\n      `Ese exceso genera presión para que el tipo de cambio ${ts>0?\'aumente\':\'disminuya\'} de TCe${prev} a TCe${idx}.`,\n      `Finalmente, se alcanza el nuevo equilibrio cambiario (${eq1}) con un tipo de cambio ${ts>0?\'mayor\':\'menor\'} (TCe${idx}).`\n    ];\n  }\n  function enrichOriginal(t,r,m,e,i){\n    let x=specialRefFix(t), k=r.changes[m].indexOf(e), next=movingCurve(r,m,e),prev=previousCurve(r,m,e);\n    if(i===0){\n      const lead=causalLead(null,r,m,e,prev,next);\n      if(m===\'goods\'&&e.stage===3){x=x.replace(/^Al aumentar los salarios|^Al disminuir los salarios/i,`Debido a que los salarios ${e.sign>0?\'aumentan\':\'disminuyen\'} de We1 a We2`);x+=` También intervienen el tipo de cambio (TCe1 a TCe2) y la tasa de interés (ie1 a ie2), que modifican X − M e I. La demanda agregada cambia de ${prev} a ${next}.`}\n      else if(m===\'labor\'&&e.stage>1){x=x.replace(/\\(DL\\)/g,`(${prev} a ${next})`).replace(/\\(DL2\\)/g,`(${prev} a ${next})`);}\n      else if(m===\'money\'&&e.curve===\'md\'){x=x.replace(/\\(Md1\\)/g,`(${prev} a ${next})`);}\n      else if(m===\'fx\'&&e.curve===\'ds\'){x=x.replace(/\\(D\\$1 a D\\$2\\)/g,`(${prev} a ${next})`);}\n      if(!x.includes(next))x+=` En términos de curvas, ${prev} se desplaza a ${next}.`; if(!new RegExp(\'\\b(?:aumenta|disminuye|aumentan|disminuyen|sube|baja|mayor|menor)\\b\',\'i\').test(x))x+=` El cambio es ${e.sign>0?\'un aumento\':\'una disminución\'} de la curva correspondiente.`;\n      if(!x.includes(\'Debido\') && !x.includes(\'debido\') && e.stage>1){x=(m===\'labor\'||m===\'money\'||m===\'fx\')?lead:lead+\' \'+x;}\n    }\n    if(i===1 && m===\'goods\')x=x.replace(/\\bDA2\\b|\\bDA3\\b/g,next);\n    return x;\n  }\n  function paragraphsFor(c,r,m){\n    const evs=r.changes[m].filter(e=>c.showSecond||e.round!==\'segundo\');\n    const vb=v=>v>0?\'aumenta\':v<0?\'disminuye\':\'se mantiene\';\n    const pl=v=>v>0?\'aumentan\':v<0?\'disminuyen\':\'se mantienen\';\n    const greater=v=>v>0?\'mayor\':v<0?\'menor\':\'igual\';\n    return evs.map(e=>{\n      const k=r.changes[m].indexOf(e),prev=previousCurve(r,m,e),next=movingCurve(r,m,e);\n      const oldEq=k+1,newEq=k+2;\n      let paras=[];\n      if(m===\'goods\'){\n        paras=[\n          `${e.why} La curva de demanda agregada ${prev} pasa a ${next} (${vb(e.sign)}), según DA = C + I + G + X − M.`,\n          `A la producción Ye${oldEq} del equilibrio anterior, ${e.sign>0?\'DA es mayor que la OA\':\'OA es mayor que la DA\'}: existe exceso de ${e.sign>0?\'demanda\':\'oferta\'} agregada.`,\n          `Los inventarios ${e.sign>0?\'comienzan a agotarse\':\'se acumulan\'}; las empresas reciben incentivos para ${e.sign>0?\'aumentar\':\'reducir\'} la producción y los precios afrontan presión ${e.sign>0?\'alcista\':\'bajista\'}.`,\n          `Se establece E${newEq} con una producción ${greater(e.sign)}: Y pasa de Ye${oldEq} a Ye${newEq} (${vb(e.sign)}).`\n        ];\n      }else if(m===\'labor\'){\n        const w=e.curve===\'ol\'?-e.sign:e.sign;\n        paras=[\n          `${e.why} La curva ${prev} se desplaza a ${next}, conforme a ${e.curve===\'ol\'?\'OL = f(migración, educación, desastres)\':\'DL = f(Y, tecnología)\'}.`,\n          `Al salario We${oldEq}, la cantidad de trabajo ${w>0?\'demandada supera la ofrecida\':\'ofrecida supera la demandada\'}; existe exceso de ${w>0?\'demanda\':\'oferta\'} laboral.`,\n          `Los salarios ${pl(w)}: We${oldEq} pasa a We${newEq}. En respuesta se ajustan las cantidades ofrecidas y demandadas de trabajo.`,\n          `En E${newEq}, el salario es ${greater(w)} (We${newEq}) y el empleo de equilibrio ${vb(e.sign)} de Le${oldEq} a Le${newEq}. En un shock migratorio, este cambio de empleo no permite deducir por sí solo la tasa de desempleo.`\n        ];\n      }else if(m===\'money\'){\n        const i=e.curve===\'ms\'?-e.sign:e.sign;\n        paras=[\n          `${e.why} La curva ${prev} se desplaza a ${next}, mientras la otra curva monetaria permanece fija en este paso.`,\n          `A la tasa ie${oldEq}, hay exceso de ${i>0?\'demanda\':\'oferta\'} de dinero, según las condiciones iniciales de Ms y Md.`,\n          `La tasa de interés nacional ${vb(i)} desde ie${oldEq} hasta ie${newEq}; se reajusta la cantidad demandada de dinero para restablecer el equilibrio.`,\n          `En el nuevo equilibrio E${newEq}, la tasa de interés es ${greater(i)} (ie${newEq}) y Ms = Md.`\n        ];\n      }else{\n        const tc=e.curve===\'os\'?-e.sign:e.sign;\n        paras=[\n          `${e.why} La curva ${prev} pasa a ${next}.`,\n          `Al tipo de cambio TCe${oldEq}, la cantidad ${tc>0?\'demandada supera la ofrecida\':\'ofrecida supera la demandada\'} de dólares, lo que produce un exceso de ${tc>0?\'demanda\':\'oferta\'} de divisas.`,\n          `El tipo de cambio ${vb(tc)} desde TCe${oldEq} hasta TCe${newEq}; oferentes y demandantes ajustan cantidades.`,\n          `El equilibrio cambiario E${newEq} se alcanza a un TC ${greater(tc)} (TCe${newEq}) con O$ = D$.`\n        ];\n      }\n      return {event:e,paragraphs:paras.map((x,j)=>c.overrides?.[`${m}:${k}:${j}`]??x),source:\'Causalidad verificada con la secuencia\'};\n    });\n  }\n  function varColorFor(r,token){\n    const m=String(token).match(/^(DA|DL|OL|Ms|Md|O\\$|D\\$|We|Le|Ye|TCe|ie|IS|BP)(\\d+)$/);\n    if(!m)return \'#1e3b50\';let v=m[1],num=+m[2];if(num===1)return \'#172d41\';\n    let mk={DA:\'goods\',DL:\'labor\',OL:\'labor\',Ms:\'money\',Md:\'money\',\'O$\':\'fx\',\'D$\':\'fx\'}[v];\n    if(!mk)return num===2?\'#1879b9\':\'#cc3889\';\n    let code={DA:\'da\',DL:\'dl\',OL:\'ol\',Ms:\'ms\',Md:\'md\',\'O$\':\'os\',\'D$\':\'ds\'}[v];\n    let ev=r.changes[mk].filter(e=>e.curve===code)[num-2];return ev?changedColor(ev.stage,num,ev.round):num===2?\'#1879b9\':\'#cc3889\';\n  }\n  function markVars(src,r){const safe=esc(src);return safe.replace(/(?<![\\w$])(?:TCe|DA|DL|OL|Ms|Md|O\\$|D\\$|We|Le|Ye|ie|IS|BP)\\d+\\b/g,(v)=>`<strong class="economic-token" style="--token:${varColorFor(r,v)}">${v}</strong>`);}\n  function explanations(c,r,m){if(!c.showExplanation)return \'\';\n    if(m===\'is\'){\n      let t=refSection(c,\'is\')?.[0]||r.explanations.is||\'La IS recoge las combinaciones de tasa de interés y producción.\';\n      t=c.overrides?.[\'is:0:0\']??t;\n      return `<div class="narrative"><div class="narrative-header"><strong>MODELO IS</strong><span>Interpretación</span></div><div class="paragraphs"><div class="paragraph"><span class="para-n">1</span><p>${markVars(t,r)}</p></div></div></div>`;\n    }\n    return paragraphsFor(c,r,m).map(sec=>`<div class="narrative ${sec.event.round===\'segundo\'?\'narrative-second\':\'\'}"><div class="narrative-header"><strong>${sec.event.round===\'segundo\'?\'EFECTO DE SEGUNDO ORDEN\':\'EFECTO DE PRIMER ORDEN\'}</strong><div><span class="small-stage">SECUENCIA ${sec.event.stage}</span></div></div><div class="paragraphs">${sec.paragraphs.map((v,j)=>`<div class="paragraph"><span class="para-n">${j+1}</span><p>${markVars(v,r)}</p></div>`).join(\'\')}</div></div>`).join(\'\');\n  }\n  const SVG={w:800,h:570,ox:155,oy:452,sx:4.65,sy:3.78};\n  const px=x=>SVG.ox+SVG.sx*x,py=y=>SVG.oy-SVG.sy*y;\n  const fmt=x=>Number(x).toFixed(1);\n  const text=(x,y,content,cls=\'axislabel\',extra=\'\')=>`<text x="${fmt(x)}" y="${fmt(y)}" class="${cls}" ${extra}>${esc(content)}</text>`;\n  const line=(x1,y1,x2,y2,cls,extra=\'\')=>`<line x1="${fmt(x1)}" y1="${fmt(y1)}" x2="${fmt(x2)}" y2="${fmt(y2)}" class="${cls}" ${extra}/>`;\n  const circle=(x,y,r,cls)=>`<circle cx="${fmt(x)}" cy="${fmt(y)}" r="${r}" class="${cls}"/>`;\n  const mstate=m=>({da:20,ol:20,dl:85,ms:52,md:87,os:15,ds:86});\n  function evolution(m,events){let b=mstate(m),states=[{...b,stage:0,change:\'none\',round:\'base\'}];for(const e of events){let amt=(e.round===\'segundo\'?6:11)*e.sign;let c=e.curve;if([\'da\',\'dl\',\'md\',\'ds\'].includes(c))b[c]+=amt;else if([\'ol\',\'os\'].includes(c))b[c]-=amt;else if(c===\'ms\')b[c]+=amt;states.push({...b,stage:e.stage,change:c,round:e.round})}return states}\n  function point(m,s){if(m===\'goods\'){let x=s.da/.5;return {x,y:x}}\n    if(m===\'labor\'){let x=(s.dl-s.ol)/1.4;return {x,y:s.ol+.65*x}}\n    if(m===\'money\'){let x=s.ms;return {x,y:s.md-.72*x}}\n    let x=(s.ds-s.os)/1.4;return{x,y:s.os+.67*x};}\n  const formula=(m,curve,s,x)=>{if(curve===\'oa\')return x;if(curve===\'da\')return s.da+.5*x;if(curve===\'ol\')return s.ol+.65*x;if(curve===\'dl\')return s.dl-.75*x;if(curve===\'md\')return s.md-.72*x;if(curve===\'os\')return s.os+.67*x;if(curve===\'ds\')return s.ds-.73*x;return 0};\n  const clabel={da:\'DA\',oa:\'OA\',ol:\'OL\',dl:\'DL\',ms:\'Ms\',md:\'Md\',os:\'O$\',ds:\'D$\'};\n  const palette=[\'#172d41\',\'#1879b9\',\'#ce418d\',\'#7e56c2\',\'#0b8d76\',\'#b66b11\',\'#e05343\',\'#6d5f9f\',\'#22777a\',\'#b35387\',\'#578c20\',\'#374cb1\',\'#b76e59\',\'#0f7a69\'];\n  function diagram(m,c,r){const ev=r.changes[m].filter(e=>c.showSecond||e.round!==\'segundo\'), states=evolution(m,ev), chart=[], seen={}, pointSet=states.map(s=>point(m,s));const suffix=SUFFIX[m];\n    chart.push(`<svg class="economic-svg" viewBox="0 0 ${SVG.w} ${SVG.h}" role="img" aria-label="Gráfico del mercado ${esc(TITLES[m])}, con curvas y equilibrios" xmlns="http://www.w3.org/2000/svg"><defs><marker id="axis-${m}" markerWidth="7" markerHeight="7" refX="5.8" refY="3.5" orient="auto"><path d="M0 0 L7 3.5 L0 7 Z" fill="#24384b"/></marker></defs>`);\n    for(let n=20;n<=80;n+=20){chart.push(line(px(n),py(0),px(n),py(100),\'gridline\'));chart.push(line(px(0),py(n),px(100),py(n),\'gridline\'))}\n    chart.push(line(px(0),py(0),px(104),py(0),\'plot-axis\',`marker-end="url(#axis-${m})"`),line(px(0),py(0),px(0),py(104),\'plot-axis\',`marker-end="url(#axis-${m})"`));\n    const ax={goods:[\'Y · Producción nacional\',\'Colones\'],labor:[\'L · Empleo\',\'W · Salarios\'],money:[\'Colones\',\'i · Tasa de interés\'],fx:[\'Dólares\',\'TC · Tipo de cambio\']}[m];\n    chart.push(text(px(50),552,ax[0],\'axis-title\',\'text-anchor="middle"\'),text(px(0)+3,32,ax[1],\'axis-title\',\'text-anchor="start"\'));\n    if(m===\'goods\'){\n      chart.push(line(px(0),py(0),px(96),py(96),\'fixed-line\'));\n      chart.push(text(px(85),py(89),\'OA\',\'curve-label fixed\', \'text-anchor="start"\'));\n    }\n    const changing=m===\'goods\'?[\'da\']:m===\'labor\'?[\'ol\',\'dl\']:m===\'money\'?[\'ms\',\'md\']:[\'os\',\'ds\'];\n    const drawnLabels=[];\n    for(const key of changing){\n      const variants=[];\n      for(const st of states){if(!variants.some(z=>z[key]===st[key]))variants.push(st)}\n      variants.forEach((st,i)=>{\n        const col=i===0?\'#24384b\':changedColor(st.stage,i+1,st.round);\n        let x0,x1,y0,y1;\n        if(key===\'ms\'){x0=x1=st.ms;y0=4;y1=97}\n        else{x0=0;x1=key===\'da\'?97:96;y0=formula(m,key,st,x0);y1=formula(m,key,st,x1)}\n        chart.push(line(px(x0),py(y0),px(x1),py(y1),\'curve-line\',`stroke="${col}"`));\n        const labelX=key===\'da\'?90:key===\'ms\'?st.ms:key===\'os\'?81:key===\'ds\'?60:69;const labelY=key===\'ms\'?88:formula(m,key,st,labelX);const a=key===\'ms\'?{x:px(labelX)+10,y:py(labelY)}:{x:px(labelX)+13,y:py(labelY)-5};\n        // Ubicar cada rótulo a la altura REAL de su curva: el orden visual\n        // de DA1/DA2/DA3 nunca puede invertirse para "evitar" un solapamiento.\n        // La separación natural entre desplazamientos supera la altura del texto.\n        let ly=Math.max(50,Math.min(a.y,443));\n        drawnLabels.push({x:a.x,y:ly,yOriginal:a.y,label:clabel[key]+(i+1),col});\n      });\n    }\n    // Proyecciones de equilibrio: se calculan desde la intersección, sin desplazar los valores económicos.\n    pointSet.forEach((q,i)=>{\n      const col=i===0?palette[0]:changedColor(states[i].stage,i+1,states[i].round),cx=px(q.x),cy=py(q.y);\n      chart.push(line(px(0),cy,cx,cy,\'guide\',`stroke="${col}"`),line(cx,cy,cx,py(0),\'guide\',`stroke="${col}"`));\n    });\n    // Evitar que We1/We2, TCe1/TCe2 o Ye1/Ye2 se tapen cuando los equilibrios estén próximos.\n    const rows=[],sides=[];\n    pointSet.forEach((q,i)=>{\n      const col=i===0?palette[0]:changedColor(states[i].stage,i+1,states[i].round),cx=px(q.x),cy=py(q.y);\n      if(m===\'goods\'||m===\'labor\'){\n        let track=0;\n        while(rows.some(z=>z.track===track && Math.abs(z.x-cx)<65) && track<3)track++;\n        rows.push({x:cx,track});\n        const label=(m===\'goods\'?\'Ye\':\'Le\')+(i+1);\n        chart.push(text(cx,py(0)+23+track*18,label,\'projection-label\',`fill="${col}" text-anchor="middle"`));\n      }\n      if(m!==\'goods\'){\n        let labelY=cy+4;\n        let attempts=0;\n        while(sides.some(z=>Math.abs(z.y-labelY)<21) && attempts<8){labelY+=(attempts%2===0?-23:46);attempts++;}\n        labelY=Math.max(47,Math.min(442,labelY));\n        sides.push({y:labelY});\n        const targetX=px(0)-15;\n        if(Math.abs(labelY-(cy+4))>5)chart.push(line(px(0)-2,cy,targetX+5,labelY-4,\'annotation-leader\',`stroke="${col}"`));\n        chart.push(text(targetX,labelY,(m===\'labor\'?\'We\':m===\'money\'?\'ie\':\'TCe\')+(i+1),\'projection-label\',`fill="${col}" text-anchor="end"`));\n      }\n    });\n    // Equilibrios en intersección exacta; separar E1/E2/E3 por una leyenda de líderes solo si colisionan.\n    const crowded=pointSet.some((q,i)=>pointSet.some((w,j)=>j>i && Math.hypot(px(w.x)-px(q.x),py(w.y)-py(q.y))<43));\n    pointSet.forEach((q,i)=>{\n      const col=i===0?palette[0]:changedColor(states[i].stage,i+1,states[i].round),cx=px(q.x),cy=py(q.y);\n      if(crowded){\n        chart.push(`<circle cx="${fmt(cx)}" cy="${fmt(cy)}" r="7.2" fill="#fff" stroke="${col}" stroke-width="3"/>`);\n        const labelX=742,labelY=119+i*44;\n        chart.push(line(cx+7,cy,labelX-26,labelY-5,\'annotation-leader\',`stroke="${col}"`));\n        chart.push(`<circle cx="${fmt(labelX)}" cy="${fmt(labelY)}" r="16" fill="#fff" stroke="${col}" stroke-width="3"/>`);\n        chart.push(text(labelX,labelY+5,\'E\'+(i+1),\'equilibrium-text\',`fill="${col}" text-anchor="middle"`));\n      }else{\n        chart.push(`<circle cx="${fmt(cx)}" cy="${fmt(cy)}" r="16" fill="#fff" stroke="${col}" stroke-width="3.2"/>`);\n        chart.push(text(cx,cy+5,\'E\'+(i+1),\'equilibrium-text\',`fill="${col}" text-anchor="middle"`));\n      }\n    });\n    drawnLabels.forEach(l=>{\n      if(Math.abs(l.yOriginal-l.y)>8)chart.push(line(l.x-3,l.yOriginal,l.x+1,l.y-4,\'annotation-leader\',`stroke="${l.col}"`));\n      chart.push(text(l.x,l.y,l.label,\'curve-label\',`fill="${l.col}" text-anchor="start"`));\n    });\n    // Los números de secuencia están arriba de cada tarjeta; la gráfica mantiene solo E, ejes y curvas.\n    chart.push(\'</svg>\');return chart.join(\'\');\n  }\n  function modelISLM(c,r){\n    const z=r.islm||{is:0,lm:0,bp:0};\n    const w=900,h=565,x0=140,y0=455,sx=5.25,sy=3.9;\n    const x=v=>x0+sx*v,y=v=>y0-sy*v;\n    const out=[`<svg class="economic-svg combined-svg" viewBox="0 0 ${w} ${h}" role="img" aria-label="Curvas IS–LM–BP iniciales y finales"><defs><marker id="v5-arrow" markerWidth="9" markerHeight="9" refX="7" refY="4.5" orient="auto"><path d="M0 0L9 4.5L0 9Z" fill="#263f50"/></marker></defs>`];\n    const l=(a,b,c,d,stroke,width=3,dash=\'\')=>`<line x1="${a.toFixed(1)}" y1="${b.toFixed(1)}" x2="${c.toFixed(1)}" y2="${d.toFixed(1)}" stroke="${stroke}" stroke-width="${width}" ${dash?`stroke-dasharray="${dash}"`:\'\'}/>`;\n    const label=(a,b,t,color,anch=\'start\',size=20)=>`<text x="${a.toFixed(1)}" y="${b.toFixed(1)}" fill="${color}" font-size="${size}" font-weight="700" text-anchor="${anch}">${esc(t)}</text>`;\n    for(let a=20;a<100;a+=20)out.push(l(x(a),y(0),x(a),y(100),\'#e8edf2\',1),l(x(0),y(a),x(104),y(a),\'#e8edf2\',1));\n    out.push(l(x(0),y(0),x(112),y(0),\'#263f50\',3),l(x(0),y(0),x(0),y(105),\'#263f50\',3));\n    out.push(label(x(48),548,\'Y · Producción nacional\',\'#24384b\',\'middle\',21),label(x(0),34,\'i · Tasa de interés\',\'#24384b\',\'start\',20));\n    // The final equilibrium E2 follows IS and LM intersection; BP is drawn independently.\n    const isB=85, lmB=11, kI=.66,kL=.66,kB=.82;\n    const iShift=13*(z.is||0),lShift=18*(z.lm||0),bpShift=14*(z.bp||0);\n    const eq=(iS,lS)=>{const u=(isB-lmB+iS+lS)/(kI+kL);return {u,v:isB+iS-kI*u}};\n    const e1=eq(0,0),e2=eq(iShift,lShift),bpB=e1.v-kB*e1.u;\n    // Geometría económica intacta; las etiquetas se posicionan fuera de las curvas\n    // con líderes claros, evitando que BP1/BP2/LM1 se encimen.\n    const curveLabels=[];\n    const curve=(tag,kind,shift,color,labelOffset)=>{\n      const a=15,b=100;let f;\n      if(kind===\'IS\')f=t=>isB+shift-kI*t;\n      else if(kind===\'LM\')f=t=>lmB-shift+kL*t;\n      else f=t=>bpB-shift+kB*t;\n      out.push(l(x(a),y(f(a)),x(b),y(f(b)),color,kind===\'BP\'?3.4:3.6,kind===\'BP\'?\'9 5\':\'\'));\n      const anchor=100;\n      curveLabels.push({tag,color,kind,anchorX:x(anchor),anchorY:y(f(anchor)),desiredY:y(f(anchor))+labelOffset});\n    };\n    curve(\'IS1\',\'IS\',0,\'#283f50\',10);\n    curve(\'LM1\',\'LM\',0,\'#956b4c\',-7);\n    curve(\'BP1\',\'BP\',0,\'#188a78\',-6);\n    if(z.is)curve(\'IS2\',\'IS\',iShift,\'#1679be\',-4);\n    if(z.lm)curve(\'LM2\',\'LM\',lShift,\'#d23f88\',-13);\n    if(z.bp)curve(\'BP2\',\'BP\',bpShift,\'#784fbe\',12);\n    // Una sola columna de rótulos y separación mínima de 27 px.\n    const ranked=curveLabels.slice().sort((a,b)=>a.desiredY-b.desiredY);\n    let last=40;\n    for(const lab of ranked){lab.outY=Math.max(42,Math.min(498,lab.desiredY),last+28);last=lab.outY;}\n    const overflow=Math.max(0,last-492);\n    if(overflow)for(const lab of ranked)lab.outY-=overflow;\n    for(const lab of ranked){\n      const tx=746,ty=lab.outY;\n      out.push(l(lab.anchorX+2,lab.anchorY,tx-9,ty-6,lab.color,1.55,\'4 4\'));\n      out.push(`<rect x="${tx-3}" y="${ty-19}" width="65" height="24" rx="5" fill="#fff" opacity=".97"/>`);\n      out.push(label(tx,ty,lab.tag,lab.color,\'start\',18));\n    }\n    const dot=(t,name,col,n)=>{const cx=x(t.u),cy=y(t.v);out.push(l(cx,cy,cx,y(0),col,1.3,\'5 5\'),l(x(0),cy,cx,cy,col,1.3,\'5 5\'));out.push(`<circle cx="${cx.toFixed(1)}" cy="${cy.toFixed(1)}" r="15" fill="white" stroke="${col}" stroke-width="3"/>`);out.push(label(cx,cy+6,name,col,\'middle\',16));out.push(label(cx,y(0)+22+n*22,\'Ye\'+(n+1),col,\'middle\',15),label(x(0)-18,cy+4,\'ie\'+(n+1),col,\'end\',15));};\n    dot(e1,\'E1\',\'#283f50\',0);\n    if(Math.hypot(e2.u-e1.u,e2.v-e1.v)>1){dot(e2,\'E2\',\'#cf398a\',1);}\n    out.push(\'</svg>\');return out.join(\'\');\n  }\n  function islmExplanation(c,r){\n    const z=r.islm||{is:0,lm:0,bp:0,isDrivers:[]};\n    const mv=v=>v>0?\'aumenta\':v<0?\'disminuye\':\'se mantiene\';\n    const to=v=>v>0?\'la derecha\':v<0?\'la izquierda\':\'la posición original\';\n    const active=(z.isDrivers||[]).filter(d=>c.showSecond||r.changes.goods.some(e=>e.stage===d.stage&&e.round!==\'segundo\'));\n    const iOnly=r.changes.goods.some(e=>e.kind===\'i-investment\');\n    const p1=z.is?`IS se desplaza hacia ${to(z.is)} desde IS1 a IS2. La causa concreta es: ${active.map(d=>\'secuencia \'+d.stage+\': \'+d.cause).join(\' Además, \')}. ${iOnly?\'La inversión modificada únicamente por i NO es causa del desplazamiento.\':\'\'}`:\n      `IS1 mantiene su posición. ${iOnly?\'Aunque I cambia por la tasa de interés, ese movimiento no desplaza IS.\':\'No se identifica un componente autónomo que la desplace.\'}`;\n    const p2=z.lm?`La oferta de dinero Ms ${mv(z.lm)} por actuación del BCCR; por ello LM pasa de LM1 a LM2 hacia ${to(z.lm)}.`:\'El BCCR no ha modificado Ms. Por tanto, la curva LM1 permanece fija; variaciones de Md modifican la tasa de interés, no constituyen por sí solas un desplazamiento de LM.\';\n    const p3=z.bp?`BP se desplaza hacia ${to(z.bp)} de BP1 a BP2. ${z.bpReason}`:`BP1 mantiene su posición. ${z.bpReason||\'No hay cambio autónomo identificado en el ingreso o salida de divisas.\'}`;\n    const p4=z.is||z.lm?`La nueva intersección IS–LM determina E2, con producción Ye2 y tasa de interés ie2 en las proyecciones. La curva BP refleja por separado la condición externa, que no se fuerza artificialmente a pasar por E2.`:\n      \'La intersección IS–LM continúa en E1. Las variaciones de equilibrio parcial están representadas en los cuatro mercados anteriores.\';\n    return [p1,p2,p3,p4];\n  }\n  // Biblioteca de relaciones, transcrita de las presentaciones del curso.\n  // f(...) significa dependencia cualitativa, no fórmula con coeficientes estimados.\n  function formulas(c,r){\n    const blocks=[\n      {m:\'Bienes y servicios · Y\',eq:\'DA = C + I + G + X − M\',lines:[\n        [\'Consumo\',\'C = C₀ + PmgC × Yd; Yd = f(W, impuestos, transferencias, riqueza familiar)\'],\n        [\'Inversión\',\'I = f(i, acceso al crédito, tramitología, disponibilidad de inversión)\'],\n        [\'Gasto público\',\'G: compras, salarios públicos, infraestructura, transferencias y otras erogaciones; instrumento fiscal\'],\n        [\'Exportaciones\',\'X = f(TC, Y*, P, P*)\'],\n        [\'Importaciones\',\'M = f(TC, Y, P, P*)\']\n      ]},\n      {m:\'Mercado laboral · W, L\',eq:\'OL = DL en equilibrio\',lines:[\n        [\'Oferta laboral\',\'OL = f(migración, educación, capacitación, desastres, PEA)\'],\n        [\'Demanda laboral\',\'DL = f(Y, tecnología)\'],\n        [\'Variable de salida\',\'W: salarios; L: empleo de equilibrio\']\n      ]},\n      {m:\'Mercado monetario · i\',eq:\'Ms = Md en equilibrio\',lines:[\n        [\'Oferta monetaria\',\'Ms = f(instrumentos del BCCR: encaje, OMA/BEMs, TPM/redescuento, emisión)\'],\n        [\'Demanda de dinero\',\'Md = f(Y, transacciones, precaución, especulación, deuda interna)\'],\n        [\'Variable de salida\',\'i: tasa de interés de equilibrio; más Ms suele reducir i, mayor Md suele elevar i\']\n      ]},\n      {m:\'Mercado cambiario · TC\',eq:\'O$ = D$ en equilibrio\',lines:[\n        [\'Oferta de dólares\',\'O$ = f(exportaciones, turismo receptor, IED, crédito externo, diferencial i − i*)\'],\n        [\'Demanda de dólares\',\'D$ = f(importaciones, turismo emisor, inversión y préstamos al exterior)\'],\n        [\'Variable de salida\',\'TC: colones por dólar; O$↑ suele implicar TC↓, y TC↓ favorece M y reduce X bajo los supuestos del curso\']\n      ]}\n    ];\n    return `<details class="formula-library"><summary>Formulario económico · relaciones de los cuatro mercados</summary><div class="formula-intro">Estas funciones explican qué variable mueve cada curva. <b>f(…)</b> indica dependencia cualitativa: no se asignan coeficientes no observados. Las expresiones corresponden a las presentaciones de Macroeconomía II del curso.</div><div class="formula-grid">${blocks.map(b=>`<section class="formula-block"><h4>${esc(b.m)}</h4><div class="formula-main">${esc(b.eq)}</div>${b.lines.map(([lab,t])=>`<p><strong>${esc(lab)}:</strong> ${esc(t)}</p>`).join(\'\')}</section>`).join(\'\')}</div><p class="formula-legend">Y: producción; Y*: ingreso de socios comerciales; P y P*: precios nacionales e internacionales; C₀: consumo autónomo; PmgC: propensión marginal a consumir; TC: colones por USD; i*: tasa internacional.</p></details>`;\n  }\n  // Diagnóstico del plano IS–LM–BP. Separa condición inicial de política aplicada.\n  // Las coordenadas son ilustrativas, sin calibración macroeconómica del país.\n  const policySamples=[\n    {id:\'A\',Y:42,i:77,policy:\'expansion_monetaria\',dir:1,title:\'Política monetaria expansiva\',body:\'Ms ↑ mediante instrumentos del BCCR → presión inicial a reducir i. Ejercicio del grupo 1.\'},\n    {id:\'B\',Y:42,i:54,policy:\'gasto_publico\',dir:1,title:\'Política fiscal expansiva\',body:\'G ↑ → DA ↑ y desplazamiento de IS. Ejercicio del grupo 2.\'},\n    {id:\'C\',Y:72,i:18,policy:\'contraccion_monetaria\',dir:1,title:\'Política monetaria restrictiva\',body:\'Ms ↓ mediante instrumentos del BCCR → presión inicial a elevar i. Ejercicio del grupo 5.\'}\n  ];\n  function policyDiagnostic(p){\n    const IS=85-.66*p.Y,LM=11+.66*p.Y,BP=-10.8+1.05*p.Y;\n    const goods=p.i<IS?\'Exceso de demanda agregada (DA)\':\'Exceso de oferta agregada (OA)\';\n    const money=p.i<LM?\'Exceso de demanda de dinero (Md)\':\'Exceso de oferta de dinero (Ms)\';\n    const dollars=p.i<BP?\'Exceso de demanda de dólares (D$)\':\'Exceso de oferta de dólares (O$)\';\n    const sample=policySamples.find(s=>Math.hypot(s.Y-p.Y,s.i-p.i)<.2);\n    return {goods,money,dollars,sample};\n  }\n  function policyMap(c){\n    const p=diagnosticPoint,d=policyDiagnostic(p);\n    const ox=98,oy=365,sx=5.7,sy=3.25;\n    const X=v=>ox+sx*v,Y=v=>oy-sy*v;\n    const line=(x1,y1,x2,y2,color)=>`<line x1="${X(x1)}" y1="${Y(y1)}" x2="${X(x2)}" y2="${Y(y2)}" stroke="${color}" stroke-width="3" stroke-linecap="round"/>`;\n    const pins=policySamples.map(s=>`<g class="policy-pin ${d.sample?.id===s.id?\'active\':\'\'}" data-act="diag-sample" data-id="${s.id}" tabindex="0" role="button" aria-label="Seleccionar diagnóstico ${s.id}"><circle cx="${X(s.Y)}" cy="${Y(s.i)}" r="13"/><text x="${X(s.Y)}" y="${Y(s.i)+4}" text-anchor="middle">${s.id}</text></g>`).join(\'\');\n    return `<section class="policy-panel"><div class="policy-heading"><div class="market-eyebrow">ANTES DE LA INTERVENCIÓN · DIAGNÓSTICO OPCIONAL</div><h3>¿Dónde está la economía en el plano IS–LM–BP?</h3><p>Elegí uno de los tres casos de la clase o tocá cualquier punto del plano para identificar los excesos. Esta vista es un <b>diagnóstico inicial ilustrativo</b>, no el equilibrio final del escenario.</p></div><div class="policy-layout"><div class="policy-graph"><svg class="policy-svg" viewBox="0 0 760 420" role="img" aria-label="Mapa de exceso de oferta y demanda entre las curvas IS, LM y BP; seleccioná un punto" data-diag-plot="1"><line x1="${X(0)}" y1="${Y(0)}" x2="${X(105)}" y2="${Y(0)}" stroke="#30485b" stroke-width="2.5"/><line x1="${X(0)}" y1="${Y(0)}" x2="${X(0)}" y2="${Y(105)}" stroke="#30485b" stroke-width="2.5"/>${line(10,85-.66*10,100,85-.66*100,\'#1879b9\')}${line(10,11+.66*10,100,11+.66*100,\'#c93e85\')}${line(10,-10.8+1.05*10,100,-10.8+1.05*100,\'#0b8d76\')}<text x="${X(100)}" y="${Y(85-.66*100)-10}" class="policy-curve" fill="#1879b9">IS</text><text x="${X(102)}" y="${Y(11+.66*102)-10}" class="policy-curve" fill="#c93e85">LM</text><text x="${X(89)}" y="${Y(-10.8+1.05*89)-12}" class="policy-curve" fill="#0b8d76">BP</text><text x="${X(108)}" y="${Y(0)+20}" fill="#415e73" text-anchor="end" font-size="15">Y · Producción</text><text x="${X(0)}" y="${Y(107)}" font-size="15" fill="#415e73">i · Tasa de interés</text><circle cx="${X(56)}" cy="${Y(48)}" r="4" fill="#233b50"/><text x="${X(56)+12}" y="${Y(48)-9}" fill="#233b50" font-size="15">E</text>${pins}<circle cx="${X(p.Y)}" cy="${Y(p.i)}" r="18" fill="none" stroke="#efaa23" stroke-width="2.8" stroke-dasharray="5 3" pointer-events="none"/></svg><p class="policy-chart-note">Plano esquemático: las distancias no representan magnitudes observadas ni garantizan que una intervención resuelva todos los excesos.</p></div><div class="policy-reading"><div class="policy-choices">${policySamples.map(s=>`<button type="button" class="button ${d.sample?.id===s.id?\'primary\':\'outline\'} small" data-act="diag-sample" data-id="${s.id}">Caso ${s.id}</button>`).join(\'\')}</div><h4>Desequilibrios identificados</h4><p><b>Bienes:</b> ${d.goods}</p><p><b>Dinero:</b> ${d.money}</p><p><b>Divisas:</b> ${d.dollars}</p><div class="policy-recommendation"><strong>${d.sample?d.sample.title:\'Sin intervención única prevalidada\'}</strong><p>${d.sample?d.sample.body:\'Para este punto libre, las presentaciones no establecen una política única. Contrastá los tres excesos antes de escoger una medida fiscal, monetaria o combinada.\'}</p></div>${d.sample?`<button type="button" class="button primary" data-act="diag-apply" data-id="${d.sample.id}">Aplicar ${d.sample.id} y resolver los cuatro mercados</button>`:\'\'}<small class="policy-note">La medida sugerida es un punto de partida docente, no una garantía de equilibrio general. El gráfico IS–LM–BP del caso se recalcula debajo.</small></div></div></section>`;\n  }\n  function routeGraph(c,r){\n    const all=r.sequence.filter(e=>c.showSecond||e.round!==\'segundo\');\n    const nodes=new Set(all.map(e=>e.id));\n    const edges=(r.edges||[]).filter(e=>nodes.has(e.from)&&nodes.has(e.to)&&Number(e.from.split(\':\')[1])<Number(e.to.split(\':\')[1]));\n    const markets=[\'goods\',\'labor\',\'money\',\'fx\'],row={goods:87,labor:187,money:287,fx:387};\n    const max=Math.max(3,...all.map(e=>e.stage)),width=1220,height=490,left=262;\n    const dx=(width-left-88)/Math.max(1,max-1),x=n=>left+(n-1)*dx;\n    const idTo=new Map(all.map(e=>[e.id,e]));\n    const curve=e=>SERIES[e.curve]||e.curve;\n    const nodeCenter=e=>({x:x(e.stage),y:row[e.market]});\n    let svg=`<svg class="causal-svg" viewBox="0 0 ${width} ${height}" aria-label="Diagrama causal trazable de mercados y secuencias"><defs><marker id="c-arrow-blue" markerWidth="7" markerHeight="7" refX="6" refY="3.5" orient="auto"><path d="M0 0L7 3.5L0 7Z" fill="#1879b9"/></marker><marker id="c-arrow-rose" markerWidth="7" markerHeight="7" refX="6" refY="3.5" orient="auto"><path d="M0 0L7 3.5L0 7Z" fill="#cc3889"/></marker></defs>`;\n    for(let n=1;n<=max;n++){svg+=`<line x1="${x(n)}" y1="57" x2="${x(n)}" y2="424" stroke="#e5eef4" stroke-width="1" stroke-dasharray="5 8"/><text x="${x(n)}" y="28" text-anchor="middle" class="causal-col-label">SECUENCIA ${n}</text>`;}\n    for(const m of markets){svg+=`<text x="12" y="${row[m]+5}" class="causal-row-label">${esc(TITLES[m])}</text>`;}\n    // Fan-in routes are explicit: in the IED case three different 3\'s converge on one 4.\n    for(const e of edges){const a=nodeCenter(idTo.get(e.from)),b=nodeCenter(idTo.get(e.to));\n      const ax=a.x+44,bx=b.x-45,offset=Math.abs(a.y-b.y)>5?10:0;\n      const c1=ax+Math.max(18,(bx-ax)*.46),c2=bx-Math.max(18,(bx-ax)*.46);\n      const endEvent=idTo.get(e.to),isSecond=endEvent.round===\'segundo\';\n      const lineColor=isSecond?\'#cc3889\':\'#1879b9\', arrowId=isSecond?\'c-arrow-rose\':\'c-arrow-blue\';\n      svg+=`<path class="causal-edge ${isSecond?\'second\':\'first\'}" data-from="${e.from}" data-to="${e.to}" d="M ${ax} ${a.y} C ${c1} ${a.y-offset}, ${c2} ${b.y+offset}, ${bx} ${b.y}" fill="none" stroke="${lineColor}" stroke-width="2.0" marker-end="url(#${arrowId})"/>`;\n    }\n    for(const e of all){const a=nodeCenter(e),color=e.round===\'segundo\'?\'#fce8f2\':\'#e4f3fb\',stroke=e.round===\'segundo\'?\'#cc3889\':\'#1879b9\';\n      svg+=`<g class="causal-node" data-node-id="${e.id}" tabindex="0" role="button" aria-label="Secuencia ${e.stage}, ${esc(TITLES[e.market])}: ${esc(e.why)}"><rect x="${a.x-57}" y="${a.y-25}" width="114" height="50" rx="22" fill="${color}" stroke="${stroke}" stroke-width="2.2"/><text x="${a.x}" y="${a.y+5}" text-anchor="middle" fill="${e.round===\'segundo\'?\'#a82c6b\':\'#126a9e\'}" font-weight="850" font-size="17">${e.stage} · ${esc(curve(e))}${sign(e.sign)}</text><title>${esc(e.why)}</title></g>`;\n    }\n    svg+=\'</svg>\';\n    const tags=`<div class="causal-legend"><span><i class="legend-first"></i> Primer orden</span><span><i class="legend-second"></i> Segundo orden</span><span>Seleccioná un nodo para revisar su causa y sus conexiones</span></div>`;\n    return `<section class="causal-panel"><div class="causal-title"><div><div class="market-eyebrow">MAPA DE CAUSALIDAD</div><h3>Diagrama de mercados y secuencias</h3><p>La misma ruta alimenta el gráfico, la redacción y las flechas. Cada número se traza hasta sus causas.</p></div><span class="causal-cases-label">${r.routeMode===\'dual\'?\'DOS INICIOS\':\'RUTA SELECCIONADA\'}</span></div><div class="causal-map">${svg}</div>${tags}<p class="causal-detail" aria-live="polite">Seleccioná cualquier nodo del diagrama para ver su antecedente económico.</p></section>`;\n  }\n  function graphCard(m,c,r){const macroEq={goods:\'DA = C + I + G + X − M · OA = Y\',labor:\'OL = f(migración, educación, PEA) · DL = f(Y, tecnología)\',money:\'Ms = f(instrumentos del BCCR) · Md = f(Y, transacciones, deuda interna)\',fx:\'O$ = f(ingreso de divisas) · D$ = f(salida de divisas)\'};return `<article class="market-card" data-market="${m}"><div class="market-head"><div><div class="market-eyebrow">MERCADO</div><h3>${TITLES[m]} <small>(${CODES[m]})</small></h3></div><div class="market-numbers">${stepDots(c,r,m)}</div></div><p class="market-formula">${esc(macroEq[m])}</p><div class="chart-zone">${diagram(m,c,r)}</div>${explanations(c,r,m)}</article>`}\n  function isCard(c,r){\n    const z=r.islm||{is:0,lm:0};\n    const ps=islmExplanation(c,r);\n    return `<article class="market-card combined-card" data-market="is"><div class="market-head"><div><div class="market-eyebrow">EQUILIBRIO SIMULTÁNEO</div><h3>Modelo IS–LM–BP <small>(i, Y)</small></h3></div><div class="market-numbers"><span class="stage-none">${z.is?\'IS1 → IS2\':\'IS1 fija\'} · ${z.lm?\'LM1 → LM2\':\'LM1 fija\'} · ${z.bp?\'BP1 → BP2\':\'BP1 fija\'}</span></div></div><div class="chart-zone">${modelISLM(c,r)}</div>${c.showExplanation?`<div class="narrative"><div class="narrative-header"><strong>INTERPRETACIÓN DEL MODELO IS–LM–BP</strong><span class="origin-tag">4 párrafos</span></div><div class="paragraphs">${ps.map((v,i)=>`<div class="paragraph"><span class="para-n">${i+1}</span><p>${markVars(c.overrides?.[\'is:0:\'+i]??v,r)}</p></div>`).join(\'\')}</div></div>`:\'\'}</article>`;\n  }\n  function caseSheet(c,expose){let r=analyze(c);const stages=r.sequence||[];\n    return `<section class="paper ${expose?\'paper-present\':\'\'}"><div class="paper-header"><div><div class="paper-overline">ANÁLISIS DEL ESCENARIO</div><h2>${esc(c.title)}</h2><p>Variable explicativa: <strong>${esc(shock(c).label)}</strong> · ${c.shockId===\'tecnologia_sust\'?\'desplaza trabajadores → DL ↓\':QDIR(c.direction)+\' \'+sign(c.direction)} · Origen: ${esc(MS.MARKETS[shock(c).market])}</p></div><div class="result-pills">${pill(\'Y \'+sign(r.dominant.output),\'blue\')}${pill(\'Empleo \'+sign(r.dominant.employment),\'teal\')}${pill(\'Salarios \'+sign(r.dominant.wage),\'pink\')}${pill(\'i \'+sign(r.dominant.interest),\'amber\')}</div></div>\n      ${(c.showSecond&&r.balance?.applicable)?`<div class="balance-declaration"><strong>Alternativa seleccionada · segundo orden ${r.balance.stage}:</strong> ${c.shockId===\'ied\'&&c.routeMode===\'dual\'?`El modelo supone que predomina ${c.dualChoice===\'investment\'?\'la caída de inversión nacional\':\'el canal de tipo de cambio\'}`:`Predomina ${r.balance.choice===\'investment\'?\'la inversión (I)\':r.balance.choice===\'exchange\'?\'el efecto cambiario (X − M)\':\'el consumo (C)\'}`}; DA ${sign(r.balance.net)}.</div>`:\'\'}\n      ${formulas(c,r)}\n      ${c.showPolicyMap?policyMap(c):\'\'}\n      ${routeGraph(c,r)}\n      <div class="sequence-band"><div class="sequence-title">RUTA CAUSAL · RESUMEN TEXTUAL</div><div class="sequence-steps">${stages.filter(e=>c.showSecond||e.round!==\'segundo\').map(e=>`<span class="sequence-event ${e.round===\'segundo\'?\'second\':\'\'}"><b>${e.stage}</b> ${esc(TITLES[e.market])}: ${esc(SERIES[e.curve])}${sign(e.sign)}</span>`).join(\'\')}</div></div>\n      <div class="chart-grid">${CHARTS.map(m=>graphCard(m,c,r)).join(\'\')}${c.showIS?isCard(c,r):\'\'}</div>\n      \n      </section>`;}\n  function exposure(all=false){const p=pres(),list=all?p.cases:[p.cases[Math.max(0,Math.min(slideIndex,p.cases.length-1))]];return `<div class="expo"><div class="expo-toolbar"><div class="identity"><div class="logomark">M<span>Σ</span></div><div><strong>${esc(p.title)}</strong><small>Vista de exposición · ${all?\'todos los casos\':\'caso \'+(slideIndex+1)+\' de \'+p.cases.length}</small></div></div><div class="expo-buttons">${btn(\'Volver al editor\',\'editor\',\'outline\')}${btn(\'Ver todos\',\'expo-all\',\'outline\')}${btn(\'← Anterior\',\'expo-prev\',\'outline\',slideIndex===0||all?\'disabled\':\'\')}${btn(\'Siguiente →\',\'expo-next\',\'outline\',slideIndex>=p.cases.length-1||all?\'disabled\':\'\')}${btn(\'Imprimir / PDF\',\'print\',\'primary\')}</div></div><div class="expo-pages">${list.map(c=>caseSheet(c,true)).join(\'\')}</div></div>`}\n  function bankView(){return `<div class="dialog-back"><div class="dialog large"><div class="dialog-head"><div><div class="eyebrow">BANCO DE CASOS</div><h2>Preguntas documentadas del profesor</h2><p>Seleccioná una pregunta para añadirla a la presentación activa. Los casos se añaden a la presentación actual.</p></div>${btn(\'✕\',\'close-bank\',\'icon\')}</div><div class="bank-list">${BANK.map((b,i)=>`<div class="bank-row"><div class="bank-number">${String(i+1).padStart(2,\'0\')}</div><div><strong>${esc(b[0])}</strong><small>${esc(b[3])}</small></div>${btn(\'Añadir caso\',\'insert-bank\',\'outline small\',`data-index="${i}"`)}</div>`).join(\'\')}</div></div></div>`}\n  function customShockDialog(){const pairs={goods:[[\'da\',\'Demanda agregada (DA)\']],labor:[[\'ol\',\'Oferta laboral (OL)\'],[\'dl\',\'Demanda laboral (DL)\']],money:[[\'ms\',\'Oferta monetaria (Ms)\'],[\'md\',\'Demanda monetaria (Md)\']],fx:[[\'os\',\'Oferta de dólares (O$)\'],[\'ds\',\'Demanda de dólares (D$)\']]};return `<div class="dialog-back"><div class="dialog"><div class="dialog-head"><div><div class="eyebrow">EDITOR DEL MOTOR</div><h2>Agregar variable explicativa</h2><p>Definí el efecto inicial de la variable para generar un nuevo caso.</p></div>${btn(\'✕\',\'close-help\',\'icon\')}</div><form id="custom-shock-form" class="custom-form"><label>Nombre de la variable<input name="label" required placeholder="Ej. Cambios en ingreso extranjero"/></label><label>Mercado directamente afectado<select name="market" id="custom-market">${Object.entries(MS.MARKETS).map(([k,v])=>`<option value="${k}">${esc(v)}</option>`).join(\'\')}</select></label><label>Curva afectada<select name="curve" id="custom-curve">${pairs.goods.map(([k,v])=>`<option value="${k}">${v}</option>`).join(\'\')}</select></label><label>Si aumenta la variable, la curva<select name="effect"><option value="1">Se desplaza a la derecha / aumenta</option><option value="-1">Se desplaza a la izquierda / disminuye</option></select></label><label>Canal o ecuación<input name="channel" required placeholder="Ej. Impuestos → ingreso disponible → consumo"/></label><label>Descripción / supuesto<textarea name="description" rows="3" placeholder="Restricciones, ceteris paribus, contexto"></textarea></label><div class="dialog-footer">${btn(\'Cancelar\',\'close-help\',\'outline\')}<button class="button primary" type="submit">Guardar y utilizar</button></div></form></div></div>`}\n  function paragraphEditor(){let c=selected(),r=analyze(c);return `<div class="dialog-back"><div class="dialog editor-dialog"><div class="dialog-head"><div><div class="eyebrow">PERSONALIZAR REDACCIÓN</div><h2>Explicaciones del caso</h2><p>Se muestra el texto completo original cuando está disponible. Los cambios se conservan exclusivamente en este caso.</p></div>${btn(\'✕\',\'close-help\',\'icon\')}</div><div class="text-editor-scroll">${CHARTS.map(m=>{let arr=paragraphsFor(c,r,m);return `<div class="edit-block"><h3>${esc(TITLES[m])}</h3>${arr.map((x,i)=>`<h4>${x.event.round===\'segundo\'?\'Segundo\':\'Primer\'} orden — secuencia ${x.event.stage}</h4>${x.paragraphs.map((v,j)=>`<label class="edit-label">Párrafo ${j+1}<textarea data-edit="${m}:${r.changes[m].indexOf(x.event)}:${j}">${esc(v)}</textarea></label>`).join(\'\')}`).join(\'\')}</div>`}).join(\'\')}<div class="edit-block"><h3>Modelo IS–LM–BP</h3><label class="edit-label">Interpretación<textarea data-edit="is:0:0">${esc(c.overrides?.[\'is:0:0\']??refSection(c,\'is\')?.[0]??r.explanations.is)}</textarea></label></div></div><div class="dialog-footer">${btn(\'Cerrar\',\'close-help\',\'primary\')}</div></div></div>`}\n  function screen(){root.innerHTML=mode===\'present\'?exposure(false):mode===\'expo-all\'?exposure(true):editor();if(help===\'bank\')root.insertAdjacentHTML(\'beforeend\',bankView());else if(help===\'paragraphs\')root.insertAdjacentHTML(\'beforeend\',paragraphEditor());else if(help===\'custom\')root.insertAdjacentHTML(\'beforeend\',customShockDialog());}\n  function update(fn){fn();save();screen()}\n  function exportFile(name,content,mime){let b=new Blob([content],{type:mime}),u=URL.createObjectURL(b),a=document.createElement(\'a\');a.href=u;a.download=name;document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),1500)}\n  const filename=s=>s.normalize(\'NFD\').replace(/[\\u0300-\\u036f]/g,\'\').replace(/[^a-z0-9_-]+/gi,\'_\').slice(0,55)||\'Macro_Studio\';\n  function exportHtml(){const copy=document.documentElement.cloneNode(true);const snap=copy.querySelector(\'#macro-snapshot\');if(snap)snap.textContent=JSON.stringify(state).replace(/</g,\'\\\\u003c\');exportFile(filename(pres().title)+\'.html\',\'<!DOCTYPE html>\\n\'+copy.outerHTML,\'text/html;charset=utf-8\');toast(\'HTML autónomo con casos guardados\')}\n  function exportJSON(){exportFile(filename(pres().title)+\'.json\',JSON.stringify(state,null,2),\'application/json;charset=utf-8\');toast(\'Respaldo JSON descargado\')}\n  function importJSON(){let input=document.createElement(\'input\');input.type=\'file\';input.accept=\'.json\';input.onchange=()=>{let file=input.files?.[0];if(!file)return;file.text().then(t=>{let data=JSON.parse(t);if(!Array.isArray(data.presentations)||!data.presentations.length)throw Error(\'No hay presentaciones\');data.customShocks??=[];state=data;state.activePres=data.presentations[0].id;state.activeCase=data.presentations[0].cases[0].id;update(()=>{mode=\'editor\'});toast(\'Biblioteca restaurada\')}).catch(e=>toast(\'No se pudo importar: \'+e.message))};input.click()}\n  function handle(action,button){let p=pres(),c=selected(),idx=p.cases.indexOf(c);\n    if(action===\'close-bank\'||action===\'close-help\'){help=false;screen();return}\n    if(action===\'bank\'){help=\'bank\';screen();return}\n    if(action===\'diag-sample\'){const p=policySamples.find(z=>z.id===button.dataset.id);if(p){diagnosticPoint={Y:p.Y,i:p.i};screen();}return}\n    if(action===\'diag-apply\'){const target=policySamples.find(z=>z.id===button.dataset.id);if(!target)return;update(()=>{c.shockId=target.policy;c.direction=target.dir;c.title=\'Diagnóstico \'+target.id+\' · \'+target.title;c.overrides={};c.showPolicyMap=true;c.showIS=true;c.showSecond=true;});toast(\'Política aplicada: revisá los cuatro mercados y el modelo IS–LM–BP.\');return}\n    if(action===\'edit-paragraphs\'){help=\'paragraphs\';screen();return}\n    if(action===\'custom-shock\'){help=\'custom\';screen();return}\n    if(action===\'insert-bank\'){let q=BANK[Number(button.dataset.index)];update(()=>{let x=mkcase(q[0],q[1],q[2]);p.cases.push(x);state.activeCase=x.id;mode=\'editor\';help=false});return}\n    if(action===\'new-case\'){update(()=>{let x=mkcase();p.cases.push(x);state.activeCase=x.id});return}\n    if(action===\'new-pres\'){let name=prompt(\'Nombre de la nueva colección:\',\'Nueva colección\');if(!name)return;update(()=>{let next=mkpres(name.trim());state.presentations.push(next);state.activePres=next.id;state.activeCase=next.cases[0].id});return}\n    if(action===\'pick-pres\'){update(()=>{state.activePres=button.dataset.id;state.activeCase=pres().cases[0].id});return}\n    if(action===\'rename-pres\'){\n      const target=state.presentations.find(x=>x.id===button.dataset.id);if(!target)return;\n      let value=prompt(\'Nuevo nombre de la colección:\',target.title);if(value===null||!value.trim())return;\n      update(()=>target.title=value.trim().slice(0,120));return;\n    }\n    if(action===\'delete-pres\'){\n      const target=state.presentations.find(x=>x.id===button.dataset.id);if(!target)return;\n      if(state.presentations.length===1){toast(\'Para dejar el espacio vacío, utilizá Limpiar espacio.\');return;}\n      if(!confirm(\'¿Eliminar la colección «\'+target.title+\'» y sus \'+target.cases.length+\' casos? Esta acción no se puede deshacer.\'))return;\n      update(()=>{state.presentations=state.presentations.filter(x=>x.id!==target.id);state.activePres=state.presentations[0].id;state.activeCase=state.presentations[0].cases[0].id});return;\n    }\n    if(action===\'duplicate-pres\'){\n      const target=state.presentations.find(x=>x.id===button.dataset.id);if(!target)return;\n      update(()=>{const copy=clone(target);copy.id=uid();copy.title+=\' — copia\';for(const c of copy.cases)c.id=uid();state.presentations.push(copy);state.activePres=copy.id;state.activeCase=copy.cases[0].id});return;\n    }\n    if(action===\'rename-case\'){\n      const target=p.cases.find(x=>x.id===button.dataset.id);if(!target)return;\n      let value=prompt(\'Nuevo título del caso:\',target.title);if(value===null||!value.trim())return;\n      update(()=>target.title=value.trim().slice(0,200));return;\n    }\n    if(action===\'delete-case\'){\n      const target=p.cases.find(x=>x.id===button.dataset.id);if(!target)return;\n      if(!confirm(\'¿Eliminar el caso «\'+target.title+\'»?\'))return;\n      update(()=>{p.cases=p.cases.filter(x=>x.id!==target.id);if(!p.cases.length)p.cases.push(mkcase(\'Nuevo caso\'));state.activeCase=p.cases[0].id});return;\n    }\n    if(action===\'clear-workspace\'){\n      if(!confirm(\'¿Limpiar TODO el espacio de trabajo? Se eliminarán tus colecciones, casos y variables personalizadas de este navegador. Guardá JSON antes si querés conservarlos.\'))return;\n      update(()=>{const next=mkpres(\'Mis casos\');state.presentations=[next];state.activePres=next.id;state.activeCase=next.cases[0].id;state.customShocks=[];mode=\'editor\';help=false});\n      toast(\'Espacio limpio. Podés crear nuevos casos o importar un respaldo JSON.\');return;\n    }\n    if(action===\'pick-case\'){update(()=>state.activeCase=button.dataset.id);return}\n    if(action===\'duplicate\'){update(()=>{let x=clone(c);x.id=uid();x.title+=\' — copia\';p.cases.splice(idx+1,0,x);state.activeCase=x.id});return}\n    if(action===\'delete\'){if(confirm(\'¿Eliminar este caso?\'))update(()=>{p.cases.splice(idx,1);if(!p.cases.length)p.cases.push(mkcase(\'Nuevo caso\'));state.activeCase=p.cases[0].id});return}\n    if(action===\'present\'){mode=\'present\';slideIndex=Math.max(0,idx);help=false;screen();return}\n    if(action===\'expo-all\'){mode=\'expo-all\';screen();return}\n    if(action===\'expo-prev\'){slideIndex=Math.max(0,slideIndex-1);screen();return}\n    if(action===\'expo-next\'){slideIndex=Math.min(p.cases.length-1,slideIndex+1);screen();return}\n    if(action===\'editor\'){mode=\'editor\';state.activeCase=p.cases[slideIndex]?.id||state.activeCase;screen();return}\n    if(action===\'print\'){window.print();return}\n    if(action===\'export-html\'){exportHtml();return}\n    if(action===\'export-json\'){exportJSON();return}\n    if(action===\'import-json\'){importJSON();return}\n  }\n  document.addEventListener(\'click\',e=>{\n    const plot=e.target.closest(\'.policy-svg\');\n    if(plot && !e.target.closest(\'[data-act]\')){const rect=plot.getBoundingClientRect(),mx=(e.clientX-rect.left)*760/rect.width,my=(e.clientY-rect.top)*420/rect.height;diagnosticPoint={Y:Math.max(5,Math.min(98,(mx-98)/5.7)),i:Math.max(5,Math.min(98,(365-my)/3.25))};screen();return;}\n    const n=e.target.closest(\'.causal-node\');\n    if(n){\n      const panel=n.closest(\'.causal-panel\'),id=n.dataset.nodeId;\n      const links=[...panel.querySelectorAll(\'.causal-edge\')];\n      const adjacent=links.filter(a=>a.dataset.from===id||a.dataset.to===id);\n      const related=new Set([id,...adjacent.map(a=>a.dataset.from),...adjacent.map(a=>a.dataset.to)]);\n      panel.querySelectorAll(\'.causal-node\').forEach(a=>a.classList.toggle(\'is-related\',related.has(a.dataset.nodeId)));\n      panel.querySelectorAll(\'.causal-edge\').forEach(a=>a.classList.toggle(\'is-related\',adjacent.includes(a)));\n      const c=selected(),r=analyze(c),ev=r.sequence.find(x=>x.id===id);\n      if(ev){const names={goods:\'Bienes y servicios\',labor:\'Mercado laboral\',money:\'Mercado monetario\',fx:\'Mercado cambiario\'};const origin=ev.parents?.map(s=>{const [m,n]=s.split(\':\');return `${names[m]||m}, secuencia ${n}`}).join(\'; \');panel.querySelector(\'.causal-detail\').textContent=`Secuencia ${ev.stage} · ${TITLES[ev.market]} · ${ev.round} orden. ${ev.why} Antecedentes: ${origin||\'variable explicativa inicial\'}.`}\n      return;\n    }\n    const b=e.target.closest(\'[data-act]\');if(b){e.preventDefault();handle(b.dataset.act,b)}});\n  document.addEventListener(\'input\',e=>{const id=e.target.id,c=selected();if(id===\'title-input\'){c.title=e.target.value;save();let h1=document.querySelector(\'.case-heading h1\');if(h1)h1.textContent=c.title;let h2=document.querySelector(\'.paper-header h2\');if(h2)h2.textContent=c.title}\n    if(e.target.dataset.edit){c.overrides[e.target.dataset.edit]=e.target.value;save();}\n  });\n  document.addEventListener(\'change\',e=>{if(e.target.classList.contains(\'zone-select\')){const states={\'ro-rm\':\'Exceso de Oferta Agregada (OA mayor que DA) y exceso de Demanda de Dinero (Md mayor que Ms).\',\'ro-lm\':\'Exceso de Oferta Agregada (OA mayor que DA) y exceso de Oferta de Dinero (Ms mayor que Md).\',\'lo-rm\':\'Exceso de Demanda Agregada (DA mayor que OA) y exceso de Demanda de Dinero (Md mayor que Ms).\',\'lo-lm\':\'Exceso de Demanda Agregada (DA mayor que OA) y exceso de Oferta de Dinero (Ms mayor que Md).\'};e.target.closest(\'.region-explorer\').querySelector(\'.zone-reading\').textContent=states[e.target.value];return;}if(e.target.id===\'custom-market\'){let opts={goods:[[\'da\',\'Demanda agregada (DA)\']],labor:[[\'ol\',\'Oferta laboral (OL)\'],[\'dl\',\'Demanda laboral (DL)\']],money:[[\'ms\',\'Oferta monetaria (Ms)\'],[\'md\',\'Demanda monetaria (Md)\']],fx:[[\'os\',\'Oferta de dólares (O$)\'],[\'ds\',\'Demanda de dólares (D$)\']]};document.querySelector(\'#custom-curve\').innerHTML=opts[e.target.value].map(([a,b])=>`<option value="${a}">${b}</option>`).join(\'\');return}let c=selected(),id=e.target.id;if(id===\'shock-input\')update(()=>{c.shockId=e.target.value;if(c.shockId===\'tecnologia_sust\')c.direction=1;c.title=titleForCase(c);c.overrides={};if(shock(c).market===\'fx\')c.showFX=true;});\n    else if(id===\'route-input\')update(()=>{c.routeMode=e.target.value;c.overrides={}});\n    else if(id===\'dual-input\')update(()=>{c.dualChoice=e.target.value;c.overrides={}});\n    else if(id===\'trade-input\')update(()=>{c.tradeChoice=e.target.value;c.overrides={}});\n    else if(id===\'direction-input\')update(()=>{c.direction=c.shockId===\'tecnologia_sust\'?1:Number(e.target.value);c.title=titleForCase(c);c.overrides={}});\n    else if(id===\'second-input\')update(()=>c.showSecond=e.target.checked);\n    else if(id===\'is-input\')update(()=>c.showIS=e.target.checked);\n    else if(id===\'policy-input\')update(()=>c.showPolicyMap=e.target.checked);\n    else if(id===\'fx-input\')update(()=>c.showFX=true);\n    else if(id===\'explain-input\')update(()=>c.showExplanation=e.target.checked);\n    else if(e.target.name===\'balanceChoice\')update(()=>{c.balanceChoice=e.target.value;c.overrides={};});\n\n  });\n  document.addEventListener(\'submit\',e=>{if(e.target.id!==\'custom-shock-form\')return;e.preventDefault();let d=Object.fromEntries(new FormData(e.target));let newShock={id:uid(),label:d.label,market:d.market,curve:d.curve,effect:Number(d.effect),channel:d.channel,variable:d.label,description:d.description};update(()=>{state.customShocks.push(newShock);selected().shockId=newShock.id;selected().title=\'¿Qué sucede si \'+d.label.toLowerCase()+\' aumenta?\';selected().direction=1;selected().overrides={};if(newShock.market===\'fx\')selected().showFX=true;help=false});toast(\'Variable registrada en esta biblioteca\');});\n  document.addEventListener(\'keydown\',e=>{if((e.key===\'Enter\'||e.key===\' \')&&e.target.closest(\'.causal-node\')){e.preventDefault();e.target.dispatchEvent(new MouseEvent(\'click\',{bubbles:true}));return}if(e.key===\'Escape\'&&help){help=false;screen()}if((mode===\'present\')&&![\'INPUT\',\'TEXTAREA\',\'SELECT\'].includes(document.activeElement.tagName)){if(e.key===\'ArrowRight\'){slideIndex=Math.min(slideIndex+1,pres().cases.length-1);screen()}if(e.key===\'ArrowLeft\'){slideIndex=Math.max(0,slideIndex-1);screen()}}});\n  window.MacroStudio={getState:()=>clone(state),analyze:c=>analyze(c||selected()),getReference:c=>REFERENCE_CASES[exactKey(c||selected())]||null,bank:()=>clone(BANK),debug:{evolution,point},addCase:(title,shockId,direction)=>{let p=pres(),c=mkcase(title,shockId,direction);update(()=>{p.cases.push(c);state.activeCase=c.id});return c.id},reset:()=>{state=seed();state.activeCase=state.presentations[0].cases[0].id;save();screen();}};\n  screen();\n})();\n'

In [ ]:
def generar_macro_studio(nombre="index.html"):
    html = (PLANTILLA_HTML
        .replace('@@MS_STYLE@@', ESTILOS_CSS)
        .replace('@@MS_SCRIPT_0@@', 'null')
        .replace('@@MS_ENGINE@@', MOTOR_ECONOMICO_JS)
        .replace('@@MS_REFERENCE@@', REFERENCIAS_JS)
        .replace('@@MS_APPLICATION@@', APLICACION_JS))
    if '@@MS_' in html:
        raise ValueError('Queda una referencia de módulo sin resolver')
    salida = Path(nombre)
    salida.write_text(html, encoding='utf-8')
    resumen = hashlib.sha256(salida.read_bytes()).hexdigest()
    print('Archivo generado:', salida.resolve())
    print('Tamaño:', f'{salida.stat().st_size:,}', 'bytes')
    print('SHA-256:', resumen)
    try: display(FileLink(str(salida)))
    except Exception: pass
    return salida

aplicacion = generar_macro_studio()

## Comprobaciones de esta versión

- Los archivos `MOTOR_ECONOMICO_JS` y `APLICACION_JS` están incrustados en el notebook, sin recursos externos.
- El HTML es autónomo y se abre directamente en el navegador; con el nombre `index.html` también puede publicarse en GitHub Pages.
- La elección de predominio solo aparece cuando hay efectos contrapuestos calculados con antecedentes válidos.
- La lectura de las fórmulas deriva de las presentaciones del curso. El diagnóstico IS–LM–BP es esquemático y utiliza puntos ilustrativos, no valores macroeconómicos observados.

**Recomendación de publicación:** conservar los archivos de la versión anterior en el historial de Git. Actualizar simultáneamente este `.ipynb` y el `index.html` generado en un mismo commit o mediante el flujo de automatización de GitHub Actions; no reemplazar el release 5.0.0 sin identificar la nueva versión.